# GPU-Based K-Means Clustering Using CUDA — Problem 5

Lloyd's K-Means on an NVIDIA GPU in CUDA C++: given $N$ points in $\mathbb{R}^D$ and $K$ initial
centroids, alternate between assigning every point to its nearest centroid by squared Euclidean
distance and recomputing every centroid as the mean of its assigned points, until assignments stop
changing or the largest centroid movement is $\le \varepsilon$, or until the iteration limit is
reached. The program reports final assignments, final centroids, the iteration count and the
execution time.

The two centroid-accumulation strategies required by the assignment — **Approach A (global
atomics)** and **Approach B (deterministic two-stage reduction)** — are both implemented and
selectable at run time, and the assignment kernel exists in a baseline and a shared-memory variant.

> **This notebook is generated.** `tools/build_notebook.py` assembles it from the files under
> `src/`, `tests/` and `tools/cells/`, and its `%%writefile` cells are checked to be byte-identical
> to those files (DEC-011). Never edit the notebook by hand: change the repository files and
> regenerate, or the next regeneration silently discards the edit.

## Design decisions (requirements.md §3.1)

| Decision | Selected policy |
|---|---|
| Centroid accumulation (A) | Element-wise global `atomicAdd` on binary64 sums and `uint32` counts (§6) |
| Centroid accumulation (B) | Two-stage ownership-based reduction: chunk-privatised partial sums, then a fixed-order tree combine (DEC-001) — race-free without atomics, and deterministic |
| Convergence | Assignment stability **or** $\Delta \le \varepsilon$ (Euclidean, inclusive); stability takes reporting precedence (DEC-002) |
| Empty clusters | Retain the previous centroid; excluded from $\Delta$ (DEC-003) |
| Final consistency | Report the last $(A^{(t)}, C^{(t)})$ pair; P1 holds by construction, P2 verified and reported as a flag (DEC-004) |
| Tie-breaking | Lowest centroid index on exact binary32 equality, via strict-less-than replacement in ascending $k$ (DEC-005) |
| Iteration counting | Number of completed centroid updates (DEC-006) |
| Numeric types | binary32 points, centroids and distances; binary64 centroid sums, movement and tolerance; `uint32` counts (DEC-007) |
| Assignment mapping | One thread per point, grid-stride, ascending-$j$ distance accumulation (DEC-008) |
| Device layout | Two read-only copies of $X$: dimension-major for assignment and Approach A, row-major for Approach B Stage 1 (DEC-009) |
| Global synchronisation | Kernel boundaries on a single stream only; no cooperative-groups grid sync (DEC-010) |
| Deliverable | This single notebook, generated from `src/` (DEC-011) |
| Workflow | Edit and test locally, push to GitHub, run on a Colab GPU, save the executed notebook under `results/` (DEC-012) |

## How to run

1. **Runtime → Change runtime type → GPU** (ENV-11). The configuration cell fails fast with no GPU.
2. Run the cells top to bottom. The environment cell records the machine (§1.6.2), the
   configuration cell derives `SM_ARCH` from the recorded compute capability, and the smoke test
   proves the toolchain before any program source is compiled.
3. Save the executed notebook with **File → Save a copy in GitHub** to
   `GPU-Based-K-Means-Clustering/results/kmeans_cuda_<YYYYMMDD-HHMM>.ipynb` (DEC-012). GPU results
   count as evidence only from an executed notebook committed there.

## 2. Environment record

Records the machine-dependent items left open by requirements.md §1.6.2 — ENV-01 (GPU model and
compute capability), ENV-02 (CUDA toolkit, driver, `compute-sanitizer`), ENV-03 (host OS and host
compiler) and ENV-08 (profiler availability) — into the `ENV` dictionary that every later cell
reads, and that PERF-008 requires in each benchmark CSV row.

No version is assumed anywhere: whatever `nvidia-smi`, `nvcc` and `g++` report is what is recorded
and printed. A missing tool is recorded as unavailable rather than raising, because ENV-08 allows
the profilers to be absent (PERF-009 and TEST-P03 then fall back to CUDA-event phase timing, which
is reported as a limitation). The configuration cell, not this one, decides whether the recorded
environment is usable.

`nvidia-smi` does not report the SM count, and ENV-06 (FP64:FP32 throughput ratio) is a vendor
specification figure, not a queryable property. The SM count comes from
`cudaGetDeviceProperties` — printed by the smoke test below and required of the program itself by
CUDA-001 — and ENV-06 is cited from the vendor specification in the evaluation section.

In [ ]:
# Environment record — requirements.md §1.6.2 (ENV-01..ENV-03, ENV-08), PERF-008.
# Every probe is tolerant: a missing tool is recorded as unavailable, never raised. Nothing here
# assumes a GPU model, a toolkit version or a driver version; the reported strings are the record.
import json
import platform
import re
import shutil
import subprocess
import sys


def _run(cmd, timeout=120):
    """Run a probe command. Returns (ok, text); never raises."""
    if shutil.which(cmd[0]) is None:
        return False, "not installed"
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except Exception as exc:  # a probe must not break the notebook
        return False, f"probe failed: {exc!r}"
    out = (p.stdout or "").strip() or (p.stderr or "").strip()
    return p.returncode == 0, out


_VERSION_RE = re.compile(r"[0-9]+[.][0-9]+")


def _version_of(cmd):
    """Version line of a tool invoked as `<tool> --version`, or None.

    compute-sanitizer and ncu print the product name on the first line and the version several
    lines later, so taking the first line alone recorded no version at all for them (ENV-02,
    ENV-08). Prefer the first line that carries a version number, keeping the product name with
    it, and fall back to the first line when no line has one.
    """
    ok, out = _run([cmd, "--version"])
    if not ok:
        return None
    lines = [line.strip() for line in out.splitlines() if line.strip()]
    if not lines:
        return None
    if _VERSION_RE.search(lines[0]):
        return lines[0]
    for line in lines[1:]:
        if _VERSION_RE.search(line) and "opyright" not in line:
            return f"{lines[0]} {line}"
    return lines[0]


ENV = {}

# ENV-03 — host OS and host compiler of the runtime VM.
ENV["os"] = {
    "platform": platform.platform(),
    "release": _run(["cat", "/etc/os-release"])[1].replace("\n", "; ") or None,
    "python": sys.version.split()[0],
}
ENV["host_compiler"] = _version_of("g++")

# ENV-01 — GPU model and compute capability.
gpu_ok, gpu_csv = _run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,driver_version,memory.total",
     "--format=csv,noheader"]
)
ENV["gpu"] = None
if gpu_ok and gpu_csv:
    fields = [f.strip() for f in gpu_csv.splitlines()[0].split(",")]
    if len(fields) == 4:
        ENV["gpu"] = {
            "name": fields[0],
            "compute_cap": fields[1],
            "driver_version": fields[2],
            "memory_total": fields[3],
        }
        ENV["gpu_count"] = len(gpu_csv.splitlines())
if ENV["gpu"] is None:
    ENV["gpu_probe"] = gpu_csv or "nvidia-smi unavailable"

# ENV-02 — CUDA toolkit and the sanitizer the §16.4 suite needs.
nvcc_ok, nvcc_out = _run(["nvcc", "--version"])
ENV["nvcc"] = nvcc_out.replace("\n", " | ") if nvcc_ok else None
ENV["compute_sanitizer"] = _version_of("compute-sanitizer")

# ENV-08 — profiler availability; absence is a recorded limitation, not an error.
ENV["profilers"] = {"nsys": _version_of("nsys"), "ncu": _version_of("ncu")}

# Notebook analysis stack (§1.5 deliverables 4 and 5; no other dependency is permitted).
ENV["python_packages"] = {}
for _pkg in ("numpy", "pandas", "matplotlib"):
    try:
        ENV["python_packages"][_pkg] = __import__(_pkg).__version__
    except Exception:
        ENV["python_packages"][_pkg] = None

print(json.dumps(ENV, indent=2))

## 3. Configuration

Derives everything machine-dependent from the record above — nothing is hard-coded before ENV-01
is known (BLD-001c):

- **`SM_ARCH`** from the reported compute capability (`7.5` → `sm_75`).
- **Fail fast** when the environment cannot run the assignment: no GPU attached, compute capability
  below 6.0 (CUDA-001 — `atomicAdd` on `double`, which ATOM-003 requires, is not native below
  6.0), or a CUDA 13.x toolkit paired with a device below CC 7.5, whose offline compilation that
  toolkit removed (BLD-001a). Each case raises with the action to take, so "restart kernel and run
  all" stops here rather than failing later inside a build.
- **Build flags** (BLD-002): a release build `-O3 -lineinfo -arch=$SM_ARCH -std=c++17` with
  `-Xptxas -v` so register and shared-memory usage appear in the build log (CUDA-010), and a
  sanitizer/debug build that adds `-DKMEANS_SYNC_CHECK` (CUDA-008). `--use_fast_math` is never
  used (NUM-002), and the CPU reference is compiled with FP contraction disabled (NUM-003).
- **`QUICK`**: `True` keeps the run to the compile, hand-dataset and lockstep cells — the checks
  that decide whether the kernels are correct at all. It skips the longer sweeps (the `G` sweep,
  partial Stage 2 tiles, the non-aligned size matrix) and, from M5/M6, the sanitizer and benchmark
  sections (BLD-007). Set it to `False` for a run that is meant to produce submission evidence.
- **Paths**: results — benchmark CSVs, output files and logs — go to Google Drive when
  `USE_DRIVE` is enabled and the mount succeeds, so a runtime disconnect does not lose them;
  otherwise a warning is printed, local disk is used, and the fallback is recorded in `ENV` so
  results that were not persisted stay identifiable (BLD-001c). Sources, binaries and generated
  datasets always stay on local disk: the Drive FUSE mount does not permit executing binaries, so
  anything built there could not be run, and datasets are reproducible from their seeds (FR-003)
  while reaching tens of GB at the top of the PERF-003 matrix. The cell `chdir`s into the build
  directory, which is why the `%%writefile` cells use flat file names.

In [ ]:
# Configuration — requirements.md §15.3 part 3; BLD-001a, BLD-001c, BLD-002, CUDA-001, NUM-002/003.
import os
import re
import subprocess

USE_DRIVE = True  # BLD-001c: persist results to Drive; falls back to local disk on failure.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/kmeans_cuda"
BUILD_DIR = "/content/kmeans"  # always local: see the note on exec permissions below

# QUICK keeps a development run to the compile, hand-dataset and lockstep cells: the checks that
# decide whether the kernels are correct at all. It drops the longer sweeps (the G sweep, partial
# Stage 2 tiles and the non-aligned size matrix) and, from M5/M6, the sanitizer and benchmark
# sections. Set it to False for a run that produces submission evidence (BLD-007).
QUICK = False

if "ENV" not in globals():
    raise RuntimeError("Run the environment cell (section 2) first: ENV is not defined.")

# --- ENV-01 -> SM_ARCH (BLD-001c: derived, never hard-coded) ------------------------------------
if not ENV.get("gpu"):
    raise RuntimeError(
        "No NVIDIA GPU visible (ENV-01). Set Runtime -> Change runtime type -> Hardware "
        "accelerator: GPU, then re-run from section 2. Probe said: "
        f"{ENV.get('gpu_probe', 'unknown')}"
    )

COMPUTE_CAP = ENV["gpu"]["compute_cap"].strip()
_cc_match = re.fullmatch(r"(\d+)\.(\d+)", COMPUTE_CAP)
if not _cc_match:
    raise RuntimeError(f"Could not parse compute capability {COMPUTE_CAP!r} reported by nvidia-smi.")
CC_MAJOR, CC_MINOR = int(_cc_match.group(1)), int(_cc_match.group(2))
CC_VALUE = CC_MAJOR + CC_MINOR / 10.0
SM_ARCH = f"sm_{CC_MAJOR}{CC_MINOR}"

# CUDA-001: ATOM-003 needs native atomicAdd(double), which arrived with compute capability 6.0.
if CC_VALUE < 6.0:
    raise RuntimeError(
        f"Compute capability {COMPUTE_CAP} is below the 6.0 minimum (CUDA-001): native "
        "atomicAdd(double) required by ATOM-003 is unavailable. Use a newer GPU runtime."
    )

# ENV-02: whatever nvcc reports is what is used; no toolkit version is assumed.
NVCC_VERSION = ENV.get("nvcc")
if not NVCC_VERSION:
    raise RuntimeError("nvcc is not available in this runtime (ENV-02); CUDA code cannot be built.")
_rel = re.search(r"release (\d+)\.(\d+)", NVCC_VERSION)
CUDA_TOOLKIT = f"{_rel.group(1)}.{_rel.group(2)}" if _rel else "unknown"

# BLD-001a: CUDA 13.x dropped offline compilation for compute capability < 7.5.
if _rel and int(_rel.group(1)) >= 13 and CC_VALUE < 7.5:
    raise RuntimeError(
        f"CUDA toolkit {CUDA_TOOLKIT} cannot compile offline for compute capability "
        f"{COMPUTE_CAP} (BLD-001a). Use a CUDA 12.x toolkit, or a GPU with CC >= 7.5."
    )

# --- BLD-002 build flags ------------------------------------------------------------------------
# No --use_fast_math anywhere (NUM-002): it would change the distance semantics of §2.2.
NVCC_COMMON = ["-std=c++17", f"-arch={SM_ARCH}", "-lineinfo"]
NVCC_RELEASE_FLAGS = NVCC_COMMON + ["-O3", "-Xptxas", "-v"]          # CUDA-010 usage in build log
NVCC_DEBUG_FLAGS = NVCC_COMMON + ["-O3", "-DKMEANS_SYNC_CHECK"]      # CUDA-008 localises faults
# NUM-003: the CPU reference is the oracle and must not contract multiply-adds.
NVCC_HOST_FP_FLAGS = ["-Xcompiler", "-ffp-contract=off"]
GXX_REFERENCE_FLAGS = ["-O2", "-std=c++17", "-ffp-contract=off"]

# --- BLD-001c paths ----------------------------------------------------------------------------
# Sources, binaries and generated datasets stay on local disk and only results go to Drive, for
# two reasons: the Drive FUSE mount does not permit executing binaries, so a kmeans built there
# could not be run, and datasets reach tens of GB at the top of the PERF-003 matrix while being
# reproducible from their seeds (FR-003), so they are not worth persisting.
RESULTS_ROOT, DRIVE_STATUS = BUILD_DIR, "disabled"
if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        RESULTS_ROOT, DRIVE_STATUS = DRIVE_RESULTS_DIR, "mounted"
    except Exception as exc:
        DRIVE_STATUS = f"unavailable: {exc!r}"
        print(
            "warning[BLD-001c]: Google Drive not mounted; results stay in "
            f"{BUILD_DIR} and are lost if the runtime disconnects, so download anything you "
            f"need before closing the session. Reason: {exc!r}"
        )

WORK_DIR = BUILD_DIR                              # cwd for the %%writefile cells and the compiler
DATA_DIR = os.path.join(BUILD_DIR, "data")        # generated datasets (FR-003), regenerable
OUT_DIR = os.path.join(RESULTS_ROOT, "out")       # .kma/.kmd outputs and JSON summaries (BLD-004)
CSV_DIR = os.path.join(RESULTS_ROOT, "csv")       # benchmark CSVs (section 13.1)
LOG_DIR = os.path.join(RESULTS_ROOT, "logs")      # test and sanitizer logs (section 1.5)
for _d in (BUILD_DIR, DATA_DIR, OUT_DIR, CSV_DIR, LOG_DIR):
    os.makedirs(_d, exist_ok=True)
os.chdir(WORK_DIR)  # the %%writefile cells below use flat names relative to this directory

# PERF-008 asks for a build identity per benchmark row. In Colab there is no git checkout, so the
# executed notebook saved under results/ is itself the evidence (DEC-012); record what is knowable.
try:
    RUN_VERSION = "git:" + subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=30
    ).stdout.strip()
except Exception:
    RUN_VERSION = "unknown"
if RUN_VERSION == "git:":
    RUN_VERSION = "unknown"

ENV["drive_status"] = DRIVE_STATUS          # BLD-001c: unpersisted results stay identifiable
ENV["quick"] = QUICK                        # which sections this run actually executed
ENV["work_dir"] = WORK_DIR
ENV["results_root"] = RESULTS_ROOT
ENV["sm_arch"] = SM_ARCH
ENV["cuda_toolkit"] = CUDA_TOOLKIT
ENV["run_version"] = RUN_VERSION

print(f"GPU             : {ENV['gpu']['name']} (compute capability {COMPUTE_CAP})")
print(f"SM_ARCH         : {SM_ARCH}   [derived from ENV-01, BLD-001c]")
print(f"CUDA toolkit    : {CUDA_TOOLKIT} (as reported by nvcc)")
print(f"Drive           : {DRIVE_STATUS}")
print(f"QUICK           : {QUICK}   [True = compile, hand datasets and lockstep only]")
print(f"build dir (cwd) : {WORK_DIR}   [local: Drive mounts are not executable]")
print(f"results root    : {RESULTS_ROOT}")
print(f"data / out / csv: {DATA_DIR} | {OUT_DIR} | {CSV_DIR}")
print(f"run version     : {RUN_VERSION}")
print(f"nvcc release    : {' '.join(NVCC_RELEASE_FLAGS)}")
print(f"nvcc debug      : {' '.join(NVCC_DEBUG_FLAGS)}")
print(f"host FP (oracle): {' '.join(NVCC_HOST_FP_FLAGS)} / g++ {' '.join(GXX_REFERENCE_FLAGS)}")

## 4. Toolchain smoke test

A nine-line kernel is cheaper to debug than a multi-file build, so before any program source is
compiled this section writes `hello.cu`, builds it with the **same release flags the real build
uses** (BLD-002, so a bad flag is caught here), runs it, and checks three things:

1. `nvcc` accepts `-arch=$SM_ARCH` for the attached device and the binary links and runs;
2. a kernel launches and its writes are visible to the host (`kernel.check=OK`, verified
   element-wise, not just by a checksum);
3. the compute capability the device reports to host code equals the one `nvidia-smi` reported in
   section 2 — if these disagree, `SM_ARCH` was derived from the wrong device.

The cell raises on a non-zero compile or run exit code, on a mismatch, or on a missing `OK`, so
"restart kernel and run all" halts at the first toolchain fault (NFR-014). It also prints the SM
count, shared memory per block, warp size and the CUDA runtime and driver versions — the
`cudaGetDeviceProperties` fields that `nvidia-smi` does not expose, which CUDA-001 requires the
program to query, MEM-004 and SMEM-003 check tile sizes against, and CUDA-007 uses to size grids.
Whatever the runtime reports is printed verbatim; no CUDA version is assumed.

In [ ]:
%%writefile hello.cu
// Toolchain smoke test (requirements.md §15.3 part 4).
//
// Proves, before any program source is compiled, that the SM_ARCH derived from ENV-01 matches the
// attached device, that the toolkit reported by nvcc can build and link for it, and that a kernel
// launches and its writes are visible to the host. It also prints the device properties that
// nvidia-smi does not expose (SM count, shared memory per block) and that CUDA-001 requires the
// program itself to query, plus the CUDA runtime and driver versions exactly as reported — no
// version is assumed anywhere.
//
// Output is `key=value` lines so the notebook cell can assert on them. Exit codes follow BLD-005:
// 0 success, 4 CUDA runtime error.

#include <cstdio>
#include <cstdlib>

#include <cuda_runtime.h>

// CUDA-008: every runtime call is checked; failure reports file, line, call and the error string.
#define CUDA_CHECK(call)                                                                     \
    do {                                                                                     \
        cudaError_t err_ = (call);                                                            \
        if (err_ != cudaSuccess) {                                                            \
            std::fprintf(stderr, "error[CUDA-008]: %s:%d: %s failed: %s\n", __FILE__,          \
                         __LINE__, #call, cudaGetErrorString(err_));                           \
            std::exit(4);                                                                      \
        }                                                                                     \
    } while (0)

static const int kElements = 256;

// One thread per element, grid-stride (CUDA-007), bounds-checked (CUDA-005). The value is i*i so
// the host can verify every element independently of launch geometry.
__global__ void square_indices(int *out, int n)
{
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x) {
        out[i] = i * i;
    }
}

int main()
{
    int device = 0;
    CUDA_CHECK(cudaGetDevice(&device));

    cudaDeviceProp prop{};
    CUDA_CHECK(cudaGetDeviceProperties(&prop, device));

    int runtime_version = 0;
    int driver_version = 0;
    CUDA_CHECK(cudaRuntimeGetVersion(&runtime_version));
    CUDA_CHECK(cudaDriverGetVersion(&driver_version));

    std::printf("device.name=%s\n", prop.name);
    std::printf("device.compute_capability=%d.%d\n", prop.major, prop.minor);
    std::printf("device.sm_count=%d\n", prop.multiProcessorCount);
    std::printf("device.global_mem_bytes=%zu\n", prop.totalGlobalMem);
    std::printf("device.shared_mem_per_block_bytes=%zu\n", prop.sharedMemPerBlock);
    std::printf("device.shared_mem_per_block_optin_bytes=%zu\n", prop.sharedMemPerBlockOptin);
    std::printf("device.warp_size=%d\n", prop.warpSize);
    std::printf("device.max_threads_per_block=%d\n", prop.maxThreadsPerBlock);
    std::printf("runtime.cuda_version=%d\n", runtime_version);
    std::printf("driver.cuda_version=%d\n", driver_version);

    int *d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_out, kElements * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_out, 0, kElements * sizeof(int)));

    square_indices<<<4, 64>>>(d_out, kElements);
    CUDA_CHECK(cudaGetLastError());  // launch error (CUDA-008)

    int host_out[kElements];
    CUDA_CHECK(cudaMemcpy(host_out, d_out, sizeof(host_out), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));

    long long sum = 0;
    long long expected_sum = 0;
    int mismatches = 0;
    for (int i = 0; i < kElements; ++i) {
        sum += host_out[i];
        expected_sum += static_cast<long long>(i) * i;
        if (host_out[i] != i * i) {
            ++mismatches;
        }
    }

    std::printf("kernel.elements=%d\n", kElements);
    std::printf("kernel.sum=%lld\n", sum);
    std::printf("kernel.expected_sum=%lld\n", expected_sum);
    std::printf("kernel.mismatches=%d\n", mismatches);
    std::printf("kernel.check=%s\n", (mismatches == 0 && sum == expected_sum) ? "OK" : "FAIL");

    return (mismatches == 0 && sum == expected_sum) ? 0 : 1;
}


In [ ]:
# Toolchain smoke test — requirements.md §15.3 part 4. Raises on any failure so that
# "restart kernel and run all" stops here instead of inside a larger build (NFR-014).
import subprocess

if "SM_ARCH" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first: SM_ARCH is not defined.")

compile_cmd = ["nvcc"] + NVCC_RELEASE_FLAGS + ["-o", "hello", "hello.cu"]
print("$ " + " ".join(compile_cmd))
compile_proc = subprocess.run(compile_cmd, capture_output=True, text=True)
print(compile_proc.stdout + compile_proc.stderr, end="")  # -Xptxas -v usage report (CUDA-010)
if compile_proc.returncode != 0:
    raise RuntimeError(
        f"nvcc failed with exit code {compile_proc.returncode} for -arch={SM_ARCH} "
        f"(toolkit {CUDA_TOOLKIT}, device CC {COMPUTE_CAP}). See the output above; check BLD-001a "
        "if the toolkit is too new for this device."
    )

run_proc = subprocess.run(["./hello"], capture_output=True, text=True)
print(run_proc.stdout + run_proc.stderr, end="")
if run_proc.returncode != 0:
    raise RuntimeError(
        f"./hello exited with code {run_proc.returncode} "
        "(4 = CUDA runtime error per BLD-005). See the output above."
    )

smoke = dict(
    line.split("=", 1) for line in run_proc.stdout.splitlines() if "=" in line
)
if smoke.get("kernel.check") != "OK":
    raise RuntimeError(
        f"Kernel result wrong: {smoke.get('kernel.mismatches')} of "
        f"{smoke.get('kernel.elements')} elements mismatched. The toolchain builds but the device "
        "does not compute correctly; do not trust any later result."
    )
if smoke.get("device.compute_capability") != COMPUTE_CAP:
    raise RuntimeError(
        f"Device reports compute capability {smoke.get('device.compute_capability')} to host code "
        f"but nvidia-smi reported {COMPUTE_CAP} in section 2, so SM_ARCH={SM_ARCH} was derived "
        "from a different device (BLD-001c). Re-run from section 2."
    )

# SM count feeds CUDA-007 grid sizing and the Approach B chunk count G (RED-006); the shared-memory
# limits bound the assign_smem tiles (SMEM-003, MEM-004). Recorded now, used from M2 on.
ENV["device_properties"] = smoke
SM_COUNT = int(smoke["device.sm_count"])
SHARED_MEM_PER_BLOCK = int(smoke["device.shared_mem_per_block_bytes"])

print(
    f"\nsmoke test OK: {smoke['device.name']}, CC {smoke['device.compute_capability']}, "
    f"{SM_COUNT} SMs, {SHARED_MEM_PER_BLOCK} B shared memory per block, "
    f"CUDA runtime {smoke['runtime.cuda_version']}, driver {smoke['driver.cuda_version']}, "
    f"kernel sum {smoke['kernel.sum']} == expected {smoke['kernel.expected_sum']}"
)

## 5. Source files

Each cell below writes one repository file verbatim into the working directory. They are generated
from `src/` and `tests/` by `tools/build_notebook.py`, which also checks that what the notebook
writes is byte-identical to what the repository holds (DEC-011) — so editing a cell here is always
wrong: change the file in the repository and regenerate.

The host side, in the ARC order of §18.2:

| File | Component | Responsibility |
|---|---|---|
| `common.h` | ARC-01 | `Status` (§2.7), exit codes (BLD-005), message format (BLD-006), the NUM-005/NUM-006 tolerances, and overflow-checked size arithmetic with the MEM-002 capacity model (CUDA-006) |
| `validate.h/.cpp` | ARC-01 | VAL-001…VAL-010, including the VAL-007 range bound $4DM^2 \le 2^{126}$ that guarantees no binary32 distance can overflow during the run |
| `io_kmd.h/.cpp` | ARC-02 | the `.kmd`/`.kma` binary formats of BLD-004, with the FR-002 exact-size check |
| `cpu_reference.h/.cpp` | ARC-03 | the oracle of NUM-003: the FP32 distance path of §2.2, a binary64 path for near-tie margins, binary64 sums, the tie rule, empty-cluster retention, the convergence loop and finalization. The step functions are exposed individually for the §10.3 lockstep verification added in M5 |
| `gen.h/.cpp`, `gen_main.cpp` | ARC-02 | the FR-003 generator and the `kmeans-gen` CLI of BLD-003a |
| `hand_data.h`, `test_cpu.cpp` | ARC-12 | H1–H6 and H2-a…e with their §16.0 expected outcomes, plus TEST-U08, U09, U11, U12 and U13 at CPU level |

The device side, added in M2 and M3:

| File | Component | Responsibility |
|---|---|---|
| `launch_config.h` | ARC-04 | launch geometry and buffer sizing as plain integer arithmetic: the CUDA-007 grid rule, the RED-006 choice of `G`, the MEM-003 budget, the MEM-004/SMEM-003 shared-memory footprints and the DEC-001 chunk partition. Free of CUDA constructs on purpose, so all of it is unit tested with `g++` on a machine with no GPU |
| `cuda_util.cuh` | ARC-04 | CUDA-008 checked calls with file, line and `cudaGetErrorString`; `KMEANS_SYNC_CHECK`; and RAII owners for device memory, pinned memory, the stream and the events, so every error path releases what it allocated (MEM-007) |
| `device_context.cuh` | ARC-04 | CUDA-001 device query and the compute-capability 6.0 gate, the shared-memory limit checks, the `G` selection against real free memory, and VAL-009 — all before the first allocation, so EDGE-26 fails with exit code 3 and a byte count instead of a CUDA out-of-memory error |
| `kernels.cuh/.cu` | ARC-05/06/07/08 | the CUDA-002 kernel set: `assign`, `accum_atomic` (§6), `reduce_stage1` and `reduce_stage2` (§7), `update`, `movement`, `max_reduce`. Thread mapping, ownership and synchronization are documented at each kernel and summarised in the two contract notes above |
| `kmeans_gpu.h/.cu` | ARC-09/ARC-10 | the §2.7 loop in the CUDA-003 launch order, §2.8 finalization, the PERF-005 timings, the §10.3 lockstep hook and the FR-019 `final` verification. Only the accumulation step differs between the two strategies (FR-018) |
| `test_gpu.cu` | ARC-12 | TEST-I01…I07, I10, U03…U07 and E16 on the device, with `--only` and `--quick` so the notebook can run the sections separately |

Still to come: `assign_smem` (M4, SMEM-001), the `kmeans` CLI executable and its `.kma`/`.kmd`/JSON
outputs (FR-015, BLD-003a) with the sanitizer suite in M5, and the benchmark harness in M6.

In [ ]:
%%writefile common.h
// common.h — shared types, statuses, exit codes, messages and size arithmetic.
//
// requirements.md: Status (§2.7), ExitCode (BLD-005), message format (BLD-006),
// overflow-checked size arithmetic (CUDA-006) and the capacity model (MEM-002).
//
// Header-only and free of CUDA constructs, so the M2 .cu translation units include it unchanged.

#ifndef KMEANS_COMMON_H
#define KMEANS_COMMON_H

#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <string>
#include <vector>

namespace kmeans {

// --- §2.7 termination statuses -----------------------------------------------------------------

enum class Status {
    CONVERGED_ASSIGNMENTS,  // A(t) == A(t-1) exactly
    CONVERGED_MOVEMENT,     // delta <= epsilon, assignments still changing
    ITERATION_LIMIT,        // t == max_iter and neither criterion held (includes max_iter == 0)
    ERROR                   // validation, resource, CUDA or I/O failure; no clustering output
};

inline const char *status_name(Status s)
{
    switch (s) {
    case Status::CONVERGED_ASSIGNMENTS: return "CONVERGED_ASSIGNMENTS";
    case Status::CONVERGED_MOVEMENT:    return "CONVERGED_MOVEMENT";
    case Status::ITERATION_LIMIT:       return "ITERATION_LIMIT";
    case Status::ERROR:                 return "ERROR";
    }
    return "UNKNOWN";
}

// §2.7: termination is never reported as convergence.
inline bool is_converged(Status s)
{
    return s == Status::CONVERGED_ASSIGNMENTS || s == Status::CONVERGED_MOVEMENT;
}

// --- BLD-005 exit codes ------------------------------------------------------------------------

enum ExitCode {
    kExitOk = 0,                  // any non-error status
    kExitInvalidInput = 2,        // VAL rule violated
    kExitUnsupportedDevice = 3,   // unsupported device or insufficient resources
    kExitCudaError = 4,           // CUDA runtime error
    kExitIoError = 5,             // I/O error
    kExitVerificationFailed = 6   // only with --verify
};

// --- BLD-006 messages --------------------------------------------------------------------------
// Single-line prefix naming the rule that was violated, then the cause and the offending value.

inline void report_error(const char *rule_id, const std::string &what)
{
    std::fprintf(stderr, "error[%s]: %s\n", rule_id, what.c_str());
}

inline void report_warning(const char *warning_id, const std::string &what)
{
    std::fprintf(stderr, "warning[%s]: %s\n", warning_id, what.c_str());
}

// --- numerical constants (§0.1, NUM-005, NUM-006) ----------------------------------------------

// u32 = 2^-24, the unit roundoff of IEEE-754 binary32.
constexpr double kUnitRoundoffF32 = 5.9604644775390625e-08;

// NUM-005: a point is a near-tie when its relative margin is at most 4 (D + 2) u32.
inline double near_tie_threshold(int64_t D)
{
    return 4.0 * static_cast<double>(D + 2) * kUnitRoundoffF32;
}

// NUM-006: |a - b| <= 4 u32 |b| + 4 u32 S_max 2^-10, with a the value under test and b the
// reference. Both sides round one binary64 quotient to binary32, so they differ by at most about
// one binary32 ulp; the absolute term covers values near zero.
inline double centroid_tolerance(float reference, double s_max)
{
    const double abs_ref = std::fabs(static_cast<double>(reference));
    return 4.0 * kUnitRoundoffF32 * abs_ref +
           4.0 * kUnitRoundoffF32 * s_max * 0.0009765625;  // 2^-10
}

inline bool centroid_agrees(float test, float reference, double s_max)
{
    const double diff = std::fabs(static_cast<double>(test) - static_cast<double>(reference));
    return diff <= centroid_tolerance(reference, s_max);
}

// NUM-006: S_max = max(M, 1e-30), with M the range bound of VAL-007.
inline double s_max_from_range(double m_range)
{
    return m_range > 1e-30 ? m_range : 1e-30;
}

// --- CUDA-006 / MEM-002 size arithmetic --------------------------------------------------------
// Every element offset and allocation size is computed in unsigned 64-bit arithmetic with an
// explicit overflow check, so an oversized configuration is rejected before any allocation
// instead of wrapping into a small, plausible-looking byte count.

inline bool checked_mul(uint64_t a, uint64_t b, uint64_t *out)
{
    if (a != 0 && b > UINT64_MAX / a) {
        return false;
    }
    *out = a * b;
    return true;
}

inline bool checked_add(uint64_t a, uint64_t b, uint64_t *out)
{
    if (a > UINT64_MAX - b) {
        return false;
    }
    *out = a + b;
    return true;
}

// Byte counts of MEM-001, in the order of that table. `reduce_chunks` is the Approach B chunk
// count G; pass 0 for Approach A, whose run allocates no P/Q buffers.
struct MemoryModel {
    uint64_t x_col = 0;        // 4ND
    uint64_t x_row = 0;        // 4ND
    uint64_t centroids = 0;    // 8KD   (C_cur and C_next)
    uint64_t assignments = 0;  // 4N
    uint64_t sums = 0;         // 8KD
    uint64_t counts = 0;       // 4K
    uint64_t partials = 0;     // 8GKD
    uint64_t partial_counts = 0;  // 4GK
    uint64_t scalars = 0;      // movement block maxima, delta, change counter, pinned scalars
    uint64_t total = 0;
    bool overflow = false;     // true when any product or sum exceeded 64 bits
};

// MEM-002: required bytes = 8ND + 4N + 16KD + 4K + [8GKD + 4GK] + small.
inline MemoryModel device_memory_bytes(int64_t N, int64_t D, int64_t K, int64_t reduce_chunks,
                                       int64_t movement_blocks = 1024)
{
    MemoryModel m;
    const uint64_t n = static_cast<uint64_t>(N);
    const uint64_t d = static_cast<uint64_t>(D);
    const uint64_t k = static_cast<uint64_t>(K);
    const uint64_t g = static_cast<uint64_t>(reduce_chunks);

    uint64_t nd = 0, kd = 0, gkd = 0, gk = 0;
    bool ok = checked_mul(n, d, &nd) && checked_mul(k, d, &kd);
    ok = ok && checked_mul(g, kd, &gkd) && checked_mul(g, k, &gk);
    if (!ok) {
        m.overflow = true;
        return m;
    }

    ok = ok && checked_mul(nd, 4, &m.x_col);
    ok = ok && checked_mul(nd, 4, &m.x_row);
    ok = ok && checked_mul(kd, 8, &m.centroids);
    ok = ok && checked_mul(n, 4, &m.assignments);
    ok = ok && checked_mul(kd, 8, &m.sums);
    ok = ok && checked_mul(k, 4, &m.counts);
    ok = ok && checked_mul(gkd, 8, &m.partials);
    ok = ok && checked_mul(gk, 4, &m.partial_counts);
    ok = ok && checked_mul(static_cast<uint64_t>(movement_blocks), 8, &m.scalars);
    ok = ok && checked_add(m.scalars, 32, &m.scalars);  // delta, change counter, pinned copies
    if (!ok) {
        m.overflow = true;
        return m;
    }

    const uint64_t parts[] = {m.x_col,  m.x_row, m.centroids, m.assignments,  m.sums,
                              m.counts, m.partials, m.partial_counts, m.scalars};
    uint64_t total = 0;
    for (uint64_t part : parts) {
        if (!checked_add(total, part, &total)) {
            m.overflow = true;
            return m;
        }
    }
    m.total = total;
    return m;
}

// --- run configuration and result --------------------------------------------------------------

enum class Strategy { kAtomic, kReduction };
enum class AssignKernel { kBaseline, kSharedMemory };

struct Config {
    int64_t N = 0;
    int64_t D = 0;
    int64_t K = 0;
    int64_t max_iter = 0;
    double tol = 0.0;
    bool fixed_iters = false;  // FR-020: disable both convergence criteria
    Strategy strategy = Strategy::kAtomic;
    AssignKernel assign_kernel = AssignKernel::kBaseline;
    int block_size = 256;          // CUDA-007 default
    int64_t reduce_chunks = 0;     // 0 = choose automatically (RED-006)
};

// §2.8 step 3 reports exactly these fields; the CPU reference and the device driver fill the same
// record so the two can be compared field by field (§10.3).
struct Result {
    std::vector<int32_t> assignments;  // A_final
    std::vector<float> centroids;      // C_final, K x D row-major, unchanged by finalization
    std::vector<uint32_t> counts;      // n_final from the finalization accumulation pass
    int64_t iterations = 0;            // DEC-006: completed centroid updates
    Status status = Status::ERROR;
    double last_delta = 0.0;           // delta of the last update
    bool has_last_delta = false;       // false when t == 0 (reported as null, FR-015)
    bool centroid_consistent = false;  // P2, verified per NUM-006
    bool assignment_consistent = true; // P1, true by construction (§2.8)
    double objective = 0.0;            // J (§2.9), binary64, outside timed regions
};

}  // namespace kmeans

#endif  // KMEANS_COMMON_H


In [ ]:
%%writefile validate.h
// validate.h — input validation rules VAL-001..VAL-010 (requirements.md §9.1, FR-001).
//
// Every rule is applied before any device allocation, and the first violation names the parameter
// and the rule ID (BLD-006) and maps to an exit code (BLD-005). Rules that accept the input with a
// warning (VAL-003 K > N, VAL-008 duplicate centroids) record the warning ID instead of failing.

#ifndef KMEANS_VALIDATE_H
#define KMEANS_VALIDATE_H

#include <string>
#include <vector>

#include "common.h"

namespace kmeans {

struct Validation {
    bool ok = true;
    std::string rule;     // failing rule ID, empty when ok
    std::string message;  // cause and offending value
    int exit_code = kExitOk;
    std::vector<std::string> warnings;  // e.g. "W-KGTN", "W-DUPC", in the order detected

    bool has_warning(const std::string &id) const;
    // Emit the failure and every warning in BLD-006 format.
    void report() const;
};

// VAL-001, VAL-002, VAL-003, VAL-004, VAL-005. Sizes arrive as 64-bit integers (§2.1) and are
// rejected unless they fit the int32 assignment and uint32 count types.
Validation validate_parameters(int64_t N, int64_t D, int64_t K, int64_t max_iter, double tol);

// VAL-010: option values within their documented ranges and the device limits. Device limits are
// passed in so the rule is testable without a GPU; pass 0 for a limit that is not yet known.
Validation validate_options(int block_size, int64_t reduce_chunks, int tile_k, int tile_d,
                            int device_max_threads_per_block, int64_t device_shared_mem_per_block);

// VAL-007 range bound: M = max(max|x_ij|, max|c0_kj|). Returns M, which NUM-006 also needs as the
// scale S_max. Returns a negative value only if the arrays are empty.
double range_bound(const std::vector<float> &X, const std::vector<float> &C0);

// VAL-006 (finite values, first offending row and column reported), VAL-007 (4 D M^2 <= 2^126
// evaluated in binary64) and VAL-008 (shape of the centroid file, duplicate centroids warned).
// `m_range` receives the M of VAL-007 when validation succeeds.
Validation validate_data(const std::vector<float> &X, int64_t N, int64_t D,
                         const std::vector<float> &C0, int64_t K, double *m_range);

// VAL-009: total required device memory (MEM-002) must fit in the free device memory less a 5 %
// safety margin. `free_bytes` comes from cudaMemGetInfo in M2 and from the test in M1.
Validation validate_capacity(int64_t N, int64_t D, int64_t K, int64_t reduce_chunks,
                             uint64_t free_bytes);

}  // namespace kmeans

#endif  // KMEANS_VALIDATE_H


In [ ]:
%%writefile validate.cpp
#include "validate.h"

#include <algorithm>
#include <cmath>
#include <map>

namespace kmeans {
namespace {

constexpr int64_t kInt32Max = 2147483647;  // 2^31 - 1 (VAL-002, VAL-004)

Validation fail(const char *rule, const std::string &message, int exit_code)
{
    Validation v;
    v.ok = false;
    v.rule = rule;
    v.message = message;
    v.exit_code = exit_code;
    return v;
}

// Distinct names rather than overloads: an overloaded num(int) would be ambiguous between the integral and
// floating-point conversions, and every message below states exactly which kind of value it holds.
std::string istr(int64_t value)
{
    return std::to_string(value);
}

std::string ustr(uint64_t value)
{
    return std::to_string(value);
}

std::string dstr(double value)
{
    char buf[64];
    std::snprintf(buf, sizeof(buf), "%.17g", value);
    return std::string(buf);
}

}  // namespace

bool Validation::has_warning(const std::string &id) const
{
    return std::find(warnings.begin(), warnings.end(), id) != warnings.end();
}

void Validation::report() const
{
    for (const std::string &w : warnings) {
        if (w == "W-KGTN") {
            report_warning(w.c_str(), "K > N: at least K - N clusters stay empty and retain their "
                                      "initial centroids (VAL-003, DEC-003)");
        } else if (w == "W-DUPC") {
            report_warning(w.c_str(), "duplicate initial centroids: the higher-index duplicate "
                                      "receives no points while identical (VAL-008, DEC-005)");
        } else {
            report_warning(w.c_str(), "see requirements.md");
        }
    }
    if (!ok) {
        report_error(rule.c_str(), message);
    }
}

Validation validate_parameters(int64_t N, int64_t D, int64_t K, int64_t max_iter, double tol)
{
    // VAL-001
    if (N < 1) {
        return fail("VAL-001", "N must be at least 1, got N=" + istr(N), kExitInvalidInput);
    }
    if (D < 1) {
        return fail("VAL-001", "D must be at least 1, got D=" + istr(D), kExitInvalidInput);
    }
    if (K < 1) {
        return fail("VAL-001", "K must be at least 1, got K=" + istr(K), kExitInvalidInput);
    }

    // VAL-002: assignments must fit int32 and counts uint32.
    if (N > kInt32Max) {
        return fail("VAL-002", "N must be at most 2^31-1, got N=" + istr(N), kExitInvalidInput);
    }
    if (D > kInt32Max) {
        return fail("VAL-002", "D must be at most 2^31-1, got D=" + istr(D), kExitInvalidInput);
    }
    if (K > kInt32Max) {
        return fail("VAL-002", "K must be at most 2^31-1, got K=" + istr(K), kExitInvalidInput);
    }

    // VAL-004: max_iter = 0 is accepted and means "initial assignment only" (§2.7).
    if (max_iter < 0 || max_iter > kInt32Max) {
        return fail("VAL-004", "max-iter must be in [0, 2^31-1], got " + istr(max_iter),
                    kExitInvalidInput);
    }

    // VAL-005: epsilon finite and non-negative; NaN, +-Inf and negative values are rejected.
    if (std::isnan(tol)) {
        return fail("VAL-005", "tol must be a finite number, got NaN", kExitInvalidInput);
    }
    if (std::isinf(tol)) {
        return fail("VAL-005", "tol must be finite, got " + dstr(tol), kExitInvalidInput);
    }
    if (tol < 0.0) {
        return fail("VAL-005", "tol must be non-negative, got " + dstr(tol), kExitInvalidInput);
    }

    Validation v;
    // VAL-003: K > N is accepted with a warning; at least K - N clusters remain empty (DEC-003).
    if (K > N) {
        v.warnings.push_back("W-KGTN");
    }
    return v;
}

Validation validate_options(int block_size, int64_t reduce_chunks, int tile_k, int tile_d,
                            int device_max_threads_per_block, int64_t device_shared_mem_per_block)
{
    // CUDA-007: block size is a multiple of 32 in [32, 1024] and within the device limit.
    if (block_size < 32 || block_size > 1024 || block_size % 32 != 0) {
        return fail("VAL-010",
                    "block-size must be a multiple of 32 in [32, 1024], got " + istr(block_size),
                    kExitInvalidInput);
    }
    if (device_max_threads_per_block > 0 && block_size > device_max_threads_per_block) {
        return fail("VAL-010", "block-size " + istr(block_size) + " exceeds the device limit of " +
                                   istr(device_max_threads_per_block),
                    kExitInvalidInput);
    }

    // RED-006: G is chosen automatically when 0, otherwise it must be at least 1.
    if (reduce_chunks < 0) {
        return fail("VAL-010", "reduce-chunks must be at least 1 (or 0 for automatic), got " +
                                   istr(reduce_chunks),
                    kExitInvalidInput);
    }

    // SMEM-001/SMEM-003: tile sizes positive, T_D a multiple of the warp size, 4 T_K T_D bytes of
    // shared memory within the per-block limit (EDGE-34 rejects oversized configurations).
    if (tile_k < 1 || tile_d < 1) {
        return fail("VAL-010", "tile sizes must be at least 1, got tile_k=" + istr(tile_k) +
                                   " tile_d=" + istr(tile_d),
                    kExitInvalidInput);
    }
    if (tile_d % 32 != 0) {
        return fail("VAL-010", "tile_d must be a multiple of the warp size 32, got " + istr(tile_d),
                    kExitInvalidInput);
    }
    uint64_t shared_bytes = 0;
    if (!checked_mul(static_cast<uint64_t>(tile_k) * 4, static_cast<uint64_t>(tile_d),
                     &shared_bytes)) {
        return fail("VAL-010", "tile sizes overflow the shared-memory calculation",
                    kExitInvalidInput);
    }
    if (device_shared_mem_per_block > 0 &&
        shared_bytes > static_cast<uint64_t>(device_shared_mem_per_block)) {
        return fail("VAL-010", "assign_smem tiles need " + ustr(shared_bytes) +
                                   " B of shared memory, device allows " +
                                   istr(device_shared_mem_per_block) + " B per block",
                    kExitInvalidInput);
    }
    return Validation{};
}

double range_bound(const std::vector<float> &X, const std::vector<float> &C0)
{
    double m = -1.0;
    for (float value : X) {
        const double a = std::fabs(static_cast<double>(value));
        if (a > m) {
            m = a;
        }
    }
    for (float value : C0) {
        const double a = std::fabs(static_cast<double>(value));
        if (a > m) {
            m = a;
        }
    }
    return m;
}

Validation validate_data(const std::vector<float> &X, int64_t N, int64_t D,
                         const std::vector<float> &C0, int64_t K, double *m_range)
{
    Validation v;

    // VAL-008: the centroid file has exactly K rows of D values; the data file N rows of D.
    const uint64_t expect_x = static_cast<uint64_t>(N) * static_cast<uint64_t>(D);
    const uint64_t expect_c = static_cast<uint64_t>(K) * static_cast<uint64_t>(D);
    if (X.size() != expect_x) {
        return fail("VAL-008", "data has " + ustr(X.size()) +
                                   " values, expected N*D = " + ustr(expect_x),
                    kExitInvalidInput);
    }
    if (C0.size() != expect_c) {
        return fail("VAL-008", "centroids have " + ustr(C0.size()) +
                                   " values, expected K*D = " + ustr(expect_c),
                    kExitInvalidInput);
    }

    // VAL-006: every coordinate finite, reporting the first offending (row, column).
    for (int64_t i = 0; i < N; ++i) {
        for (int64_t j = 0; j < D; ++j) {
            const float value = X[static_cast<size_t>(i * D + j)];
            if (!std::isfinite(value)) {
                return fail("VAL-006", "data value at (row " + istr(i) + ", column " + istr(j) +
                                           ") is not finite: " + dstr(static_cast<double>(value)),
                            kExitInvalidInput);
            }
        }
    }
    for (int64_t k = 0; k < K; ++k) {
        for (int64_t j = 0; j < D; ++j) {
            const float value = C0[static_cast<size_t>(k * D + j)];
            if (!std::isfinite(value)) {
                return fail("VAL-006", "initial centroid at (row " + istr(k) + ", column " + istr(j) +
                                           ") is not finite: " + dstr(static_cast<double>(value)),
                            kExitInvalidInput);
            }
        }
    }

    // VAL-007: with M the largest magnitude in X and C0, require 4 D M^2 <= 2^126 so that no
    // binary32 distance can overflow during the run. Later centroids are means of points or
    // retained initial centroids, so they stay within [-M, M] up to rounding.
    const double m = range_bound(X, C0);
    const double bound = std::ldexp(1.0, 126);
    const double value = 4.0 * static_cast<double>(D) * m * m;
    if (!(value <= bound)) {
        return fail("VAL-007", "range bound exceeded: M=" + dstr(m) + ", 4*D*M^2=" + dstr(value) +
                                   " > 2^126=" + dstr(bound),
                    kExitInvalidInput);
    }
    if (m_range != nullptr) {
        *m_range = m;
    }

    // VAL-008: duplicate initial centroids are accepted with a warning. Compared bit-wise on the
    // float values, which is what the tie rule of DEC-005 sees.
    std::map<std::string, int64_t> seen;
    for (int64_t k = 0; k < K; ++k) {
        std::string key(reinterpret_cast<const char *>(&C0[static_cast<size_t>(k * D)]),
                        static_cast<size_t>(D) * sizeof(float));
        if (seen.count(key) != 0) {
            v.warnings.push_back("W-DUPC");
            break;
        }
        seen.emplace(std::move(key), k);
    }
    return v;
}

Validation validate_capacity(int64_t N, int64_t D, int64_t K, int64_t reduce_chunks,
                             uint64_t free_bytes)
{
    const MemoryModel m = device_memory_bytes(N, D, K, reduce_chunks);
    if (m.overflow) {
        // EDGE-30: index products near or above 2^63 are rejected by the checked arithmetic.
        return fail("VAL-009",
                    "required device memory overflows 64-bit arithmetic for N=" + istr(N) +
                        " D=" + istr(D) + " K=" + istr(K) + " G=" + istr(reduce_chunks),
                    kExitInvalidInput);
    }
    // A 5 % safety margin keeps allocation metadata and the context out of the way.
    const uint64_t budget = static_cast<uint64_t>(static_cast<double>(free_bytes) * 0.95);
    if (m.total > budget) {
        return fail("VAL-009",
                    "required " + ustr(m.total) + " B exceeds the usable " +
                        ustr(budget) + " B (95 % of " +
                        ustr(free_bytes) + " B free)",
                    kExitUnsupportedDevice);
    }
    return Validation{};
}

}  // namespace kmeans


In [ ]:
%%writefile io_kmd.h
// io_kmd.h — binary dataset and output files (requirements.md BLD-004, FR-002, FR-015).
//
// .kmd: magic "KMD1", uint32 version (= 1), uint64 rows, uint64 cols, then rows*cols binary32
//       little-endian in row-major order. Used for data and for centroids.
// .kma: magic "KMA1", uint32 version (= 1), uint64 N, then N int32 assignments.
//
// Every stream is opened with std::ios::binary. On Windows the default text mode translates byte
// 0x0A on write and 0x0D 0x0A on read, which silently corrupts float payloads containing those
// bytes, and the exact-size check below would then fail on a file that was written correctly.

#ifndef KMEANS_IO_KMD_H
#define KMEANS_IO_KMD_H

#include <cstdint>
#include <string>
#include <vector>

#include "common.h"

namespace kmeans {

struct Matrix {
    uint64_t rows = 0;
    uint64_t cols = 0;
    std::vector<float> data;  // row-major, rows*cols values
};

// BLD-004 requires little-endian on disk; the host must therefore be little-endian.
inline bool host_is_little_endian()
{
    const uint32_t one = 1;
    unsigned char bytes[sizeof(one)];
    std::memcpy(bytes, &one, sizeof(one));
    return bytes[0] == 1;
}

// Returns kExitOk, or kExitIoError for a missing, unreadable, truncated, over-long or
// structurally invalid file (bad magic or version). `err` receives a BLD-006 style message.
int read_kmd(const std::string &path, Matrix *out, std::string *err);
int read_kma(const std::string &path, std::vector<int32_t> *out, std::string *err);

// As read_kmd, plus the FR-002 check that the header dimensions equal the declared ones. A
// dimension mismatch is a validation failure (kExitInvalidInput), not an I/O failure.
int read_kmd_expect(const std::string &path, uint64_t rows, uint64_t cols, Matrix *out,
                    std::string *err);

int write_kmd(const std::string &path, uint64_t rows, uint64_t cols, const float *data,
              std::string *err);
int write_kma(const std::string &path, const std::vector<int32_t> &assignments, std::string *err);

}  // namespace kmeans

#endif  // KMEANS_IO_KMD_H


In [ ]:
%%writefile io_kmd.cpp
#include "io_kmd.h"

#include <fstream>

namespace kmeans {
namespace {

constexpr uint32_t kFormatVersion = 1;
constexpr size_t kKmdHeaderBytes = 4 + 4 + 8 + 8;  // magic, version, rows, cols
constexpr size_t kKmaHeaderBytes = 4 + 4 + 8;      // magic, version, N

int io_fail(std::string *err, const std::string &message)
{
    if (err != nullptr) {
        *err = message;
    }
    return kExitIoError;
}

// Opens the file, returns its exact size and leaves the stream positioned at the start.
bool open_sized(const std::string &path, std::ifstream *in, uint64_t *size)
{
    in->open(path, std::ios::binary | std::ios::ate);
    if (!in->is_open()) {
        return false;
    }
    const std::streampos end = in->tellg();
    if (end < 0) {
        return false;
    }
    *size = static_cast<uint64_t>(end);
    in->seekg(0, std::ios::beg);
    return true;
}

bool read_exact(std::ifstream *in, void *dst, size_t bytes)
{
    in->read(static_cast<char *>(dst), static_cast<std::streamsize>(bytes));
    return static_cast<size_t>(in->gcount()) == bytes && !in->bad();
}

}  // namespace

int read_kmd(const std::string &path, Matrix *out, std::string *err)
{
    if (!host_is_little_endian()) {
        return io_fail(err, "host is big-endian; BLD-004 fixes little-endian byte order on disk");
    }

    std::ifstream in;
    uint64_t file_size = 0;
    if (!open_sized(path, &in, &file_size)) {
        return io_fail(err, "cannot open '" + path + "' for reading");
    }
    if (file_size < kKmdHeaderBytes) {
        return io_fail(err, "'" + path + "' is " + std::to_string(file_size) +
                                " B, shorter than the " + std::to_string(kKmdHeaderBytes) +
                                " B .kmd header");
    }

    char magic[4] = {0, 0, 0, 0};
    uint32_t version = 0;
    uint64_t rows = 0;
    uint64_t cols = 0;
    if (!read_exact(&in, magic, 4) || !read_exact(&in, &version, 4) ||
        !read_exact(&in, &rows, 8) || !read_exact(&in, &cols, 8)) {
        return io_fail(err, "'" + path + "': header could not be read");
    }
    if (std::memcmp(magic, "KMD1", 4) != 0) {
        return io_fail(err, "'" + path + "': bad magic, expected KMD1");
    }
    if (version != kFormatVersion) {
        return io_fail(err, "'" + path + "': unsupported .kmd version " + std::to_string(version));
    }

    // FR-002: the file size must equal header + payload exactly, so a truncated file and a file
    // with trailing bytes are both rejected rather than silently accepted.
    uint64_t values = 0;
    uint64_t payload = 0;
    if (!checked_mul(rows, cols, &values) || !checked_mul(values, 4, &payload)) {
        return io_fail(err, "'" + path + "': header dimensions " + std::to_string(rows) + "x" +
                                std::to_string(cols) + " overflow the payload size");
    }
    const uint64_t expected = kKmdHeaderBytes + payload;
    if (file_size != expected) {
        return io_fail(err, "'" + path + "': size is " + std::to_string(file_size) +
                                " B but the header implies " + std::to_string(expected) + " B (" +
                                std::to_string(rows) + "x" + std::to_string(cols) + " binary32)");
    }

    out->rows = rows;
    out->cols = cols;
    out->data.assign(static_cast<size_t>(values), 0.0f);
    if (values != 0 && !read_exact(&in, out->data.data(), static_cast<size_t>(payload))) {
        return io_fail(err, "'" + path + "': payload could not be read");
    }
    return kExitOk;
}

int read_kma(const std::string &path, std::vector<int32_t> *out, std::string *err)
{
    if (!host_is_little_endian()) {
        return io_fail(err, "host is big-endian; BLD-004 fixes little-endian byte order on disk");
    }

    std::ifstream in;
    uint64_t file_size = 0;
    if (!open_sized(path, &in, &file_size)) {
        return io_fail(err, "cannot open '" + path + "' for reading");
    }
    if (file_size < kKmaHeaderBytes) {
        return io_fail(err, "'" + path + "' is shorter than the .kma header");
    }

    char magic[4] = {0, 0, 0, 0};
    uint32_t version = 0;
    uint64_t count = 0;
    if (!read_exact(&in, magic, 4) || !read_exact(&in, &version, 4) ||
        !read_exact(&in, &count, 8)) {
        return io_fail(err, "'" + path + "': header could not be read");
    }
    if (std::memcmp(magic, "KMA1", 4) != 0) {
        return io_fail(err, "'" + path + "': bad magic, expected KMA1");
    }
    if (version != kFormatVersion) {
        return io_fail(err, "'" + path + "': unsupported .kma version " + std::to_string(version));
    }

    uint64_t payload = 0;
    if (!checked_mul(count, 4, &payload)) {
        return io_fail(err, "'" + path + "': N=" + std::to_string(count) + " overflows the payload");
    }
    const uint64_t expected = kKmaHeaderBytes + payload;
    if (file_size != expected) {
        return io_fail(err, "'" + path + "': size is " + std::to_string(file_size) +
                                " B but the header implies " + std::to_string(expected) + " B");
    }

    out->assign(static_cast<size_t>(count), 0);
    if (count != 0 && !read_exact(&in, out->data(), static_cast<size_t>(payload))) {
        return io_fail(err, "'" + path + "': payload could not be read");
    }
    return kExitOk;
}

int read_kmd_expect(const std::string &path, uint64_t rows, uint64_t cols, Matrix *out,
                    std::string *err)
{
    const int code = read_kmd(path, out, err);
    if (code != kExitOk) {
        return code;
    }
    if (out->rows != rows || out->cols != cols) {
        if (err != nullptr) {
            *err = "'" + path + "' holds " + std::to_string(out->rows) + "x" +
                   std::to_string(out->cols) + " values but " + std::to_string(rows) + "x" +
                   std::to_string(cols) + " was declared";
        }
        return kExitInvalidInput;  // shape mismatch is a validation failure (FR-002)
    }
    return kExitOk;
}

int write_kmd(const std::string &path, uint64_t rows, uint64_t cols, const float *data,
              std::string *err)
{
    std::ofstream out(path, std::ios::binary | std::ios::trunc);
    if (!out.is_open()) {
        return io_fail(err, "cannot open '" + path + "' for writing");
    }
    const uint32_t version = kFormatVersion;
    out.write("KMD1", 4);
    out.write(reinterpret_cast<const char *>(&version), 4);
    out.write(reinterpret_cast<const char *>(&rows), 8);
    out.write(reinterpret_cast<const char *>(&cols), 8);

    uint64_t values = 0;
    if (!checked_mul(rows, cols, &values)) {
        return io_fail(err, "'" + path + "': dimensions overflow the payload size");
    }
    if (values != 0) {
        out.write(reinterpret_cast<const char *>(data),
                  static_cast<std::streamsize>(values * sizeof(float)));
    }
    out.flush();
    if (!out.good()) {
        return io_fail(err, "'" + path + "': write failed");
    }
    return kExitOk;
}

int write_kma(const std::string &path, const std::vector<int32_t> &assignments, std::string *err)
{
    std::ofstream out(path, std::ios::binary | std::ios::trunc);
    if (!out.is_open()) {
        return io_fail(err, "cannot open '" + path + "' for writing");
    }
    const uint32_t version = kFormatVersion;
    const uint64_t count = assignments.size();
    out.write("KMA1", 4);
    out.write(reinterpret_cast<const char *>(&version), 4);
    out.write(reinterpret_cast<const char *>(&count), 8);
    if (count != 0) {
        out.write(reinterpret_cast<const char *>(assignments.data()),
                  static_cast<std::streamsize>(count * sizeof(int32_t)));
    }
    out.flush();
    if (!out.good()) {
        return io_fail(err, "'" + path + "': write failed");
    }
    return kExitOk;
}

}  // namespace kmeans


In [ ]:
%%writefile cpu_reference.h
// cpu_reference.h — single-threaded CPU oracle (requirements.md NUM-003, ARC-03).
//
// Implements §2 exactly: the binary32 distance path of §2.2 with ascending-j accumulation, a
// binary64 path used only for near-tie diagnosis (NUM-005), binary64 sums in ascending i (§2.4),
// the lowest-index tie rule (DEC-005), empty-cluster retention (DEC-003), the convergence rule
// (DEC-002), iteration counting (DEC-006) and finalization (§2.8).
//
// The step functions are exposed individually, not only through run(), because §10.3 lockstep
// verification feeds the device's own state into each step and compares the result.
//
// Must be compiled with -ffp-contract=off and without fast-math: contracting a multiply-add in
// distance2_fp32 would change the oracle's rounding and make near-tie diagnosis meaningless.

#ifndef KMEANS_CPU_REFERENCE_H
#define KMEANS_CPU_REFERENCE_H

#include <cstdint>
#include <vector>

#include "common.h"

namespace kmeans {
namespace cpu {

// §2.2 in binary32: terms and accumulation in binary32, ascending j, never reordered.
float distance2_fp32(const float *x, const float *c, int64_t D);

// The same quantity in binary64, for NUM-005 margins and for the objective (§2.9). Not used for
// assignment: that must see exactly what the device sees.
double distance2_fp64(const float *x, const float *c, int64_t D);

// §2.3 for one point: nearest centroid by binary32 distance, lowest index among exactly equal
// computed values (strict-less-than replacement in ascending k).
int32_t nearest_centroid(const float *x, int64_t D, const float *C, int64_t K);

// §2.3 for all points, writing A. Returns the number of points whose assignment differs from
// A_prev (FR-007). With A_prev == nullptr this is the initial pass and the return value is N,
// which the driver ignores because changes are only counted inside the loop.
int64_t assign(const float *X, int64_t N, int64_t D, const float *C, int64_t K, int32_t *A,
               const int32_t *A_prev);

// NUM-005 diagnosis for one point, in binary64.
struct TieInfo {
    double d1 = 0.0;      // smallest binary64 distance
    double d2 = 0.0;      // second smallest
    double margin = 0.0;  // (d2 - d1) / max(d2, FLT_MIN)
    int32_t k1 = 0;       // index attaining d1 (lowest index on ties)
    int32_t k2 = 0;       // index attaining d2
    bool near_tie = false;
};
TieInfo tie_info(const float *x, int64_t D, const float *C, int64_t K);

// §2.4: counts exactly, sums in binary64 in ascending i. S has K*D entries, n has K.
void accumulate(const float *X, int64_t N, int64_t D, const int32_t *A, int64_t K, double *S,
                uint32_t *n);

// §2.5: C_next[k] = fl32(S[k]/n[k]) for n[k] > 0, else C_cur[k] retained bit-exactly. No division
// is performed when n[k] == 0. C_next must differ from C_cur (double buffering, FR-010).
void update(const double *S, const uint32_t *n, int64_t K, int64_t D, const float *C_cur,
            float *C_next);

// §2.6: max over clusters with n[k] > 0 of the binary64 Euclidean displacement. Empty clusters
// contribute delta_k = 0 by construction and are excluded from the maximum.
double movement(const float *C_cur, const float *C_next, const uint32_t *n, int64_t K, int64_t D);

// §2.9: J in binary64, computed after the run and outside any timed region (FR-016).
double objective(const float *X, int64_t N, int64_t D, const int32_t *A, const float *C);

// §2.8 steps 1 and 2: one accumulation pass over A_final, then the P2 check against NUM-006.
// C_final is never modified. `m_range` is the M of VAL-007, which fixes the NUM-006 scale.
struct Finalization {
    std::vector<uint32_t> counts;       // n_final
    std::vector<float> means;           // fl32(S/n) for non-empty k, C_final retained otherwise
    bool centroid_consistent = false;   // P2
    double max_centroid_difference = 0.0;
};
Finalization finalize(const float *X, int64_t N, int64_t D, const int32_t *A_final,
                      const float *C_final, int64_t K, double m_range);

// §2.7 driver. `fixed_iters` (FR-020) disables both convergence criteria so exactly max_iter
// updates run and the status is ITERATION_LIMIT.
Result run(const float *X, int64_t N, int64_t D, const float *C0, int64_t K, int64_t max_iter,
           double tol, double m_range, bool fixed_iters);

}  // namespace cpu
}  // namespace kmeans

#endif  // KMEANS_CPU_REFERENCE_H


In [ ]:
%%writefile cpu_reference.cpp
#include "cpu_reference.h"

#include <cfloat>
#include <cmath>

namespace kmeans {
namespace cpu {

float distance2_fp32(const float *x, const float *c, int64_t D)
{
    // NUM-002: each term is formed in binary32 and accumulated in binary32 in ascending j,
    // starting from 0. The order is never changed, and -ffp-contract=off keeps the compiler from
    // fusing the multiply and add, so this is the exact value the device must reproduce.
    float acc = 0.0f;
    for (int64_t j = 0; j < D; ++j) {
        const float diff = x[j] - c[j];
        acc += diff * diff;
    }
    return acc;
}

double distance2_fp64(const float *x, const float *c, int64_t D)
{
    double acc = 0.0;
    for (int64_t j = 0; j < D; ++j) {
        const double diff = static_cast<double>(x[j]) - static_cast<double>(c[j]);
        acc += diff * diff;
    }
    return acc;
}

int32_t nearest_centroid(const float *x, int64_t D, const float *C, int64_t K)
{
    // DEC-005: start at k = 0 and replace only on strictly smaller distance, so exactly equal
    // computed binary32 distances keep the lowest index.
    int32_t best_k = 0;
    float best_d = distance2_fp32(x, C, D);
    for (int64_t k = 1; k < K; ++k) {
        const float d = distance2_fp32(x, C + k * D, D);
        if (d < best_d) {
            best_d = d;
            best_k = static_cast<int32_t>(k);
        }
    }
    return best_k;
}

int64_t assign(const float *X, int64_t N, int64_t D, const float *C, int64_t K, int32_t *A,
               const int32_t *A_prev)
{
    int64_t changed = 0;
    for (int64_t i = 0; i < N; ++i) {
        const int32_t k = nearest_centroid(X + i * D, D, C, K);
        if (A_prev != nullptr && A_prev[i] != k) {
            ++changed;
        }
        A[i] = k;
    }
    return A_prev == nullptr ? N : changed;
}

TieInfo tie_info(const float *x, int64_t D, const float *C, int64_t K)
{
    TieInfo info;
    info.d1 = HUGE_VAL;
    info.d2 = HUGE_VAL;
    info.k1 = 0;
    info.k2 = K > 1 ? 1 : 0;

    for (int64_t k = 0; k < K; ++k) {
        const double d = distance2_fp64(x, C + k * D, D);
        if (d < info.d1) {
            info.d2 = info.d1;
            info.k2 = info.k1;
            info.d1 = d;
            info.k1 = static_cast<int32_t>(k);
        } else if (d < info.d2) {
            info.d2 = d;
            info.k2 = static_cast<int32_t>(k);
        }
    }

    if (K < 2) {
        // With one centroid there is no competitor, so no point can be a near-tie.
        info.d2 = info.d1;
        info.margin = HUGE_VAL;
        info.near_tie = false;
        return info;
    }
    const double denom = info.d2 > static_cast<double>(FLT_MIN) ? info.d2
                                                                : static_cast<double>(FLT_MIN);
    info.margin = (info.d2 - info.d1) / denom;
    info.near_tie = info.margin <= near_tie_threshold(D);
    return info;
}

void accumulate(const float *X, int64_t N, int64_t D, const int32_t *A, int64_t K, double *S,
                uint32_t *n)
{
    for (int64_t e = 0; e < K * D; ++e) {
        S[e] = 0.0;
    }
    for (int64_t k = 0; k < K; ++k) {
        n[k] = 0;
    }
    // §2.4: ascending i, binary64 sums, exact integer counts. The reference fixes this order so
    // that Approach B (which also has a fixed order) can be compared against it meaningfully.
    for (int64_t i = 0; i < N; ++i) {
        const int64_t k = A[i];
        n[k] += 1;
        const float *row = X + i * D;
        double *sums = S + k * D;
        for (int64_t j = 0; j < D; ++j) {
            sums[j] += static_cast<double>(row[j]);
        }
    }
}

void update(const double *S, const uint32_t *n, int64_t K, int64_t D, const float *C_cur,
            float *C_next)
{
    for (int64_t k = 0; k < K; ++k) {
        const float *cur = C_cur + k * D;
        float *next = C_next + k * D;
        if (n[k] == 0) {
            // DEC-003: retain the previous centroid bit-exactly; no division is performed.
            for (int64_t j = 0; j < D; ++j) {
                next[j] = cur[j];
            }
            continue;
        }
        const double count = static_cast<double>(n[k]);
        const double *sums = S + k * D;
        for (int64_t j = 0; j < D; ++j) {
            // §2.5: divide in binary64 and round the quotient once to binary32.
            next[j] = static_cast<float>(sums[j] / count);
        }
    }
}

double movement(const float *C_cur, const float *C_next, const uint32_t *n, int64_t K, int64_t D)
{
    double max_delta = 0.0;
    for (int64_t k = 0; k < K; ++k) {
        if (n[k] == 0) {
            continue;  // §2.6: empty clusters have delta_k = 0 and are excluded from the maximum
        }
        const float *cur = C_cur + k * D;
        const float *next = C_next + k * D;
        double sum = 0.0;
        for (int64_t j = 0; j < D; ++j) {
            const double diff = static_cast<double>(next[j]) - static_cast<double>(cur[j]);
            sum += diff * diff;
        }
        const double delta = std::sqrt(sum);
        if (delta > max_delta) {
            max_delta = delta;
        }
    }
    return max_delta;
}

double objective(const float *X, int64_t N, int64_t D, const int32_t *A, const float *C)
{
    double j_total = 0.0;
    for (int64_t i = 0; i < N; ++i) {
        j_total += distance2_fp64(X + i * D, C + static_cast<int64_t>(A[i]) * D, D);
    }
    return j_total;
}

Finalization finalize(const float *X, int64_t N, int64_t D, const int32_t *A_final,
                      const float *C_final, int64_t K, double m_range)
{
    Finalization out;
    out.counts.assign(static_cast<size_t>(K), 0);
    out.means.assign(static_cast<size_t>(K * D), 0.0f);

    std::vector<double> S(static_cast<size_t>(K * D), 0.0);
    accumulate(X, N, D, A_final, K, S.data(), out.counts.data());
    update(S.data(), out.counts.data(), K, D, C_final, out.means.data());

    // §2.8 step 2: P2 holds when every non-empty cluster's reported centroid agrees with the mean
    // of its assigned points within NUM-006. Empty clusters are not evaluated: their centroid is
    // retained by DEC-003 and is not a mean of anything.
    const double s_max = s_max_from_range(m_range);
    bool consistent = true;
    double worst = 0.0;
    for (int64_t k = 0; k < K; ++k) {
        if (out.counts[static_cast<size_t>(k)] == 0) {
            continue;
        }
        for (int64_t j = 0; j < D; ++j) {
            const size_t e = static_cast<size_t>(k * D + j);
            const float reported = C_final[e];
            const float mean = out.means[e];
            const double diff =
                std::fabs(static_cast<double>(reported) - static_cast<double>(mean));
            if (diff > worst) {
                worst = diff;
            }
            if (!centroid_agrees(reported, mean, s_max)) {
                consistent = false;
            }
        }
    }
    out.centroid_consistent = consistent;
    out.max_centroid_difference = worst;
    return out;
}

Result run(const float *X, int64_t N, int64_t D, const float *C0, int64_t K, int64_t max_iter,
           double tol, double m_range, bool fixed_iters)
{
    Result result;
    std::vector<float> c_cur(C0, C0 + K * D);
    std::vector<float> c_next(static_cast<size_t>(K * D), 0.0f);
    std::vector<int32_t> a_cur(static_cast<size_t>(N), 0);
    std::vector<int32_t> a_prev(static_cast<size_t>(N), 0);
    std::vector<double> sums(static_cast<size_t>(K * D), 0.0);
    std::vector<uint32_t> counts(static_cast<size_t>(K), 0);

    // §2.7: one assignment pass before the loop, so A is always the assignment of the current C.
    assign(X, N, D, c_cur.data(), K, a_cur.data(), nullptr);

    int64_t t = 0;
    Status status = Status::ITERATION_LIMIT;
    double last_delta = 0.0;
    bool has_last_delta = false;

    while (t < max_iter) {
        a_prev = a_cur;

        accumulate(X, N, D, a_cur.data(), K, sums.data(), counts.data());
        update(sums.data(), counts.data(), K, D, c_cur.data(), c_next.data());
        const double delta = movement(c_cur.data(), c_next.data(), counts.data(), K, D);
        c_cur.swap(c_next);  // FR-010: double buffering, pointers swapped after the update

        ++t;  // DEC-006: the iteration count is the number of completed centroid updates
        last_delta = delta;
        has_last_delta = true;

        const int64_t changed = assign(X, N, D, c_cur.data(), K, a_cur.data(), a_prev.data());

        if (fixed_iters) {
            continue;  // FR-020: both criteria disabled, so exactly max_iter updates run
        }
        // DEC-002: assignment stability is checked first and takes reporting precedence.
        if (changed == 0) {
            status = Status::CONVERGED_ASSIGNMENTS;
            break;
        }
        if (delta <= tol) {
            status = Status::CONVERGED_MOVEMENT;  // inclusive comparison (ENV-10)
            break;
        }
    }

    const Finalization fin = finalize(X, N, D, a_cur.data(), c_cur.data(), K, m_range);

    result.assignments = a_cur;
    result.centroids = c_cur;  // §2.8: finalization never replaces C_final
    result.counts = fin.counts;
    result.iterations = t;
    result.status = status;
    result.last_delta = last_delta;
    result.has_last_delta = has_last_delta;
    result.centroid_consistent = fin.centroid_consistent;
    result.assignment_consistent = true;  // P1 holds by construction
    result.objective = objective(X, N, D, a_cur.data(), c_cur.data());
    return result;
}

}  // namespace cpu
}  // namespace kmeans


In [ ]:
%%writefile gen.h
// gen.h — deterministic Gaussian-blob dataset generator (requirements.md FR-003, ARC-02).
//
// Uses the raw output of std::mt19937_64 plus a self-implemented Box-Muller transform. Standard
// library distributions (std::normal_distribution, std::uniform_int_distribution) are deliberately
// not used: their algorithms are unspecified, so the same seed can yield different numbers on a
// different standard library, and FR-003 requires identical parameters to reproduce bit-identical
// files.

#ifndef KMEANS_GEN_H
#define KMEANS_GEN_H

#include <cstdint>
#include <random>
#include <string>
#include <vector>

namespace kmeans {

struct GenParams {
    int64_t N = 0;
    int64_t D = 0;
    int64_t K = 0;        // K_true = K (PERF-003)
    double sep = 10.0;    // scale of the cluster-centre spread
    double spread = 1.0;  // standard deviation of the points around their centre
    uint64_t seed = 0;
};

// Raw-output wrapper. Every draw below consumes a documented, fixed number of 64-bit words, so the
// generation order fixes the stream and the output is reproducible.
class Rng {
public:
    explicit Rng(uint64_t seed) : engine_(seed) {}

    // Uniform in the open interval (0, 1): the top 53 bits plus a half-step, so the result is
    // never exactly 0 (which would make std::log in Box-Muller return -inf) and never exactly 1.
    double uniform_open();

    // Standard normal via Box-Muller. Both values of each pair are used, so two normals cost one
    // pair of uniforms; the second is cached.
    double normal();

    // Uniform integer in [0, bound) by rejection, which is unbiased for every bound.
    uint64_t bounded(uint64_t bound);

private:
    std::mt19937_64 engine_;
    bool has_cached_normal_ = false;
    double cached_normal_ = 0.0;
};

struct Dataset {
    std::vector<float> X;    // N x D row-major
    std::vector<float> C0;   // K x D row-major
    std::string dataset_id;
    double min_centre_distance = 0.0;  // smallest pairwise centre distance, measured not assumed
    double spread = 0.0;               // echoed so the caller can report separation in sigma units
};

// Identifier derived from every generator parameter (FR-003), so a dataset is traceable from a
// benchmark CSV row back to the exact inputs that produced it.
std::string dataset_id(const GenParams &p);

// Generation order, fixed by FR-003 and by this function:
//   1. K centre vectors, each coordinate sep * normal()
//   2. N points: point i belongs to centre i % K and gets spread * normal() per coordinate
//   3. a seeded Fisher-Yates permutation of [0, N)
//   4. initial centroids: the points at permutation indices floor(q*N/K), q in [0, K)
Dataset generate(const GenParams &p);

}  // namespace kmeans

#endif  // KMEANS_GEN_H


In [ ]:
%%writefile gen.cpp
#include "gen.h"

#include <cmath>
#include <cstdio>

namespace kmeans {
namespace {

constexpr double kTwoPi = 6.283185307179586476925286766559;

// FNV-1a over the parameter string, so a dataset ID is short but still parameter-specific.
uint64_t fnv1a(const std::string &text)
{
    uint64_t hash = 1469598103934665603ull;
    for (unsigned char c : text) {
        hash ^= static_cast<uint64_t>(c);
        hash *= 1099511628211ull;
    }
    return hash;
}

}  // namespace

double Rng::uniform_open()
{
    // 53 significant bits is the most a binary64 can hold; adding half a step moves the value off
    // both endpoints without biasing the distribution's mean.
    const uint64_t raw = engine_();
    return (static_cast<double>(raw >> 11) + 0.5) * (1.0 / 9007199254740992.0);  // 2^-53
}

double Rng::normal()
{
    if (has_cached_normal_) {
        has_cached_normal_ = false;
        return cached_normal_;
    }
    const double u1 = uniform_open();
    const double u2 = uniform_open();
    const double radius = std::sqrt(-2.0 * std::log(u1));
    const double angle = kTwoPi * u2;
    cached_normal_ = radius * std::sin(angle);
    has_cached_normal_ = true;
    return radius * std::cos(angle);
}

uint64_t Rng::bounded(uint64_t bound)
{
    if (bound <= 1) {
        return 0;
    }
    // Rejection sampling: discard the incomplete final block of the 2^64 range so every value in
    // [0, bound) is equally likely. std::uniform_int_distribution would do this too, but its
    // algorithm is implementation-defined and FR-003 needs a fixed one.
    const uint64_t limit = UINT64_MAX - (UINT64_MAX % bound) - 1;
    uint64_t raw = engine_();
    while (raw > limit) {
        raw = engine_();
    }
    return raw % bound;
}

std::string dataset_id(const GenParams &p)
{
    char buf[256];
    std::snprintf(buf, sizeof(buf), "n%lld_d%lld_k%lld_sep%.6g_spread%.6g_seed%llu",
                  static_cast<long long>(p.N), static_cast<long long>(p.D),
                  static_cast<long long>(p.K), p.sep, p.spread,
                  static_cast<unsigned long long>(p.seed));
    const std::string base(buf);
    char suffix[32];
    std::snprintf(suffix, sizeof(suffix), "-%016llx",
                  static_cast<unsigned long long>(fnv1a(base)));
    return base + suffix;
}

Dataset generate(const GenParams &p)
{
    Dataset out;
    out.dataset_id = dataset_id(p);
    out.spread = p.spread;
    const int64_t N = p.N;
    const int64_t D = p.D;
    const int64_t K = p.K;

    Rng rng(p.seed);

    // 1. Cluster centres. Drawing them from the same normal stream keeps the generator simple and
    // deterministic; the realised separation is measured below rather than assumed, because
    // §10.3 calls a dataset well-separated only at separation >= 10 * spread.
    std::vector<double> centres(static_cast<size_t>(K * D), 0.0);
    for (int64_t k = 0; k < K; ++k) {
        for (int64_t j = 0; j < D; ++j) {
            centres[static_cast<size_t>(k * D + j)] = p.sep * rng.normal();
        }
    }

    // 2. Points: point i belongs to centre i % K, which spreads the clusters evenly without
    // consuming randomness for the membership choice.
    out.X.assign(static_cast<size_t>(N * D), 0.0f);
    for (int64_t i = 0; i < N; ++i) {
        const int64_t k = i % K;
        for (int64_t j = 0; j < D; ++j) {
            const double value = centres[static_cast<size_t>(k * D + j)] + p.spread * rng.normal();
            out.X[static_cast<size_t>(i * D + j)] = static_cast<float>(value);
        }
    }

    // 3. Fisher-Yates permutation of the point indices, descending as in the classical algorithm.
    std::vector<int64_t> perm(static_cast<size_t>(N));
    for (int64_t i = 0; i < N; ++i) {
        perm[static_cast<size_t>(i)] = i;
    }
    for (int64_t i = N - 1; i > 0; --i) {
        const uint64_t j = rng.bounded(static_cast<uint64_t>(i) + 1);
        const int64_t tmp = perm[static_cast<size_t>(i)];
        perm[static_cast<size_t>(i)] = perm[static_cast<size_t>(j)];
        perm[static_cast<size_t>(j)] = tmp;
    }

    // 4. FR-003: the initial centroids are the points at permutation indices floor(q*N/K). With
    // K > N the same point can be chosen more than once, which VAL-008 then flags as W-DUPC.
    out.C0.assign(static_cast<size_t>(K * D), 0.0f);
    for (int64_t q = 0; q < K; ++q) {
        const int64_t slot = static_cast<int64_t>((static_cast<uint64_t>(q) *
                                                   static_cast<uint64_t>(N)) /
                                                  static_cast<uint64_t>(K));
        const int64_t point = perm[static_cast<size_t>(slot % N)];
        for (int64_t j = 0; j < D; ++j) {
            out.C0[static_cast<size_t>(q * D + j)] = out.X[static_cast<size_t>(point * D + j)];
        }
    }

    // Measured minimum pairwise centre distance, in the generated binary32 values' own units.
    double min_sq = -1.0;
    for (int64_t a = 0; a < K; ++a) {
        for (int64_t b = a + 1; b < K; ++b) {
            double sum = 0.0;
            for (int64_t j = 0; j < D; ++j) {
                const double diff = centres[static_cast<size_t>(a * D + j)] -
                                    centres[static_cast<size_t>(b * D + j)];
                sum += diff * diff;
            }
            if (min_sq < 0.0 || sum < min_sq) {
                min_sq = sum;
            }
        }
    }
    out.min_centre_distance = min_sq < 0.0 ? 0.0 : std::sqrt(min_sq);
    return out;
}

}  // namespace kmeans


In [ ]:
%%writefile gen_main.cpp
// gen_main.cpp — the kmeans-gen executable (requirements.md BLD-003a, FR-003).
//
//   kmeans-gen --n N --d D --k K --sep S --spread SIGMA --seed SEED --out-prefix PATH
//
// Writes PATH_X.kmd (N x D) and PATH_C0.kmd (K x D) in the BLD-004 format and prints a one-line
// JSON summary including the dataset ID and the measured minimum centre separation, so a benchmark
// row can be traced back to the exact dataset (PERF-008) and the "well-separated" claim of §10.3
// is a measurement rather than an assumption.

#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <string>

#include "common.h"
#include "gen.h"
#include "io_kmd.h"
#include "validate.h"

namespace {

void usage()
{
    std::fprintf(stderr,
                 "usage: kmeans-gen --n N --d D --k K --sep S --spread SIGMA --seed SEED "
                 "--out-prefix PATH\n");
}

bool parse_int64(const char *text, int64_t *out)
{
    char *end = nullptr;
    const long long value = std::strtoll(text, &end, 10);
    if (end == text || *end != '\0') {
        return false;
    }
    *out = static_cast<int64_t>(value);
    return true;
}

bool parse_uint64(const char *text, uint64_t *out)
{
    char *end = nullptr;
    const unsigned long long value = std::strtoull(text, &end, 10);
    if (end == text || *end != '\0') {
        return false;
    }
    *out = static_cast<uint64_t>(value);
    return true;
}

bool parse_double(const char *text, double *out)
{
    char *end = nullptr;
    const double value = std::strtod(text, &end);
    if (end == text || *end != '\0') {
        return false;
    }
    *out = value;
    return true;
}

}  // namespace

int main(int argc, char **argv)
{
    using namespace kmeans;

    GenParams params;
    std::string out_prefix;
    bool have_n = false, have_d = false, have_k = false;

    for (int i = 1; i < argc; ++i) {
        const char *flag = argv[i];
        const bool needs_value = std::strncmp(flag, "--", 2) == 0;
        if (!needs_value) {
            report_error("VAL-010", "unexpected argument '" + std::string(flag) + "'");
            usage();
            return kExitInvalidInput;
        }
        if (i + 1 >= argc) {
            report_error("VAL-010", "option '" + std::string(flag) + "' needs a value");
            usage();
            return kExitInvalidInput;
        }
        const char *value = argv[++i];
        bool parsed = true;
        if (std::strcmp(flag, "--n") == 0) {
            parsed = parse_int64(value, &params.N);
            have_n = true;
        } else if (std::strcmp(flag, "--d") == 0) {
            parsed = parse_int64(value, &params.D);
            have_d = true;
        } else if (std::strcmp(flag, "--k") == 0) {
            parsed = parse_int64(value, &params.K);
            have_k = true;
        } else if (std::strcmp(flag, "--sep") == 0) {
            parsed = parse_double(value, &params.sep);
        } else if (std::strcmp(flag, "--spread") == 0) {
            parsed = parse_double(value, &params.spread);
        } else if (std::strcmp(flag, "--seed") == 0) {
            parsed = parse_uint64(value, &params.seed);
        } else if (std::strcmp(flag, "--out-prefix") == 0) {
            out_prefix = value;
        } else {
            report_error("VAL-010", "unknown option '" + std::string(flag) + "'");
            usage();
            return kExitInvalidInput;
        }
        if (!parsed) {
            report_error("VAL-010", "value '" + std::string(value) + "' for option '" +
                                        std::string(flag) + "' could not be parsed");
            return kExitInvalidInput;
        }
    }

    if (!have_n || !have_d || !have_k || out_prefix.empty()) {
        report_error("VAL-010", "--n, --d, --k and --out-prefix are required");
        usage();
        return kExitInvalidInput;
    }

    // The generator's sizes obey the same VAL-001/VAL-002 limits as a clustering run, so an
    // impossible dataset is refused before any memory is touched.
    const Validation params_ok = validate_parameters(params.N, params.D, params.K, 0, 0.0);
    if (!params_ok.ok) {
        params_ok.report();
        return params_ok.exit_code;
    }
    if (!(params.spread >= 0.0) || !(params.sep >= 0.0)) {
        report_error("VAL-010", "--sep and --spread must be non-negative and not NaN");
        return kExitInvalidInput;
    }

    const Dataset data = generate(params);

    std::string err;
    const std::string x_path = out_prefix + "_X.kmd";
    const std::string c_path = out_prefix + "_C0.kmd";
    int code = write_kmd(x_path, static_cast<uint64_t>(params.N), static_cast<uint64_t>(params.D),
                         data.X.data(), &err);
    if (code != kExitOk) {
        report_error("FR-002", err);
        return code;
    }
    code = write_kmd(c_path, static_cast<uint64_t>(params.K), static_cast<uint64_t>(params.D),
                     data.C0.data(), &err);
    if (code != kExitOk) {
        report_error("FR-002", err);
        return code;
    }

    const double sigma_ratio =
        params.spread > 0.0 ? data.min_centre_distance / params.spread : 0.0;
    std::printf("{\"dataset_id\": \"%s\", \"N\": %lld, \"D\": %lld, \"K\": %lld, \"sep\": %.17g, "
                "\"spread\": %.17g, \"seed\": %llu, \"min_centre_distance\": %.17g, "
                "\"min_separation_sigma\": %.17g, \"well_separated\": %s, \"data\": \"%s\", "
                "\"centroids\": \"%s\"}\n",
                data.dataset_id.c_str(), static_cast<long long>(params.N),
                static_cast<long long>(params.D), static_cast<long long>(params.K), params.sep,
                params.spread, static_cast<unsigned long long>(params.seed),
                data.min_centre_distance, sigma_ratio, sigma_ratio >= 10.0 ? "true" : "false",
                x_path.c_str(), c_path.c_str());
    return kExitOk;
}


In [ ]:
%%writefile hand_data.h
// hand_data.h — the hand-verifiable datasets H1-H6 and H2-a..H2-e of requirements.md §16.0,
// together with their expected outcomes.
//
// These are the oracle for the oracle: every value here is derived by hand in the specification,
// so a CPU reference that reproduces them is trustworthy enough to judge the device against
// (TEST-I01..I06 reuse exactly these cases on the GPU). All values except 5/3 are exactly
// representable in binary32 and are compared exactly; 5/3 is compared with NUM-006.

#ifndef KMEANS_TESTS_HAND_DATA_H
#define KMEANS_TESTS_HAND_DATA_H

#include <cstdint>
#include <string>
#include <vector>

#include "common.h"

namespace kmeans {
namespace tests {

struct HandCase {
    std::string id;
    int64_t N = 0;
    int64_t D = 0;
    int64_t K = 0;
    std::vector<float> X;
    std::vector<float> C0;
    int64_t max_iter = 10;
    double tol = 0.0;

    // Expected outcome (§16.0).
    std::vector<int32_t> expected_A;
    std::vector<float> expected_C;
    std::vector<uint32_t> expected_n;
    int64_t expected_iterations = 0;
    Status expected_status = Status::ERROR;
    bool expected_centroid_consistent = false;
    bool expect_warning_kgtn = false;  // VAL-003
    bool expect_warning_dupc = false;  // VAL-008
    bool centroids_exact = true;       // false where 5/3 appears, compared via NUM-006
    std::string note;
};

// The 1-D H2 point set, reused by H2 and its five variants.
inline std::vector<float> h2_points()
{
    return {0.0f, 2.0f, 3.0f, 10.0f};
}

inline std::vector<float> h2_initial_centroids()
{
    return {0.0f, 3.0f};
}

inline std::vector<HandCase> hand_cases()
{
    std::vector<HandCase> cases;

    // H1: two well-separated pairs. A(0) = {0,0,1,1}; C(1) = {0.5, 9.5}; A(1) = A(0).
    {
        HandCase c;
        c.id = "H1";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = {0.0f, 1.0f, 9.0f, 10.0f};
        c.C0 = {0.0f, 10.0f};
        c.expected_A = {0, 0, 1, 1};
        c.expected_C = {0.5f, 9.5f};
        c.expected_n = {2, 2};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        cases.push_back(c);
    }

    // H2: four updates of drift. A(0)={0,1,1,1}; C(1)={0,5} (delta 2); A(1)={0,0,1,1};
    // C(2)={1,6.5} (delta 1.5); A(2)={0,0,0,1}; C(3)={5/3,10} (delta 3.5); A(3)=A(2).
    {
        HandCase c;
        c.id = "H2";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.expected_A = {0, 0, 0, 1};
        c.expected_C = {5.0f / 3.0f, 10.0f};
        c.expected_n = {3, 1};
        c.expected_iterations = 3;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        c.centroids_exact = false;  // 5/3 is not representable in binary32 (NUM-006)
        cases.push_back(c);
    }

    // H2-a: tolerance 2 equals the first delta exactly, so the inclusive comparison of DEC-002
    // stops after one update. P2 is false: cluster 0 now holds {0,2}, whose mean is 1, not 0.
    {
        HandCase c;
        c.id = "H2-a";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.tol = 2.0;
        c.expected_A = {0, 0, 1, 1};
        c.expected_C = {0.0f, 5.0f};
        c.expected_n = {2, 2};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_MOVEMENT;
        c.expected_centroid_consistent = false;
        c.note = "delta == tol exactly: inclusive comparison (ENV-10)";
        cases.push_back(c);
    }

    // H2-b: tolerance 1.5 equals the second delta exactly.
    {
        HandCase c;
        c.id = "H2-b";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.tol = 1.5;
        c.expected_A = {0, 0, 0, 1};
        c.expected_C = {1.0f, 6.5f};
        c.expected_n = {3, 1};
        c.expected_iterations = 2;
        c.expected_status = Status::CONVERGED_MOVEMENT;
        c.expected_centroid_consistent = false;
        cases.push_back(c);
    }

    // H2-c: just below that delta, so the run continues to assignment stability as in H2.
    {
        HandCase c;
        c.id = "H2-c";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.tol = 1.4999;
        c.expected_A = {0, 0, 0, 1};
        c.expected_C = {5.0f / 3.0f, 10.0f};
        c.expected_n = {3, 1};
        c.expected_iterations = 3;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        c.centroids_exact = false;
        cases.push_back(c);
    }

    // H2-d: the limit cuts the run short; termination is not reported as convergence.
    {
        HandCase c;
        c.id = "H2-d";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.max_iter = 2;
        c.expected_A = {0, 0, 0, 1};
        c.expected_C = {1.0f, 6.5f};
        c.expected_n = {3, 1};
        c.expected_iterations = 2;
        c.expected_status = Status::ITERATION_LIMIT;
        c.expected_centroid_consistent = false;
        cases.push_back(c);
    }

    // H2-e: max_iter = 0 runs the initial assignment pass only (§2.7 boundary behaviour).
    {
        HandCase c;
        c.id = "H2-e";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.max_iter = 0;
        c.expected_A = {0, 1, 1, 1};
        c.expected_C = {0.0f, 3.0f};
        c.expected_n = {1, 3};
        c.expected_iterations = 0;
        c.expected_status = Status::ITERATION_LIMIT;
        c.expected_centroid_consistent = false;  // cluster 1 holds {2,3,10}, mean 5, not 3
        cases.push_back(c);
    }

    // H3: one point exactly between two centroids. The tie goes to the lower index (DEC-005) and
    // cluster 1 is empty from the start, so it retains its initial centroid (DEC-003).
    {
        HandCase c;
        c.id = "H3";
        c.N = 1; c.D = 1; c.K = 2;
        c.X = {1.0f};
        c.C0 = {0.0f, 2.0f};
        c.expected_A = {0};
        c.expected_C = {1.0f, 2.0f};
        c.expected_n = {1, 0};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;  // the only non-empty cluster holds its own mean
        // K = 2 > N = 1, so VAL-003 applies here as well. The §16.0 row for H3 does not mention
        // the warning, but the rule is normative and unconditional.
        c.expect_warning_kgtn = true;
        c.note = "exact tie -> lowest index; empty cluster retained and excluded from delta";
        cases.push_back(c);
    }

    // H4: duplicate initial centroids. Every point ties and goes to index 0 (W-DUPC).
    {
        HandCase c;
        c.id = "H4";
        c.N = 2; c.D = 1; c.K = 2;
        c.X = {0.0f, 1.0f};
        c.C0 = {5.0f, 5.0f};
        c.expected_A = {0, 0};
        c.expected_C = {0.5f, 5.0f};
        c.expected_n = {2, 0};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        c.expect_warning_dupc = true;
        cases.push_back(c);
    }

    // H5: K > N, so at least K - N clusters stay empty for the whole run (VAL-003, W-KGTN).
    {
        HandCase c;
        c.id = "H5";
        c.N = 1; c.D = 1; c.K = 3;
        c.X = {4.0f};
        c.C0 = {0.0f, 1.0f, 2.0f};
        c.expected_A = {2};
        c.expected_C = {0.0f, 1.0f, 4.0f};
        c.expected_n = {0, 0, 1};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        c.expect_warning_kgtn = true;
        cases.push_back(c);
    }

    // H6: the 2-D case, which catches a transposed or stride-confused layout that every 1-D case
    // above would pass.
    {
        HandCase c;
        c.id = "H6";
        c.N = 4; c.D = 2; c.K = 2;
        c.X = {0.0f, 0.0f, 0.0f, 1.0f, 10.0f, 0.0f, 10.0f, 1.0f};
        c.C0 = {0.0f, 0.0f, 10.0f, 0.0f};
        c.expected_A = {0, 0, 1, 1};
        c.expected_C = {0.0f, 0.5f, 10.0f, 0.5f};
        c.expected_n = {2, 2};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        cases.push_back(c);
    }

    return cases;
}

}  // namespace tests
}  // namespace kmeans

#endif  // KMEANS_TESTS_HAND_DATA_H


In [ ]:
%%writefile test_cpu.cpp
// test_cpu.cpp — CPU-level test driver for milestone M1.
//
// Covers requirements.md §16.0 (H1-H6 and H2-a..e against the CPU reference), §16.3 (the FP32 and
// FP64 distance paths agree except on near-ties), TEST-U08 (movement and the inclusive tolerance
// boundary), TEST-U09 (empty clusters), TEST-U11 (size and overflow arithmetic), TEST-U12 (file
// I/O) and TEST-U13 (generator determinism).
//
// One executable, non-zero exit on any failure, so a single notebook cell can gate the run
// (NFR-015). GPU-level versions of these cases arrive in later milestones.

#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <string>
#include <vector>

#include "common.h"
#include "cpu_reference.h"
#include "gen.h"
#include "hand_data.h"
#include "io_kmd.h"
#include "launch_config.h"
#include "validate.h"

using namespace kmeans;

namespace {

int g_checks = 0;
int g_failures = 0;
int g_case_failures = 0;

void check(bool condition, const std::string &what)
{
    ++g_checks;
    if (!condition) {
        ++g_failures;
        ++g_case_failures;
        std::printf("      FAIL  %s\n", what.c_str());
    }
}

void check_eq_i64(int64_t got, int64_t want, const std::string &what)
{
    check(got == want, what + ": got " + std::to_string(got) + ", want " + std::to_string(want));
}

std::string float_str(float v)
{
    char buf[48];
    std::snprintf(buf, sizeof(buf), "%.9g", static_cast<double>(v));
    return std::string(buf);
}

std::string double_str(double v)
{
    char buf[48];
    std::snprintf(buf, sizeof(buf), "%.17g", v);
    return std::string(buf);
}

// Scope guard for the per-test PASS/FAIL line.
class TestScope {
public:
    TestScope(const char *id, const std::string &title) : id_(id)
    {
        g_case_failures = 0;
        std::printf("  %-7s %s\n", id, title.c_str());
    }
    ~TestScope()
    {
        std::printf("      %s\n", g_case_failures == 0 ? "PASS" : "-> FAILED");
    }

private:
    const char *id_;
};

// --- §16.0 hand datasets -----------------------------------------------------------------------

void test_hand_cases()
{
    for (const tests::HandCase &c : tests::hand_cases()) {
        TestScope scope("H-case", c.id + (c.note.empty() ? "" : "  (" + c.note + ")"));

        // Validation first, because two of the cases exist to exercise VAL-003 and VAL-008.
        const Validation params = validate_parameters(c.N, c.D, c.K, c.max_iter, c.tol);
        check(params.ok, c.id + ": parameters accepted");
        double m_range = 0.0;
        const Validation data = validate_data(c.X, c.N, c.D, c.C0, c.K, &m_range);
        check(data.ok, c.id + ": data accepted");
        check(params.has_warning("W-KGTN") == c.expect_warning_kgtn,
              c.id + ": W-KGTN warning " + (c.expect_warning_kgtn ? "expected" : "not expected"));
        check(data.has_warning("W-DUPC") == c.expect_warning_dupc,
              c.id + ": W-DUPC warning " + (c.expect_warning_dupc ? "expected" : "not expected"));

        const Result r = cpu::run(c.X.data(), c.N, c.D, c.C0.data(), c.K, c.max_iter, c.tol,
                                  m_range, false);

        check_eq_i64(r.iterations, c.expected_iterations, c.id + ": iteration count");
        check(r.status == c.expected_status,
              c.id + ": status: got " + status_name(r.status) + ", want " +
                  status_name(c.expected_status));
        check(r.centroid_consistent == c.expected_centroid_consistent,
              c.id + ": P2 flag: got " + (r.centroid_consistent ? "true" : "false") + ", want " +
                  (c.expected_centroid_consistent ? "true" : "false"));
        check(r.assignment_consistent, c.id + ": P1 flag true by construction");

        for (int64_t i = 0; i < c.N; ++i) {
            check_eq_i64(r.assignments[static_cast<size_t>(i)], c.expected_A[static_cast<size_t>(i)],
                         c.id + ": A[" + std::to_string(i) + "]");
        }
        for (int64_t k = 0; k < c.K; ++k) {
            check_eq_i64(r.counts[static_cast<size_t>(k)], c.expected_n[static_cast<size_t>(k)],
                         c.id + ": n[" + std::to_string(k) + "]");
        }
        const double s_max = s_max_from_range(m_range);
        for (int64_t e = 0; e < c.K * c.D; ++e) {
            const float got = r.centroids[static_cast<size_t>(e)];
            const float want = c.expected_C[static_cast<size_t>(e)];
            const bool ok = c.centroids_exact ? (got == want) : centroid_agrees(got, want, s_max);
            check(ok, c.id + ": C[" + std::to_string(e) + "]: got " + float_str(got) + ", want " +
                          float_str(want) + (c.centroids_exact ? " (exact)" : " (NUM-006)"));
        }

        // Empty clusters must retain their initial centroid bit-exactly (DEC-003). This holds for
        // the H cases because no empty cluster is ever re-populated in them.
        for (int64_t k = 0; k < c.K; ++k) {
            if (c.expected_n[static_cast<size_t>(k)] != 0) {
                continue;
            }
            for (int64_t j = 0; j < c.D; ++j) {
                const size_t e = static_cast<size_t>(k * c.D + j);
                check(r.centroids[e] == c.C0[e],
                      c.id + ": empty cluster " + std::to_string(k) + " retained C0[" +
                          std::to_string(j) + "] bit-exactly");
            }
        }
    }
}

// --- TEST-U08: movement and the inclusive tolerance boundary -----------------------------------

void test_u08_movement()
{
    {
        TestScope scope("U08", "movement: exact delta, empty cluster excluded");
        // Cluster 0 moves (0,0) -> (3,4), a 3-4-5 triangle, so delta is exactly 5 in binary64.
        // Cluster 1 does not move. Cluster 2 is empty: its delta is 0 by construction and it is
        // excluded from the maximum, so a 0/0 division would show up here as NaN or 0.
        const std::vector<float> c_cur = {0.0f, 0.0f, 1.0f, 1.0f, 5.0f, 5.0f};
        const std::vector<float> c_next = {3.0f, 4.0f, 1.0f, 1.0f, 5.0f, 5.0f};
        const std::vector<uint32_t> n = {2, 1, 0};
        const double delta = cpu::movement(c_cur.data(), c_next.data(), n.data(), 3, 2);
        check(delta == 5.0, "delta: got " + double_str(delta) + ", want exactly 5");

        // With every cluster empty the maximum is over an empty set, which §2.6 cannot reach
        // because N >= 1, but the function must still return 0 rather than a NaN.
        const std::vector<uint32_t> none = {0, 0, 0};
        const double delta_none = cpu::movement(c_cur.data(), c_next.data(), none.data(), 3, 2);
        check(delta_none == 0.0, "delta with no non-empty cluster is 0");
    }

    {
        TestScope scope("U08", "inclusive comparison at delta == tol (DEC-002, ENV-10)");
        const std::vector<float> X = tests::h2_points();
        const std::vector<float> C0 = tests::h2_initial_centroids();
        double m_range = 0.0;
        const Validation v = validate_data(X, 4, 1, C0, 2, &m_range);
        check(v.ok, "H2 data accepted");

        // The first update of H2 moves a centroid by exactly 2. With tol = 2 the inclusive
        // comparison stops there; one ulp below 2 it must not.
        const Result at_boundary =
            cpu::run(X.data(), 4, 1, C0.data(), 2, 10, 2.0, m_range, false);
        check(at_boundary.status == Status::CONVERGED_MOVEMENT,
              std::string("tol == delta: status ") + status_name(at_boundary.status));
        check_eq_i64(at_boundary.iterations, 1, "tol == delta: iterations");

        const double just_below = std::nextafter(2.0, 0.0);
        const Result below =
            cpu::run(X.data(), 4, 1, C0.data(), 2, 10, just_below, m_range, false);
        check(below.status == Status::CONVERGED_MOVEMENT,
              std::string("tol one ulp below delta: status ") + status_name(below.status));
        check_eq_i64(below.iterations, 2, "tol one ulp below delta: iterations (not 1)");
        check(below.last_delta == 1.5, "tol one ulp below delta: last delta is the second one");
    }

    {
        TestScope scope("U08", "tol = 0 means bit-identical centroids");
        // H1 converges by assignment stability at t = 1 even though its delta (0.5) exceeds 0,
        // which is what DEC-002's "assignment stability takes precedence" means in practice.
        const std::vector<float> X = {0.0f, 1.0f, 9.0f, 10.0f};
        const std::vector<float> C0 = {0.0f, 10.0f};
        const Result r = cpu::run(X.data(), 4, 1, C0.data(), 2, 10, 0.0, 10.0, false);
        check(r.status == Status::CONVERGED_ASSIGNMENTS,
              std::string("status ") + status_name(r.status));
        check(r.last_delta == 0.5, "last delta: got " + double_str(r.last_delta) + ", want 0.5");
    }

    {
        TestScope scope("U08", "fixed-iteration mode disables both criteria (FR-020)");
        const std::vector<float> X = tests::h2_points();
        const std::vector<float> C0 = tests::h2_initial_centroids();
        const Result r = cpu::run(X.data(), 4, 1, C0.data(), 2, 7, 1e9, 10.0, true);
        check_eq_i64(r.iterations, 7, "exactly max_iter updates run");
        check(r.status == Status::ITERATION_LIMIT, std::string("status ") + status_name(r.status));
    }
}

// --- TEST-U09: empty clusters ------------------------------------------------------------------

void test_u09_empty_clusters()
{
    {
        TestScope scope("U09", "empty cluster at the update step retains its centroid");
        // The mid-iteration case of EDGE-14 at step level: cluster 1 held points in an earlier
        // accumulation and holds none now. update() must copy its centroid through untouched and
        // never divide, and movement() must report only cluster 0's displacement.
        const std::vector<float> c_cur = {1.0f, 7.25f};
        const std::vector<double> S = {11.0, 0.0};
        const std::vector<uint32_t> n = {4, 0};
        std::vector<float> c_next(2, -1.0f);
        cpu::update(S.data(), n.data(), 2, 1, c_cur.data(), c_next.data());

        check(c_next[0] == 2.75f, "non-empty cluster: 11/4 = 2.75 exactly, got " +
                                      float_str(c_next[0]));
        check(c_next[1] == 7.25f, "empty cluster retained bit-exactly, got " +
                                      float_str(c_next[1]));
        check(std::isfinite(c_next[1]), "empty cluster is not NaN or Inf (no 0/0 division)");

        const double delta = cpu::movement(c_cur.data(), c_next.data(), n.data(), 2, 1);
        check(delta == 1.75, "delta is cluster 0's 1.75 only, got " + double_str(delta));
    }

    {
        TestScope scope("U09", "single-point cluster is bit-exact (EDGE-15)");
        // Division by 1 is exact in binary64 and the quotient is representable, so the centroid
        // must equal the point bit for bit, not merely within NUM-006.
        const std::vector<float> X = {0.1f, 1234.5678f, -7.25f};
        const std::vector<float> C0 = {0.1f, 1234.5678f, -7.25f};
        const std::vector<int32_t> A = {0, 1, 2};
        std::vector<double> S(3, 0.0);
        std::vector<uint32_t> n(3, 0);
        cpu::accumulate(X.data(), 3, 1, A.data(), 3, S.data(), n.data());
        std::vector<float> C_next(3, 0.0f);
        cpu::update(S.data(), n.data(), 3, 1, C0.data(), C_next.data());
        for (int k = 0; k < 3; ++k) {
            check(C_next[static_cast<size_t>(k)] == X[static_cast<size_t>(k)],
                  "cluster " + std::to_string(k) + " equals its only point bit-exactly");
            check_eq_i64(n[static_cast<size_t>(k)], 1, "count of cluster " + std::to_string(k));
        }
    }

    {
        TestScope scope("U09", "K > N leaves at least K - N clusters empty (EDGE-04)");
        const std::vector<float> X = {4.0f};
        const std::vector<float> C0 = {0.0f, 1.0f, 2.0f};
        const Result r = cpu::run(X.data(), 1, 1, C0.data(), 3, 10, 0.0, 4.0, false);
        int empty = 0;
        for (uint32_t count : r.counts) {
            if (count == 0) {
                ++empty;
            }
        }
        check(empty >= 2, "at least K - N = 2 clusters empty, got " + std::to_string(empty));
        check(r.centroids[0] == 0.0f && r.centroids[1] == 1.0f,
              "both empty clusters retained their initial centroids");
    }
}

// --- TEST-U11: size and overflow arithmetic ----------------------------------------------------

void test_u11_size_calculator()
{
    {
        TestScope scope("U11", "MEM-002 byte counts");
        // N=1000, D=32, K=64, Approach A (G = 0), 1024 movement blocks:
        //   8ND = 256000, 4N = 4000, 16KD = 32768, 4K = 256, scalars = 1024*8 + 32 = 8224
        const MemoryModel m = device_memory_bytes(1000, 32, 64, 0, 1024);
        check(!m.overflow, "no overflow for a modest configuration");
        check_eq_i64(static_cast<int64_t>(m.x_col), 128000, "x_col bytes (4ND)");
        check_eq_i64(static_cast<int64_t>(m.x_row), 128000, "x_row bytes (4ND)");
        check_eq_i64(static_cast<int64_t>(m.centroids), 16384, "centroid bytes (8KD)");
        check_eq_i64(static_cast<int64_t>(m.assignments), 4000, "assignment bytes (4N)");
        check_eq_i64(static_cast<int64_t>(m.sums), 16384, "sum bytes (8KD)");
        check_eq_i64(static_cast<int64_t>(m.counts), 256, "count bytes (4K)");
        check_eq_i64(static_cast<int64_t>(m.partials), 0, "no P buffer for Approach A");
        check_eq_i64(static_cast<int64_t>(m.partial_counts), 0, "no Q buffer for Approach A");
        check_eq_i64(static_cast<int64_t>(m.total), 128000 + 128000 + 16384 + 4000 + 16384 + 256 +
                                                        8224,
                     "total bytes");
    }

    {
        TestScope scope("U11", "Approach B adds 8GKD + 4GK");
        const MemoryModel a = device_memory_bytes(1000, 32, 64, 0, 1024);
        const MemoryModel b = device_memory_bytes(1000, 32, 64, 7, 1024);
        check_eq_i64(static_cast<int64_t>(b.partials), 8 * 7 * 64 * 32, "P buffer bytes (8GKD)");
        check_eq_i64(static_cast<int64_t>(b.partial_counts), 4 * 7 * 64, "Q buffer bytes (4GK)");
        check_eq_i64(static_cast<int64_t>(b.total - a.total), 8 * 7 * 64 * 32 + 4 * 7 * 64,
                     "difference is exactly the P and Q buffers");
    }

    {
        TestScope scope("U11", "overflow is detected, not wrapped (CUDA-006, EDGE-30)");
        uint64_t out = 0;
        check(checked_mul(UINT64_MAX / 2, 2, &out), "a product that just fits is accepted");
        check(!checked_mul(UINT64_MAX / 2 + 1, 2, &out), "a product that overflows is rejected");
        check(!checked_add(UINT64_MAX, 1, &out), "a sum that overflows is rejected");
        check(checked_mul(0, UINT64_MAX, &out) && out == 0, "zero times anything is zero");

        const int64_t huge = 2147483647;  // 2^31 - 1, the VAL-002 maximum
        const MemoryModel m = device_memory_bytes(huge, huge, huge, huge, 1024);
        check(m.overflow, "N = D = K = G = 2^31-1 overflows the capacity model");
        check_eq_i64(static_cast<int64_t>(m.total), 0, "an overflowing model reports no total");

        const Validation v = validate_capacity(huge, huge, huge, huge, 16ull * 1024 * 1024 * 1024);
        check(!v.ok && v.exit_code == kExitInvalidInput,
              "VAL-009 rejects the overflowing configuration with exit 2");
    }

    {
        TestScope scope("U11", "VAL-009 compares against 95 % of free memory");
        // 15 GB free, as on the recorded Tesla T4 (§1.6.2).
        const uint64_t free_bytes = 15637086208ull;
        const Validation ok = validate_capacity(1000000, 32, 64, 0, free_bytes);
        check(ok.ok, "the PERF-003 base point fits");

        const MemoryModel m = device_memory_bytes(1000000, 32, 64, 0);
        const Validation tight = validate_capacity(1000000, 32, 64, 0, m.total);
        check(!tight.ok && tight.exit_code == kExitUnsupportedDevice,
              "exactly the required bytes free is rejected by the 5 % margin with exit 3");
    }

    {
        TestScope scope("U11", "VAL-001..VAL-010 reject what they must");
        check(!validate_parameters(0, 1, 1, 10, 0.0).ok, "N = 0 rejected (VAL-001, EDGE-01)");
        check(!validate_parameters(1, 0, 1, 10, 0.0).ok, "D = 0 rejected (VAL-001, EDGE-02)");
        check(!validate_parameters(1, 1, 0, 10, 0.0).ok, "K = 0 rejected (VAL-001, EDGE-03)");
        check(validate_parameters(1, 1, 1, 0, 0.0).ok, "max_iter = 0 accepted (EDGE-22)");
        check(!validate_parameters(1, 1, 1, -1, 0.0).ok, "negative max_iter rejected (VAL-004)");
        check(!validate_parameters(1, 1, 1, 10, -1e-30).ok, "negative tol rejected (VAL-005)");
        check(!validate_parameters(1, 1, 1, 10, std::nan("")).ok, "NaN tol rejected (VAL-005)");
        check(!validate_parameters(1, 1, 1, 10, HUGE_VAL).ok, "infinite tol rejected (VAL-005)");
        check(validate_parameters(2, 1, 5, 10, 0.0).has_warning("W-KGTN"),
              "K > N warns W-KGTN and continues (VAL-003)");
        check(validate_parameters(1, 1, 1, 2147483648ll, 0.0).ok == false,
              "max_iter above 2^31-1 rejected (VAL-004)");
        check(validate_parameters(2147483648ll, 1, 1, 10, 0.0).ok == false,
              "N above 2^31-1 rejected (VAL-002)");

        // VAL-006 and VAL-007 on the data.
        double m_range = 0.0;
        const std::vector<float> good = {1.0f, 2.0f};
        const std::vector<float> c_one = {1.0f};
        const std::vector<float> with_nan = {1.0f, std::nanf("")};
        const std::vector<float> with_inf = {1.0f, HUGE_VALF};
        check(validate_data(good, 2, 1, c_one, 1, &m_range).ok, "finite data accepted");
        check(m_range == 2.0, "M is the largest magnitude, got " + double_str(m_range));
        const Validation nan_v = validate_data(with_nan, 2, 1, c_one, 1, &m_range);
        check(!nan_v.ok && nan_v.rule == "VAL-006", "NaN in data rejected (EDGE-20)");
        const Validation inf_v = validate_data(with_inf, 2, 1, c_one, 1, &m_range);
        check(!inf_v.ok && inf_v.rule == "VAL-006", "Inf in data rejected (EDGE-20)");

        // VAL-007: 4 D M^2 <= 2^126. With D = 1 the largest admissible M is 2^62, and a value an
        // octave above it must be refused.
        const float at_bound = static_cast<float>(std::ldexp(1.0, 62));
        const std::vector<float> big = {at_bound};
        check(validate_data(big, 1, 1, big, 1, &m_range).ok, "M = 2^62 with D = 1 is accepted");
        const float over = static_cast<float>(std::ldexp(1.0, 63));
        const std::vector<float> too_big = {over};
        const Validation range_v = validate_data(too_big, 1, 1, too_big, 1, &m_range);
        check(!range_v.ok && range_v.rule == "VAL-007", "M = 2^63 with D = 1 is rejected");

        // VAL-008 duplicate centroids warn but do not fail.
        const std::vector<float> dup = {5.0f, 5.0f};
        const Validation dup_v = validate_data(good, 2, 1, dup, 2, &m_range);
        check(dup_v.ok && dup_v.has_warning("W-DUPC"), "duplicate centroids warn W-DUPC");

        // VAL-010 option ranges.
        check(validate_options(256, 0, 8, 32, 1024, 49152).ok, "the defaults are accepted");
        check(!validate_options(200, 0, 8, 32, 1024, 49152).ok, "block size 200 is not a multiple "
                                                                "of 32");
        check(!validate_options(2048, 0, 8, 32, 1024, 49152).ok, "block size above 1024 rejected");
        check(!validate_options(256, -1, 8, 32, 1024, 49152).ok, "negative G rejected");
        check(!validate_options(256, 0, 8, 48, 1024, 49152).ok, "tile_d not a multiple of 32");
        check(!validate_options(256, 0, 1024, 32, 1024, 49152).ok,
              "tiles larger than the shared-memory limit rejected (EDGE-34)");
    }
}

// --- TEST-U12: file I/O ------------------------------------------------------------------------

void test_u12_file_io()
{
    TestScope scope("U12", "BLD-004 .kmd/.kma round trip and rejection of damaged files");

    check(host_is_little_endian(), "host is little-endian, as BLD-004 requires");

    const std::string path = "test_u12.kmd";
    const std::vector<float> values = {0.0f, 1.5f, -2.25f, 1e30f, 10.0f, 0.1f};
    std::string err;
    check(write_kmd(path, 3, 2, values.data(), &err) == kExitOk, "write a 3x2 .kmd: " + err);

    Matrix m;
    check(read_kmd(path, &m, &err) == kExitOk, "read it back: " + err);
    check(m.rows == 3 && m.cols == 2, "dimensions survive the round trip");
    check(m.data == values, "every value survives the round trip bit-exactly");

    // The payload of this file contains byte 0x0A, which Windows text mode would translate. A
    // failure here means a stream was opened without std::ios::binary.
    const std::vector<float> newline_byte = {1.4012985e-45f * 10.0f};  // 0x0000000A as a float
    const float raw_newline = []() {
        uint32_t bits = 0x0000000Au;
        float f = 0.0f;
        std::memcpy(&f, &bits, sizeof(f));
        return f;
    }();
    const std::vector<float> with_lf = {raw_newline, raw_newline, 1.0f};
    check(write_kmd("test_u12_lf.kmd", 3, 1, with_lf.data(), &err) == kExitOk,
          "write a payload containing 0x0A bytes");
    Matrix lf;
    check(read_kmd("test_u12_lf.kmd", &lf, &err) == kExitOk, "read it back: " + err);
    check(lf.data == with_lf, "0x0A payload bytes are not translated (binary mode)");
    (void)newline_byte;

    // FR-002: the declared shape must match the header.
    check(read_kmd_expect(path, 3, 2, &m, &err) == kExitOk, "matching declared shape accepted");
    const int mismatch = read_kmd_expect(path, 4, 2, &m, &err);
    check(mismatch == kExitInvalidInput, "declared shape mismatch is exit 2, got " +
                                             std::to_string(mismatch));

    // Truncation and trailing bytes are both caught by the exact-size rule.
    {
        std::vector<unsigned char> bytes;
        std::FILE *f = std::fopen(path.c_str(), "rb");
        check(f != nullptr, "reopen the file to copy its bytes");
        if (f != nullptr) {
            unsigned char buf[512];
            const size_t got = std::fread(buf, 1, sizeof(buf), f);
            bytes.assign(buf, buf + got);
            std::fclose(f);
        }

        std::FILE *t = std::fopen("test_u12_trunc.kmd", "wb");
        if (t != nullptr && bytes.size() > 4) {
            std::fwrite(bytes.data(), 1, bytes.size() - 4, t);
            std::fclose(t);
        }
        const int code = read_kmd("test_u12_trunc.kmd", &m, &err);
        check(code == kExitIoError, "truncated file is exit 5, got " + std::to_string(code));

        std::FILE *x = std::fopen("test_u12_extra.kmd", "wb");
        if (x != nullptr) {
            std::fwrite(bytes.data(), 1, bytes.size(), x);
            const unsigned char junk[4] = {1, 2, 3, 4};
            std::fwrite(junk, 1, sizeof(junk), x);
            std::fclose(x);
        }
        const int extra = read_kmd("test_u12_extra.kmd", &m, &err);
        check(extra == kExitIoError, "trailing bytes give exit 5, got " + std::to_string(extra));

        std::FILE *b = std::fopen("test_u12_magic.kmd", "wb");
        if (b != nullptr && bytes.size() > 4) {
            std::fwrite("XXXX", 1, 4, b);
            std::fwrite(bytes.data() + 4, 1, bytes.size() - 4, b);
            std::fclose(b);
        }
        const int magic = read_kmd("test_u12_magic.kmd", &m, &err);
        check(magic == kExitIoError, "bad magic gives exit 5, got " + std::to_string(magic));
    }

    const int missing = read_kmd("test_u12_does_not_exist.kmd", &m, &err);
    check(missing == kExitIoError, "missing file is exit 5, got " + std::to_string(missing));

    // .kma assignments.
    const std::vector<int32_t> assignments = {0, 1, 2, 1, 0, 2147483647};
    check(write_kma("test_u12.kma", assignments, &err) == kExitOk, "write .kma: " + err);
    std::vector<int32_t> read_back;
    check(read_kma("test_u12.kma", &read_back, &err) == kExitOk, "read .kma: " + err);
    check(read_back == assignments, ".kma round trip is bit-exact");

    for (const char *p : {"test_u12.kmd", "test_u12_lf.kmd", "test_u12_trunc.kmd",
                          "test_u12_extra.kmd", "test_u12_magic.kmd", "test_u12.kma"}) {
        std::remove(p);
    }
}

// --- TEST-U13: generator determinism -----------------------------------------------------------

void test_u13_generator()
{
    {
        TestScope scope("U13", "identical parameters reproduce identical datasets (FR-003)");
        GenParams p;
        p.N = 500;
        p.D = 4;
        p.K = 5;
        p.sep = 10.0;
        p.spread = 1.0;
        p.seed = 20261017;

        const Dataset a = generate(p);
        const Dataset b = generate(p);
        check(a.X == b.X, "the data matrix is bit-identical across runs");
        check(a.C0 == b.C0, "the initial centroids are bit-identical across runs");
        check(a.dataset_id == b.dataset_id, "the dataset ID is stable: " + a.dataset_id);
        check_eq_i64(static_cast<int64_t>(a.X.size()), p.N * p.D, "data size is N*D");
        check_eq_i64(static_cast<int64_t>(a.C0.size()), p.K * p.D, "centroid size is K*D");

        // A different seed must change the data; otherwise the seed is being ignored.
        GenParams q = p;
        q.seed = p.seed + 1;
        const Dataset c = generate(q);
        check(c.X != a.X, "a different seed changes the data");
        check(c.dataset_id != a.dataset_id, "a different seed changes the dataset ID");

        // The written files must be byte-identical too, which is what TEST-U13 compares.
        std::string err;
        check(write_kmd("test_u13_a.kmd", static_cast<uint64_t>(p.N),
                        static_cast<uint64_t>(p.D), a.X.data(), &err) == kExitOk, "write A");
        check(write_kmd("test_u13_b.kmd", static_cast<uint64_t>(p.N),
                        static_cast<uint64_t>(p.D), b.X.data(), &err) == kExitOk, "write B");
        Matrix ma, mb;
        check(read_kmd("test_u13_a.kmd", &ma, &err) == kExitOk, "read A");
        check(read_kmd("test_u13_b.kmd", &mb, &err) == kExitOk, "read B");
        check(ma.data == mb.data, "the two files hold identical bytes");
        std::remove("test_u13_a.kmd");
        std::remove("test_u13_b.kmd");

        // Every generated value must pass the VAL rules the loader will apply.
        double m_range = 0.0;
        const Validation v = validate_data(a.X, p.N, p.D, a.C0, p.K, &m_range);
        check(v.ok, "generated data passes VAL-006 and VAL-007");
        check(a.min_centre_distance > 0.0, "a positive minimum centre distance is measured");
    }

    {
        TestScope scope("U13", "well-separated blobs are separated by at least 10 sigma (§10.3)");
        GenParams p;
        p.N = 1000;
        p.D = 8;
        p.K = 4;
        p.sep = 10.0;
        p.spread = 1.0;
        p.seed = 7;
        const Dataset d = generate(p);
        const double sigma = d.min_centre_distance / p.spread;
        check(sigma >= 10.0, "minimum centre separation is " + double_str(sigma) +
                                 " sigma, want >= 10 for the well-separated datasets");
    }

    {
        TestScope scope("U13", "the uniform draw stays inside (0, 1)");
        // log(u1) in Box-Muller must never see 0, and a draw of exactly 1 would bias the angle.
        Rng rng(12345);
        double lo = 1.0;
        double hi = 0.0;
        for (int i = 0; i < 200000; ++i) {
            const double u = rng.uniform_open();
            if (u < lo) {
                lo = u;
            }
            if (u > hi) {
                hi = u;
            }
        }
        check(lo > 0.0, "smallest draw is above 0: " + double_str(lo));
        check(hi < 1.0, "largest draw is below 1: " + double_str(hi));
    }

    {
        TestScope scope("U13", "the normal draw has the expected first two moments");
        // A sanity check on Box-Muller, not a statistical test: with 400k samples the sample mean
        // is within about 0.005 of 0 and the variance within about 0.01 of 1.
        Rng rng(999);
        const int count = 400000;
        double sum = 0.0;
        double sum_sq = 0.0;
        for (int i = 0; i < count; ++i) {
            const double z = rng.normal();
            sum += z;
            sum_sq += z * z;
        }
        const double mean = sum / count;
        const double variance = sum_sq / count - mean * mean;
        check(std::fabs(mean) < 0.02, "sample mean " + double_str(mean) + " is near 0");
        check(std::fabs(variance - 1.0) < 0.02,
              "sample variance " + double_str(variance) + " is near 1");
    }
}

// --- §16.3: the two distance paths agree except on near-ties -----------------------------------

void test_distance_paths_agree()
{
    TestScope scope("U01-pre", "FP32 and FP64 assignment agree except on near-ties (NUM-005)");

    GenParams p;
    p.N = 2000;
    p.D = 6;
    p.K = 8;
    p.sep = 3.0;   // deliberately tight, so near-ties actually occur
    p.spread = 1.0;
    p.seed = 424242;
    const Dataset d = generate(p);

    int64_t mismatches = 0;
    int64_t near_ties = 0;
    int64_t hard_mismatches = 0;
    double worst_margin = 0.0;

    for (int64_t i = 0; i < p.N; ++i) {
        const float *x = d.X.data() + i * p.D;
        const int32_t k32 = cpu::nearest_centroid(x, p.D, d.C0.data(), p.K);
        const cpu::TieInfo info = cpu::tie_info(x, p.D, d.C0.data(), p.K);
        if (info.near_tie) {
            ++near_ties;
        }
        if (k32 != info.k1) {
            ++mismatches;
            if (!info.near_tie) {
                ++hard_mismatches;
                if (info.margin > worst_margin) {
                    worst_margin = info.margin;
                }
            }
        }
    }

    std::printf("      %lld points, %lld near-ties, %lld FP32/FP64 mismatches, "
                "%lld of them outside the near-tie band\n",
                static_cast<long long>(p.N), static_cast<long long>(near_ties),
                static_cast<long long>(mismatches), static_cast<long long>(hard_mismatches));
    check(hard_mismatches == 0,
          "no mismatch outside the near-tie band; worst offending margin " +
              double_str(worst_margin) + " against a threshold of " +
              double_str(near_tie_threshold(p.D)));

    // Random blobs rarely land on a decision boundary, so the band above is usually empty and the
    // check passes without exercising NUM-005 at all. These constructed points sit exactly on, or
    // one ulp off, the perpendicular bisector of two centroids, which is where the rule bites
    // (EDGE-13, EDGE-19; TEST-U02 repeats them on the device).
    const std::vector<float> C = {0.0f, 0.0f, 2.0f, 0.0f};  // two centroids 2 apart in 2-D
    const std::vector<float> on_bisector = {1.0f, 0.0f, 1.0f, 5.0f, 1.0f, -3.25f};
    for (int64_t i = 0; i < 3; ++i) {
        const float *x = on_bisector.data() + i * 2;
        const int32_t k = cpu::nearest_centroid(x, 2, C.data(), 2);
        const cpu::TieInfo info = cpu::tie_info(x, 2, C.data(), 2);
        check_eq_i64(k, 0, "point on the bisector takes the lowest index (DEC-005)");
        check(info.margin == 0.0, "its binary64 margin is exactly 0");
        check(info.near_tie, "it is classified as a near-tie");
    }

    // Just off the bisector the nearer centroid must win outright, and the margin must be small
    // enough to still count as a near-tie, which is what makes a device mismatch here reportable
    // rather than a failure.
    const float nudge = std::nextafter(1.0f, 2.0f);
    const std::vector<float> off_bisector = {nudge, 0.0f};
    const cpu::TieInfo off = cpu::tie_info(off_bisector.data(), 2, C.data(), 2);
    check_eq_i64(cpu::nearest_centroid(off_bisector.data(), 2, C.data(), 2), 1,
                 "one ulp past the bisector the second centroid wins");
    check(off.near_tie, "a one-ulp margin is inside the near-tie band: margin " +
                            double_str(off.margin) + " <= " + double_str(near_tie_threshold(2)));

    // A point far from the boundary must not be excused as a near-tie.
    const std::vector<float> clear = {0.25f, 0.0f};
    const cpu::TieInfo far = cpu::tie_info(clear.data(), 2, C.data(), 2);
    check(!far.near_tie, "a clearly nearer centroid is not a near-tie: margin " +
                             double_str(far.margin));
    check_eq_i64(far.k1, 0, "and it is the lower-index one here");

    // Duplicate centroids make every point an exact tie (EDGE-10, EDGE-12).
    const std::vector<float> dup = {3.0f, 3.0f, 3.0f, 3.0f};
    const std::vector<float> any_point = {-7.5f, 2.0f};
    check_eq_i64(cpu::nearest_centroid(any_point.data(), 2, dup.data(), 2), 0,
                 "with duplicate centroids the lower index always wins");
}

// --- end-to-end sanity on generated data -------------------------------------------------------

void test_end_to_end_blobs()
{
    TestScope scope("I11-pre", "well-separated blobs recover their structure on the CPU");

    GenParams p;
    p.N = 600;
    p.D = 3;
    p.K = 4;
    p.sep = 20.0;
    p.spread = 1.0;
    p.seed = 31337;
    const Dataset d = generate(p);

    double m_range = 0.0;
    const Validation v = validate_data(d.X, p.N, p.D, d.C0, p.K, &m_range);
    check(v.ok, "generated data is valid");

    const Result r = cpu::run(d.X.data(), p.N, p.D, d.C0.data(), p.K, 100, 1e-6, m_range, false);
    check(is_converged(r.status), std::string("the run converged: ") + status_name(r.status));
    check(r.centroid_consistent, "P2 holds after convergence by assignment stability");
    check(r.assignment_consistent, "P1 holds by construction");

    uint64_t total = 0;
    for (uint32_t count : r.counts) {
        total += count;
    }
    check_eq_i64(static_cast<int64_t>(total), p.N, "the counts sum to N");

    for (int64_t i = 0; i < p.N; ++i) {
        if (r.assignments[static_cast<size_t>(i)] < 0 ||
            r.assignments[static_cast<size_t>(i)] >= p.K) {
            check(false, "assignment " + std::to_string(i) + " is outside [0, K)");
            break;
        }
    }

    // P1 independently: every reported assignment is the nearest centroid of the reported
    // centroids (§2.8 verification mode, TEST-I07 at CPU level).
    int64_t p1_violations = 0;
    for (int64_t i = 0; i < p.N; ++i) {
        const int32_t expect =
            cpu::nearest_centroid(d.X.data() + i * p.D, p.D, r.centroids.data(), p.K);
        if (expect != r.assignments[static_cast<size_t>(i)]) {
            ++p1_violations;
        }
    }
    check_eq_i64(p1_violations, 0, "P1 re-checked against the reported centroids");
    check(r.objective > 0.0, "the objective J is positive and finite");
    check(std::isfinite(r.objective), "the objective J is finite");
}

// --- launch geometry and the choice of G (CUDA-007, RED-006, MEM-003, DEC-001) -----------------
//
// src/launch_config.h is deliberately free of CUDA constructs so that all of this arithmetic --
// which decides how much memory Approach B allocates and in what order it sums -- is testable
// without a GPU. The kernels use the identical expressions.

void test_launch_geometry()
{
    {
        TestScope scope("U11", "CUDA-007 grid sizing");
        // grid = min(ceil(work / block), 32 * SM count). With 40 SMs the cap is 1280.
        check_eq_i64(grid_size(1000000, 256, 40), 1280, "large work is capped at 32 * SMs");
        check_eq_i64(grid_size(1000, 256, 40), 4, "small work uses ceil(work / block)");
        check_eq_i64(grid_size(256, 256, 40), 1, "exactly one block of work");
        check_eq_i64(grid_size(257, 256, 40), 2, "one element past a block rounds up");
        check_eq_i64(grid_size(1, 256, 40), 1, "a single element still launches one block");
        check_eq_i64(grid_size(0, 256, 40), 1,
                     "zero work launches one block, which the bounds checks make a no-op");

        // The movement grid is capped at 1024 so max_reduce fits in one block and the cap matches
        // the movement_blocks default of device_memory_bytes.
        check_eq_i64(movement_grid_size(100000000, 256, 40), kMaxMovementBlocks,
                     "the movement grid is capped at 1024 blocks");
        check_eq_i64(movement_grid_size(512, 256, 40), 2, "small K is not affected by the cap");
    }

    {
        TestScope scope("U11", "MEM-004 / SMEM-003 shared-memory footprints");
        // Against the 48 KiB per-block default of ENV-01.
        const int64_t limit = 49152;
        check_eq_i64(stage2_shared_bytes(), 8 * 8 * 32 + 4 * 8, "reduce_stage2 needs 2080 B");
        check(stage2_shared_bytes() <= limit, "reduce_stage2 fits the 48 KiB limit");
        check_eq_i64(assign_smem_bytes(kAssignTileK, kAssignTileD), 1024,
                     "assign_smem needs 1 KiB at T_K = 8, T_D = 32");
        check(assign_smem_bytes(16, 32) <= limit, "the largest tile sweep configuration fits");
        check_eq_i64(block_reduction_bytes(1024), 8192,
                     "a 1024-thread block reduction needs 8 KiB");
        check(block_reduction_bytes(1024) <= limit, "the largest block reduction fits");
    }

    {
        TestScope scope("U11", "Stage 2 tile geometry (RED-003)");
        check_eq_i64(stage2_dim_tiles(1), 1, "D = 1 is one partial tile");
        check_eq_i64(stage2_dim_tiles(32), 1, "D = 32 is exactly one tile");
        check_eq_i64(stage2_dim_tiles(33), 2, "D = 33 needs a second, partial tile");
        check_eq_i64(stage2_dim_tiles(65), 3, "D = 65 needs three tiles");
        check_eq_i64(stage2_tiles(1024, 65), 1024 * 3, "K * T tiles in total");
    }

    {
        TestScope scope("U11", "RED-006 choice of G");
        // 40 SMs, so T_target = 81920. With D = 32 the target is ceil(81920/32) = 2560.
        const uint64_t plenty = 16ull * 1024 * 1024 * 1024;
        const ChunkSelection big =
            choose_reduce_chunks(1000000, 32, 64, 40, plenty, 1024 * 1024, 0, 0);
        check(big.ok, "a modest configuration selects a G");
        check_eq_i64(big.g_target, 2560, "G_target = ceil(T_target / D)");
        check_eq_i64(big.G, 2560, "memory is ample, so the target wins");
        check_eq_i64(static_cast<int64_t>(big.bytes_per_chunk), 8 * 64 * 32 + 4 * 64,
                     "8KD + 4K bytes per chunk");
        check_eq_i64(static_cast<int64_t>(big.required_bytes),
                     2560 * (8 * 64 * 32 + 4 * 64), "the reported byte count is 8GKD + 4GK");

        // G <= N always, however large the target.
        const ChunkSelection small_n =
            choose_reduce_chunks(100, 32, 64, 40, plenty, 1024 * 1024, 0, 0);
        check_eq_i64(small_n.G, 100, "G is clamped to N");

        // A tight budget makes G_mem the binding constraint.
        const uint64_t per_chunk = 8 * 64 * 32 + 4 * 64;
        const ChunkSelection tight =
            choose_reduce_chunks(1000000, 32, 64, 40, 4 * 100 * per_chunk, 0, 0, 0);
        check(tight.ok, "a tight budget still selects a G");
        check_eq_i64(tight.G, 100, "G_mem binds: floor(B_red / (8KD + 4K)) with B_red = free/4");
        check_eq_i64(tight.g_memory, 100, "G_mem is reported");

        // MEM-003: an explicit user cap overrides the quarter-of-free-memory rule.
        const ChunkSelection capped =
            choose_reduce_chunks(1000000, 32, 64, 40, plenty, 0, 10 * per_chunk, 0);
        check_eq_i64(capped.G, 10, "the user cap on B_red binds");

        // RED-007: no usable G at all fails, naming the bytes the run needs.
        const ChunkSelection impossible =
            choose_reduce_chunks(1000000, 32, 64, 40, per_chunk, 0, 0, 0);
        check(!impossible.ok, "a budget below one chunk is rejected");
        check(impossible.G == 0, "no G is reported when the selection fails");
        check(impossible.required_bytes == per_chunk,
              "the message reports the cost of a single chunk");
        check(!impossible.message.empty(), "the failure carries a RED-007 explanation");

        // The --reduce-chunks override is still bounded by N and by the budget.
        const ChunkSelection override_ok =
            choose_reduce_chunks(1000000, 32, 64, 40, plenty, 1024 * 1024, 0, 7);
        check(override_ok.ok && override_ok.G == 7, "an override within the limits is honoured");
        check(override_ok.user_override, "the override is recorded");
        const ChunkSelection override_too_big =
            choose_reduce_chunks(100, 32, 64, 40, plenty, 1024 * 1024, 0, 101);
        check(!override_too_big.ok, "an override above N is rejected (DEC-001 requires G <= N)");
        const ChunkSelection override_no_memory =
            choose_reduce_chunks(1000000, 32, 64, 40, 4 * 10 * per_chunk, 0, 0, 1000);
        check(!override_no_memory.ok, "an override past the budget is rejected, not silently used");

        // Small D leans on a large G for Stage 1 parallelism (RED-008).
        const ChunkSelection small_d =
            choose_reduce_chunks(10000000, 1, 4, 40, plenty, 1024 * 1024, 0, 0);
        check_eq_i64(small_d.g_target, 81920, "D = 1 gives G_target = T_target");
    }

    {
        TestScope scope("U11", "DEC-001 chunk partition covers [0, N) exactly (RED-005)");
        const int64_t sizes[][2] = {{1, 1},   {10, 1},   {10, 10}, {10, 3},  {100, 7},
                                    {97, 13}, {1000, 1}, {7, 7},   {12345, 101}};
        for (const auto &pair : sizes) {
            const int64_t N = pair[0], G = pair[1];
            const std::string tag = "N=" + std::to_string(N) + " G=" + std::to_string(G);
            check(chunk_begin(0, N, G) == 0, tag + ": the first chunk starts at 0");
            check(chunk_end(G - 1, N, G) == N, tag + ": the last chunk ends at N");
            int64_t covered = 0;
            bool contiguous = true;
            for (int64_t g = 0; g < G; ++g) {
                const int64_t begin = chunk_begin(g, N, G);
                const int64_t end = chunk_end(g, N, G);
                if (end < begin) {
                    contiguous = false;
                }
                if (g > 0 && begin != chunk_end(g - 1, N, G)) {
                    contiguous = false;  // a gap or an overlap between consecutive chunks
                }
                covered += end - begin;
            }
            check(contiguous, tag + ": consecutive chunks share endpoints");
            check_eq_i64(covered, N, tag + ": the chunks cover exactly N points");
            if (G <= N) {
                bool all_non_empty = true;
                for (int64_t g = 0; g < G; ++g) {
                    if (chunk_end(g, N, G) <= chunk_begin(g, N, G)) {
                        all_non_empty = false;
                    }
                }
                check(all_non_empty, tag + ": G <= N means every chunk is non-empty");
            }
        }

        // CUDA-006: the products stay inside int64 at the top of the valid range, where g*N is
        // close to 2^62. A 32-bit intermediate would wrap and silently mis-partition the data.
        const int64_t huge = 2147483647;  // 2^31 - 1, the VAL-002 limit
        check(chunk_end(huge - 1, huge, huge) == huge,
              "G = N at the VAL-002 limit still ends at N");
        check(chunk_begin(huge / 2, huge, 2) >= 0, "no overflow at the top of the range");
    }
}

}  // namespace

int main()
{
    std::printf("M1-M3 CPU test suite (requirements.md §16.0, §16.3, TEST-U08/U09/U11/U12/U13,\n"
                "plus the host-side launch geometry of CUDA-007, RED-006 and DEC-001)\n\n");

    test_hand_cases();
    std::printf("\n");
    test_u08_movement();
    test_u09_empty_clusters();
    test_u11_size_calculator();
    test_launch_geometry();
    test_u12_file_io();
    test_u13_generator();
    test_distance_paths_agree();
    test_end_to_end_blobs();

    std::printf("\n%d checks, %d failures\n", g_checks, g_failures);
    if (g_failures != 0) {
        std::printf("RESULT: FAILED\n");
        return 1;
    }
    std::printf("RESULT: all CPU-level tests passed\n");
    return 0;
}


### Kernel contracts, part 1 — assignment, atomic accumulation, update, convergence

One note per kernel: **thread mapping**, **memory ownership**, **synchronization points**. These are
the arguments to defend in review; the reduction kernels of Approach B have their own note below.
Common to all of them: every global index is guarded against its range (CUDA-005), out-of-range
threads skip work but still reach every barrier, all element offsets are unsigned 64-bit
(CUDA-006), and no kernel reads what another block of the same launch wrote — phase boundaries are
kernel boundaries on one stream (CUDA-004, DEC-010).

#### `assign` — §2.3, FR-004…FR-007, DEC-008

| | |
|---|---|
| Thread mapping | One thread per point `i`, 1-D grid-stride. Each thread loops over all `k` and all `j` itself. |
| Ownership | Thread `i` is the **only** reader and writer of `assignments[i]`, which is what lets FR-007 update it in place: read the old value, compute, compare, write. `x_col` and `centroids` are read-only. |
| Synchronization | None for the assignment. The change count is reduced through shared memory, so every thread of the block reaches every `__syncthreads()`, including threads whose grid-stride loop body never ran. One 64-bit `atomicAdd` per **block** into the change counter, not one per point. |
| Numerics | NUM-002: terms in binary32, accumulated in binary32 in ascending `j` from 0, never reordered. DEC-005: `k` ascends from 0 and the candidate is replaced only on a **strictly** smaller distance, so exactly equal computed distances keep the lowest index. |
| Coalescing | `x_col` is dimension-major, so for a fixed `j` consecutive threads read consecutive addresses; the centroid read is uniform across the warp and broadcasts (CUDA-009). |

The change counter is an integer, and integer atomics are exact, so it is deterministic and does
not compromise Approach B's atomic-free accumulation of centroid data.

#### `accum_atomic` — §6, ATOM-001…ATOM-007

| | |
|---|---|
| Thread mapping | One thread per element `e` of `X`, 1-D grid-stride over `N·D`, with `i = e mod N` and `j = e / N` (ATOM-002). Because `x_col` holds element `(i,j)` at `j·N + i`, that index *is* `e`, so the read is a sequential walk and coalesces (CUDA-009). |
| Ownership | `sums` and `counts` are shared by every thread; every write is an atomic read-modify-write, so no update is lost. The caller zeroes both buffers on the same stream first (ATOM-001), which is safe on reuse because the memset is stream-ordered after the previous consumer, `update`. |
| Synchronization | None inside the kernel. Ordering against `update` is the kernel boundary of ATOM-005 / DEC-010. |
| Numerics | Binary64 sums (DEC-007) in a hardware-chosen order, so `S` may differ in its last bits run to run (ATOM-007). Counts are integers and are always exact. **No bitwise reproducibility is claimed for Approach A.** |
| Contention | All threads for points of one cluster target the same `D` addresses, so contention grows as `K` falls — which is why the benchmark matrix includes small `K` (ATOM-010, PERF-003). |

#### `update` — §2.5, FR-010, FR-011, DEC-003

| | |
|---|---|
| Thread mapping | One thread per centroid element, 1-D grid-stride over `K·D`. |
| Ownership | Element `(k,j)` of `centroids_next` is written by exactly one thread; `sums`, `counts` and `centroids_cur` are read-only. Double buffering (FR-010) is what makes that true — writing into `centroids_cur` would race with the reads of the retention branch and with `movement`. |
| Synchronization | None inside the kernel; the boundary to `movement` is a kernel boundary. |
| Numerics | The division is performed in binary64 and the quotient rounded **once** to binary32. When the count is zero no division happens at all and the previous centroid is copied bit-exactly (DEC-003). Every `(k,j)` is written on every call, so `centroids_next` needs no initialization (MEM-005). |

#### `movement` and `max_reduce` — §2.6, FR-012

| | |
|---|---|
| Thread mapping | `movement`: one thread per cluster, grid-stride over `K`, each thread computing the whole squared displacement of its cluster in ascending `j`. `max_reduce`: a single block, grid-stride over the per-block maxima. |
| Ownership | `block_max[blockIdx.x]` is written by thread 0 of that block and no one else; `*delta` by thread 0 of the single `max_reduce` block. The centroid buffers, counts and `block_max` are read-only to their readers. |
| Synchronization | Both reduce through shared memory with a barrier after each tree level, and all threads reach all barriers. One block in `max_reduce`, so no grid-wide synchronization is assumed (CUDA-004). |
| Numerics | Differences are formed in binary64 from the binary32 centroids and summed in ascending `j`. Empty clusters have `δ_k = 0` by construction and are excluded from the maximum, so no `0/0` can arise. `movement` keeps `δ_k²`; the single `sqrt` is applied once in `max_reduce`. Because `sqrt` is monotone this picks the same cluster as §2.6 and rounds the same binary64 sum once, so the value is identical to taking the root per cluster and then the maximum. |

The movement grid is capped at **1024** blocks — the "unless overridden" clause of CUDA-007. That
cap lets `max_reduce` use one thread per per-block maximum in a single block, and it equals the
`movement_blocks` default of `device_memory_bytes`, so the MEM-002 capacity model and the real
allocation agree.

#### Non-power-of-two block sizes

CUDA-007 validates the block size as a multiple of 32, which need not be a power of two (96, 160
and 192 are all legal). The shared-memory reductions therefore start from the next power of two at
or above `blockDim.x` and guard the partner index, which is correct for any block size.

#### Per-iteration order (CUDA-003)

accumulation → `update` → `movement` → `max_reduce` → memset of the change counter → `assign` →
one 16-byte async copy of `Δ` and `changed` into pinned host memory → stream synchronization → host
decision (DEC-002) → centroid pointer swap. The swap is bookkeeping, so the `assign` of iteration
`t` reads the buffer `update` just wrote; the swap happens on every path, including the one that
breaks out of the loop, so the reported pair is always `(A(t), C(t))` and P1 holds by construction.

### Kernel contracts, part 2 — Approach B, the two-stage ownership reduction (DEC-001, §7)

Approach B accumulates centroid sums **without atomics** by giving every element of the
intermediate buffers exactly one writer. The buffers are `P ∈ ℝ^{G×K×D}` (binary64, element
`(g,k,j)` at `(g·K + k)·D + j`) and `Q ∈ ℕ^{G×K}` (uint32, element `(g,k)` at `g·K + k`), and the
outputs `S` (`K×D` binary64) and `n` (`K` uint32) are the same buffers the `update` kernel consumes
in Approach A (RED-001, FR-018).

#### `reduce_stage1` — RED-002

| | |
|---|---|
| Thread mapping | One logical worker per `(g, j)`, flat index `w = g·D + j`, 1-D grid-stride over `G·D`; one thread may process several workers in sequence. |
| Ownership | Worker `(g,j)` **exclusively owns** the column `P[g][·][j]` — the `K` elements at stride `D` — and, when `j = 0`, the count row `Q[g][·]`. Every element of `P` and `Q` has exactly one writer. `x_row` and `assignments` are read-only. |
| Synchronization | **None at all inside the kernel**: no atomics, no barriers. The only ordering needed is Stage 1 → Stage 2, which is a kernel boundary (DEC-010). |
| Initialization | Step 1 writes zero to every element the worker owns, on **every** call, so no value from an earlier iteration can survive (RED-006, MEM-006; verified by poisoning in TEST-U06 and by `initcheck` in TEST-S03). |
| Numerics | Ascending `i` within the chunk, so for fixed `(N, D, K, G)` the order is fixed (RED-009). |
| Coalescing | `x_row` is row-major, so for a fixed `i` the `D` workers of one chunk read consecutive addresses (CUDA-009). |

The chunk bounds are `chunk_g = [⌊gN/G⌋, ⌊(g+1)N/G⌋)`, formed in 64-bit: `g ≤ G ≤ N ≤ 2³¹−1`, so
`gN ≤ 2⁶²` and cannot overflow.

#### `reduce_stage2` — RED-003

| | |
|---|---|
| Thread mapping | One **32×8** block per tile `τ ∈ [0, K·T)` with `T = ⌈D/32⌉`, grid-stride over tiles. Lane `x ∈ [0,32)` selects `j = (τ mod T)·32 + x`; lane `y ∈ [0,8)` selects the strided subset `g = y, y+8, y+16, …`. |
| Ownership | Each element of `S` is written by exactly one thread — lane `(x, 0)` of the tile owning that `j` — and each `n[k]` by lane `(0,0)` of that cluster's `j`-tile 0. `P` and `Q` are read-only. Each shared slot is written only by its own thread and read by others only after a barrier. |
| Synchronization | Per tile: one barrier **before** the shared slots are overwritten (so the previous tile's tree reads have finished), one **after** they are filled, and one after each of the three tree levels. `τ` derives from `blockIdx` only, so the tile loop and the `j ≥ D` and `τ mod T = 0` guards are **block-uniform** — no barrier is reached by only part of the block (CUDA-005). |
| Numerics | Ascending `g` within each `y`-lane, then the fixed pairwise tree with strides 4, 2, 1. The order depends only on `(G, D, K)`, never on scheduling (RED-009). |

#### RED-004 — the race-freedom argument (reviewed by TEST-S05)

1. **Stage 1.** Each element of `P` and `Q` has exactly one owning worker, which performs all
   writes to it. `A` and `X^row` are read-only throughout. No two threads write the same address,
   so there are no data races and no lost updates — and therefore no need for atomics.
2. **Stage 2.** Each element of `S` and `n` is written by exactly one thread. `P` and `Q` are
   read-only. Shared-memory slots are written by their own thread before a barrier and read by
   others only after it.
3. **Boundaries.** Stage 1 → Stage 2 and Stage 2 → `update` are kernel boundaries on one stream
   (DEC-010); stream order guarantees a kernel observes all global writes of its predecessors. No
   grid-wide synchronization inside a kernel is assumed anywhere.

`racecheck` detects shared-memory hazards only, so the global-memory claim rests on this argument,
supported by TEST-R01 (a hidden race would normally surface as run-to-run variation) and TEST-U05.

#### RED-005 — exactly-once inclusion

The chunks partition `[0, N)`: consecutive bounds share endpoints, `chunk_0` starts at 0 and
`chunk_{G−1}` ends at `N`. Each worker covers every point of its chunk for its own `j`, and Stage 2
sums all `g ∈ [0, G)`. Hence every `(i, j)` contributes exactly once. Checked by `Σ_k n_k = N`
exactly and by `Σ_k S_{kj}` reproducing the column sums of `X` (TEST-U05, TEST-U07).

#### RED-006 — choice of `G`, and RED-008 trade-offs

`G = min(N, G_mem, max(1, ⌈T_target/D⌉))` with `T_target = 2048 × SM count` and
`G_mem = ⌊B_red / (8KD + 4K)⌋`, where `B_red = min(user cap, 0.25 × (free − other buffers))`
(MEM-003). `G` is fixed for the whole run, reported, and overridable by `--reduce-chunks`; an
override is still checked against `N` and the budget. If no usable `G` exists the run fails before
any allocation with exit code 3 and a message stating the required bytes (RED-007).

Larger `G` increases Stage 1 parallelism but also the memory and the Stage 2 combine work; smaller
`G` lengthens the per-worker sequential loop. Stage 1 parallelism is `G·D`, so small `D` depends on
a large `G` to fill the device. Stage 1's read-modify-writes to `P` go through L2 and DRAM, in
binary64 — on a device whose FP64 rate is 1/32 of FP32 (ENV-06) that cost is the central thing to
measure, not to predict (PERF-010).

Whether Approach B beats Approach A is an **empirical** question, answered in M6 with measurements.

In [ ]:
%%writefile launch_config.h
// launch_config.h — launch geometry and buffer sizing arithmetic (requirements.md CUDA-007,
// RED-006, MEM-003, MEM-004, SMEM-003, DEC-001).
//
// Deliberately free of CUDA constructs: every number a kernel launch needs is computed here from
// plain integers, so the G selection of RED-006 and the chunk partition of DEC-001 can be unit
// tested with g++ on a machine that has no GPU (CLAUDE.md). The .cu translation units include this
// header and only turn its results into launch arguments.

#ifndef KMEANS_LAUNCH_CONFIG_H
#define KMEANS_LAUNCH_CONFIG_H

#include <cstdint>
#include <string>

#include "common.h"

namespace kmeans {

// --- CUDA-007 grid sizing ----------------------------------------------------------------------

// Every kernel uses a 1-D grid with a grid-stride loop, so the grid may be smaller than the work:
// grid = min(ceil(work / block), 32 * SM count). A grid of 0 would launch nothing, so one block is
// the floor even for zero work; the bounds checks of CUDA-005 make that block a no-op.
inline int grid_size(int64_t work, int block, int sm_count)
{
    if (work <= 0 || block <= 0) {
        return 1;
    }
    const int64_t needed = (work + block - 1) / block;
    const int64_t cap = 32 * static_cast<int64_t>(sm_count > 0 ? sm_count : 1);
    const int64_t chosen = needed < cap ? needed : cap;
    return static_cast<int>(chosen < 1 ? 1 : chosen);
}

// `movement` writes one maximum per block and `max_reduce` reduces that array in a single block.
// Capping the movement grid at 1024 (the "unless overridden" clause of CUDA-007) serves two
// purposes: max_reduce can use one thread per block maximum in a single 1024-thread block, and the
// cap equals the `movement_blocks` default of device_memory_bytes (common.h), so the MEM-002
// capacity model and the real allocation agree.
constexpr int kMaxMovementBlocks = 1024;

inline int movement_grid_size(int64_t K, int block, int sm_count)
{
    const int grid = grid_size(K, block, sm_count);
    return grid < kMaxMovementBlocks ? grid : kMaxMovementBlocks;
}

// --- Section 7 Stage 2 tile geometry -----------------------------------------------------------
// RED-003 fixes the Stage 2 block at 32 x 8: lane x selects j within a 32-dimension tile, lane y
// selects a strided subset of g. Both are compile-time constants because the fixed pairwise tree
// (strides 4, 2, 1) is written for exactly eight y-lanes.
constexpr int kStage2DimTile = 32;
constexpr int kStage2ChunkLanes = 8;

// Number of 32-dimension tiles per cluster, T = ceil(D / 32).
inline int64_t stage2_dim_tiles(int64_t D)
{
    return (D + kStage2DimTile - 1) / kStage2DimTile;
}

// Total Stage 2 tiles, K * T, enumerated by a 1-D grid-stride index (DEC-001).
inline int64_t stage2_tiles(int64_t K, int64_t D)
{
    return K * stage2_dim_tiles(D);
}

// --- SMEM-001 / SMEM-003 assign_smem tile defaults ---------------------------------------------
// Declared here so the startup capacity check and VAL-010 can see them before M4 implements the
// kernel; shared memory per block is 4 * T_K * T_D bytes (1 KiB at the defaults).
constexpr int kAssignTileK = 8;
constexpr int kAssignTileD = 32;

inline int64_t assign_smem_bytes(int tile_k, int tile_d)
{
    return 4 * static_cast<int64_t>(tile_k) * static_cast<int64_t>(tile_d);
}

// --- MEM-004 shared-memory footprints ----------------------------------------------------------

// `assign` reduces its per-thread change flags through one 8-byte slot per thread and `movement`
// one 8-byte partial maximum per thread, so both are at most 8 * block size bytes.
inline int64_t block_reduction_bytes(int block)
{
    return 8 * static_cast<int64_t>(block);
}

// `reduce_stage2`: 8 * 32 binary64 sums plus 8 uint32 counts = 2080 B, independent of the launch.
inline int64_t stage2_shared_bytes()
{
    return 8 * static_cast<int64_t>(kStage2ChunkLanes) * kStage2DimTile +
           4 * static_cast<int64_t>(kStage2ChunkLanes);
}

// --- RED-006 choice of G -----------------------------------------------------------------------

// MEM-003: B_red = min(user cap if given, 0.25 * (free - other buffers)). `free_bytes` is what
// cudaMemGetInfo reports and `other_bytes` is everything in MEM-001 except P and Q. A
// configuration whose non-reduction buffers already exceed free memory yields a budget of 0,
// which then fails the G_mem >= 1 requirement of RED-007 with a byte count in the message.
inline uint64_t reduction_budget_bytes(uint64_t free_bytes, uint64_t other_bytes,
                                       uint64_t user_cap_bytes)
{
    const uint64_t headroom = free_bytes > other_bytes ? free_bytes - other_bytes : 0;
    uint64_t budget = static_cast<uint64_t>(static_cast<double>(headroom) * 0.25);
    if (user_cap_bytes > 0 && user_cap_bytes < budget) {
        budget = user_cap_bytes;
    }
    return budget;
}

struct ChunkSelection {
    int64_t G = 0;                 // the selected chunk count, 0 when ok == false
    int64_t g_memory = 0;          // G_mem = floor(B_red / (8KD + 4K))
    int64_t g_target = 0;          // max(1, ceil(T_target / D))
    uint64_t budget_bytes = 0;     // B_red
    uint64_t bytes_per_chunk = 0;  // 8KD + 4K
    uint64_t required_bytes = 0;   // bytes for the selected G, or for the G that did not fit
    bool user_override = false;
    bool ok = true;
    std::string message;  // RED-007 explanation when ok == false
};

// G = min(N, G_mem, max(1, ceil(T_target / D))) with T_target = 2048 * SM count (DEC-001).
// `user_G` > 0 is the --reduce-chunks override. It is still checked against N and the budget: an
// override that does not fit must fail loudly rather than silently allocate past the budget.
inline ChunkSelection choose_reduce_chunks(int64_t N, int64_t D, int64_t K, int sm_count,
                                           uint64_t free_bytes, uint64_t other_bytes,
                                           uint64_t user_cap_bytes, int64_t user_G)
{
    ChunkSelection sel;
    sel.budget_bytes = reduction_budget_bytes(free_bytes, other_bytes, user_cap_bytes);

    uint64_t kd = 0, per_chunk = 0, counts = 0;
    const bool size_ok = checked_mul(static_cast<uint64_t>(K), static_cast<uint64_t>(D), &kd) &&
                         checked_mul(kd, 8, &per_chunk) &&
                         checked_mul(static_cast<uint64_t>(K), 4, &counts) &&
                         checked_add(per_chunk, counts, &per_chunk);
    if (!size_ok || per_chunk == 0) {
        sel.ok = false;
        sel.message = "the Approach B partial-sum buffer size 8KD + 4K overflows 64-bit "
                      "arithmetic for K=" + std::to_string(K) + " D=" + std::to_string(D);
        return sel;
    }
    sel.bytes_per_chunk = per_chunk;
    sel.g_memory = static_cast<int64_t>(sel.budget_bytes / per_chunk);

    // T_target = 2048 * SM count: enough workers to fill the device several times over, divided
    // among the D workers that each chunk already provides.
    const int64_t t_target = 2048 * static_cast<int64_t>(sm_count > 0 ? sm_count : 1);
    const int64_t by_target = (t_target + D - 1) / D;
    sel.g_target = by_target > 1 ? by_target : 1;

    int64_t g = sel.g_target;
    if (g > N) {
        g = N;
    }
    if (user_G > 0) {
        sel.user_override = true;
        g = user_G;
        if (g > N) {
            sel.ok = false;
            sel.message = "--reduce-chunks " + std::to_string(user_G) + " exceeds N=" +
                          std::to_string(N) + "; DEC-001 requires 1 <= G <= N";
            return sel;
        }
    } else if (g > sel.g_memory) {
        g = sel.g_memory;
    }

    if (g < 1 || g > sel.g_memory) {
        // RED-007: no usable G. Report what the requested G costs, or what one chunk would cost
        // when even that does not fit, so the message states how much memory the run needs.
        const int64_t wanted = g < 1 ? 1 : g;
        sel.ok = false;
        sel.required_bytes = per_chunk * static_cast<uint64_t>(wanted);
        sel.message = "Approach B needs " + std::to_string(sel.required_bytes) +
                      " B of partial-sum buffers for G=" + std::to_string(wanted) +
                      " (8KD + 4K = " + std::to_string(per_chunk) + " B per chunk), but the "
                      "MEM-003 budget is only " + std::to_string(sel.budget_bytes) + " B";
        return sel;
    }

    sel.G = g;
    sel.required_bytes = per_chunk * static_cast<uint64_t>(g);
    return sel;
}

// --- DEC-001 chunk partition -------------------------------------------------------------------
// chunk_g = [floor(gN/G), floor((g+1)N/G)). The products are formed in int64: g <= G <= N and
// N <= 2^31 - 1, so gN <= 2^62 and cannot overflow. Consecutive bounds share endpoints, the first
// chunk starts at 0 and the last ends at N, so the chunks partition [0, N) exactly (RED-005). The
// device code uses the identical expression.
inline int64_t chunk_begin(int64_t g, int64_t N, int64_t G)
{
    return (g * N) / G;
}

inline int64_t chunk_end(int64_t g, int64_t N, int64_t G)
{
    return ((g + 1) * N) / G;
}

}  // namespace kmeans

#endif  // KMEANS_LAUNCH_CONFIG_H


In [ ]:
%%writefile cuda_util.cuh
// cuda_util.cuh — CUDA error checking (CUDA-008) and RAII ownership of device resources
// (ARC-04, MEM-007).
//
// Two rules this header exists to enforce:
//
//   * Every CUDA runtime call is checked, and a failure names the file, line, call text and
//     cudaGetErrorString (CUDA-008). Kernel launches are followed by cudaGetLastError, which
//     catches launch-configuration faults; asynchronous execution faults surface at the
//     per-iteration stream synchronisation, or immediately when KMEANS_SYNC_CHECK is defined.
//   * Every device and pinned allocation is owned by a destructor, so an error path that returns
//     early still frees it (MEM-007, verified by --leak-check full in TEST-S01).
//
// Failures are reported through a thrown CudaError rather than exit(): the caller (the driver or a
// test) decides whether to translate it into exit code 4, because a test needs to keep running
// after a deliberately provoked failure (EDGE-27, EDGE-28).

#ifndef KMEANS_CUDA_UTIL_CUH
#define KMEANS_CUDA_UTIL_CUH

#include <cuda_runtime.h>

#include <cstdio>
#include <stdexcept>
#include <string>
#include <utility>

#include "common.h"

namespace kmeans {

// BLD-006 message shape, carried as an exception so RAII owners unwind on the way out.
class CudaError : public std::runtime_error {
public:
    CudaError(const std::string &message, int exit_code)
        : std::runtime_error(message), exit_code_(exit_code)
    {
    }
    int exit_code() const { return exit_code_; }

private:
    int exit_code_;
};

inline void cuda_throw(cudaError_t status, const char *call, const char *file, int line)
{
    const std::string message = std::string("CUDA call '") + call + "' failed at " + file + ":" +
                                std::to_string(line) + ": " + cudaGetErrorName(status) + ": " +
                                cudaGetErrorString(status);
    // Clear the sticky error so a test that provoked this deliberately can continue.
    cudaGetLastError();
    throw CudaError(message, kExitCudaError);
}

#define KMEANS_CUDA_CHECK(call)                                                   \
    do {                                                                          \
        const cudaError_t kmeans_status_ = (call);                                \
        if (kmeans_status_ != cudaSuccess) {                                      \
            kmeans::cuda_throw(kmeans_status_, #call, __FILE__, __LINE__);         \
        }                                                                         \
    } while (0)

// CUDA-008: after every launch, cudaGetLastError reports launch errors. With KMEANS_SYNC_CHECK
// defined (the debug build of BLD-002) a synchronisation follows, so an asynchronous fault is
// attributed to the kernel that caused it instead of to the next synchronisation point.
#ifdef KMEANS_SYNC_CHECK
#define KMEANS_CHECK_LAUNCH(name)                                                 \
    do {                                                                          \
        KMEANS_CUDA_CHECK(cudaGetLastError());                                    \
        KMEANS_CUDA_CHECK(cudaDeviceSynchronize());                               \
    } while (0)
#else
#define KMEANS_CHECK_LAUNCH(name)                                                 \
    do {                                                                          \
        KMEANS_CUDA_CHECK(cudaGetLastError());                                    \
    } while (0)
#endif

// --- device memory -----------------------------------------------------------------------------

// Owning, move-only device array. Copying is deleted because two owners would double-free.
template <typename T>
class DeviceBuffer {
public:
    DeviceBuffer() = default;

    explicit DeviceBuffer(size_t count) { allocate(count); }

    ~DeviceBuffer() { release(); }

    DeviceBuffer(const DeviceBuffer &) = delete;
    DeviceBuffer &operator=(const DeviceBuffer &) = delete;

    DeviceBuffer(DeviceBuffer &&other) noexcept : ptr_(other.ptr_), count_(other.count_)
    {
        other.ptr_ = nullptr;
        other.count_ = 0;
    }

    DeviceBuffer &operator=(DeviceBuffer &&other) noexcept
    {
        if (this != &other) {
            release();
            ptr_ = other.ptr_;
            count_ = other.count_;
            other.ptr_ = nullptr;
            other.count_ = 0;
        }
        return *this;
    }

    void allocate(size_t count)
    {
        release();
        if (count == 0) {
            return;  // a zero-element buffer is legal and stays null; no kernel indexes into it
        }
        void *raw = nullptr;
        KMEANS_CUDA_CHECK(cudaMalloc(&raw, count * sizeof(T)));
        ptr_ = static_cast<T *>(raw);
        count_ = count;
    }

    void release()
    {
        if (ptr_ != nullptr) {
            // A free failure during unwinding must not throw again, so it is reported, not raised.
            const cudaError_t status = cudaFree(ptr_);
            if (status != cudaSuccess) {
                std::fprintf(stderr, "error[MEM-007]: cudaFree failed: %s\n",
                             cudaGetErrorString(status));
            }
            ptr_ = nullptr;
            count_ = 0;
        }
    }

    T *get() const { return ptr_; }
    size_t count() const { return count_; }
    size_t bytes() const { return count_ * sizeof(T); }

    void copy_from_host(const T *src, size_t count, cudaStream_t stream)
    {
        if (count == 0) {
            return;
        }
        KMEANS_CUDA_CHECK(
            cudaMemcpyAsync(ptr_, src, count * sizeof(T), cudaMemcpyHostToDevice, stream));
    }

    void copy_to_host(T *dst, size_t count, cudaStream_t stream) const
    {
        if (count == 0) {
            return;
        }
        KMEANS_CUDA_CHECK(
            cudaMemcpyAsync(dst, ptr_, count * sizeof(T), cudaMemcpyDeviceToHost, stream));
    }

    // Byte-pattern fill. Used for the zeroing of ATOM-001 / MEM-005 and, with 0xFF, for the
    // buffer poisoning of TEST-U06.
    void fill_bytes(int value, cudaStream_t stream)
    {
        if (ptr_ == nullptr || count_ == 0) {
            return;
        }
        KMEANS_CUDA_CHECK(cudaMemsetAsync(ptr_, value, count_ * sizeof(T), stream));
    }

private:
    T *ptr_ = nullptr;
    size_t count_ = 0;
};

// Pinned host memory for the 16-byte per-iteration convergence copy (CUDA-011, MEM-001).
template <typename T>
class PinnedBuffer {
public:
    PinnedBuffer() = default;
    explicit PinnedBuffer(size_t count) { allocate(count); }
    ~PinnedBuffer() { release(); }

    PinnedBuffer(const PinnedBuffer &) = delete;
    PinnedBuffer &operator=(const PinnedBuffer &) = delete;

    void allocate(size_t count)
    {
        release();
        if (count == 0) {
            return;
        }
        void *raw = nullptr;
        KMEANS_CUDA_CHECK(cudaHostAlloc(&raw, count * sizeof(T), cudaHostAllocDefault));
        ptr_ = static_cast<T *>(raw);
        count_ = count;
    }

    void release()
    {
        if (ptr_ != nullptr) {
            const cudaError_t status = cudaFreeHost(ptr_);
            if (status != cudaSuccess) {
                std::fprintf(stderr, "error[MEM-007]: cudaFreeHost failed: %s\n",
                             cudaGetErrorString(status));
            }
            ptr_ = nullptr;
            count_ = 0;
        }
    }

    T *get() const { return ptr_; }

private:
    T *ptr_ = nullptr;
    size_t count_ = 0;
};

// DEC-010: one stream realises every global phase boundary, so the stream is a run-scoped
// resource with the same ownership discipline as the buffers.
class CudaStream {
public:
    CudaStream() { KMEANS_CUDA_CHECK(cudaStreamCreate(&stream_)); }

    ~CudaStream()
    {
        if (stream_ != nullptr) {
            cudaStreamDestroy(stream_);
        }
    }

    CudaStream(const CudaStream &) = delete;
    CudaStream &operator=(const CudaStream &) = delete;

    cudaStream_t get() const { return stream_; }
    void synchronize() const { KMEANS_CUDA_CHECK(cudaStreamSynchronize(stream_)); }

private:
    cudaStream_t stream_ = nullptr;
};

// PERF-005: T_alg is measured with CUDA events recorded on the stream.
class CudaEvent {
public:
    CudaEvent() { KMEANS_CUDA_CHECK(cudaEventCreate(&event_)); }

    ~CudaEvent()
    {
        if (event_ != nullptr) {
            cudaEventDestroy(event_);
        }
    }

    CudaEvent(const CudaEvent &) = delete;
    CudaEvent &operator=(const CudaEvent &) = delete;

    void record(cudaStream_t stream) { KMEANS_CUDA_CHECK(cudaEventRecord(event_, stream)); }
    cudaEvent_t get() const { return event_; }

    // PERF-005: elapsed time is read only after synchronising on the end event.
    static float elapsed_ms(const CudaEvent &start, const CudaEvent &end)
    {
        KMEANS_CUDA_CHECK(cudaEventSynchronize(end.event_));
        float ms = 0.0f;
        KMEANS_CUDA_CHECK(cudaEventElapsedTime(&ms, start.event_, end.event_));
        return ms;
    }

private:
    cudaEvent_t event_ = nullptr;
};

}  // namespace kmeans

#endif  // KMEANS_CUDA_UTIL_CUH


In [ ]:
%%writefile device_context.cuh
// device_context.cuh — device selection, capability record and capacity checks (ARC-04,
// CUDA-001, VAL-009, MEM-002, MEM-004, SMEM-003, RED-006/007).
//
// Everything here runs before the first kernel launch. The order matters: the device is queried,
// the capability gate of CUDA-001 is applied, the chunk count G of RED-006 is chosen from the free
// memory actually reported, and only then is the MEM-002 total compared with free memory
// (VAL-009). No allocation happens until all of that holds, which is what lets EDGE-26 fail with
// exit code 3 and a byte count instead of a CUDA out-of-memory error.
//
// Header-only: it is pure host code over the CUDA runtime API, so inlining it avoids a translation
// unit that exists only to call cudaGetDeviceProperties.

#ifndef KMEANS_DEVICE_CONTEXT_CUH
#define KMEANS_DEVICE_CONTEXT_CUH

#include <cuda_runtime.h>

#include <string>

#include "common.h"
#include "cuda_util.cuh"
#include "launch_config.h"

namespace kmeans {

// CUDA-001: the properties recorded at startup. These are the same fields the environment record
// of PERF-008 carries, so a benchmark row can be traced to the device that produced it.
struct DeviceInfo {
    int device = 0;
    std::string name;
    int cc_major = 0;
    int cc_minor = 0;
    int sm_count = 0;
    int max_threads_per_block = 0;
    int warp_size = 0;
    size_t shared_mem_per_block = 0;      // default limit
    size_t shared_mem_per_block_optin = 0;  // opt-in maximum
    size_t total_global_mem = 0;
    size_t free_global_mem = 0;
    int runtime_version = 0;
    int driver_version = 0;

    double compute_capability() const { return cc_major + cc_minor / 10.0; }
    // DEC-007 / ATOM-003: native atomicAdd(double) exists from compute capability 6.0.
    bool has_native_fp64_atomics() const { return compute_capability() >= 6.0; }
};

inline DeviceInfo query_device(int device)
{
    DeviceInfo info;
    int count = 0;
    KMEANS_CUDA_CHECK(cudaGetDeviceCount(&count));
    if (count <= 0) {
        throw CudaError("no CUDA device is visible (CUDA-001)", kExitUnsupportedDevice);
    }
    if (device < 0 || device >= count) {
        throw CudaError("device " + std::to_string(device) + " does not exist; " +
                            std::to_string(count) + " device(s) visible",
                        kExitUnsupportedDevice);
    }
    KMEANS_CUDA_CHECK(cudaSetDevice(device));

    cudaDeviceProp prop{};
    KMEANS_CUDA_CHECK(cudaGetDeviceProperties(&prop, device));
    info.device = device;
    info.name = prop.name;
    info.cc_major = prop.major;
    info.cc_minor = prop.minor;
    info.sm_count = prop.multiProcessorCount;
    info.max_threads_per_block = prop.maxThreadsPerBlock;
    info.warp_size = prop.warpSize;
    info.shared_mem_per_block = prop.sharedMemPerBlock;
    info.shared_mem_per_block_optin = prop.sharedMemPerBlockOptin;
    info.total_global_mem = prop.totalGlobalMem;
    KMEANS_CUDA_CHECK(cudaRuntimeGetVersion(&info.runtime_version));
    KMEANS_CUDA_CHECK(cudaDriverGetVersion(&info.driver_version));

    size_t free_bytes = 0, total_bytes = 0;
    KMEANS_CUDA_CHECK(cudaMemGetInfo(&free_bytes, &total_bytes));
    info.free_global_mem = free_bytes;

    // CUDA-001: reject below compute capability 6.0 with exit code 3. Below 6.0 there is no native
    // atomicAdd(double), which ATOM-003 requires and for which this project has no CAS fallback.
    if (info.compute_capability() < 6.0) {
        throw CudaError("device '" + info.name + "' has compute capability " +
                            std::to_string(info.cc_major) + "." + std::to_string(info.cc_minor) +
                            ", below the 6.0 minimum of CUDA-001 (native atomicAdd(double) is "
                            "required by ATOM-003)",
                        kExitUnsupportedDevice);
    }
    return info;
}

inline std::string describe_device(const DeviceInfo &info)
{
    return info.name + " (cc " + std::to_string(info.cc_major) + "." +
           std::to_string(info.cc_minor) + ", " + std::to_string(info.sm_count) + " SMs, " +
           std::to_string(info.free_global_mem / (1024 * 1024)) + " MiB free of " +
           std::to_string(info.total_global_mem / (1024 * 1024)) + " MiB, shared/block " +
           std::to_string(info.shared_mem_per_block) + " B)";
}

// MEM-004, SMEM-003: every per-block shared-memory request is checked against the device limit at
// startup rather than discovered as a launch failure. The movement and assign reductions scale
// with the block size, so the check depends on the validated block size.
inline void check_shared_memory(const DeviceInfo &info, int block_size, int tile_k, int tile_d)
{
    const int64_t limit = static_cast<int64_t>(info.shared_mem_per_block);
    struct Request {
        const char *rule;
        const char *what;
        int64_t bytes;
    };
    const Request requests[] = {
        {"MEM-004", "assign change-count reduction", block_reduction_bytes(block_size)},
        {"MEM-004", "movement block maximum reduction", block_reduction_bytes(block_size)},
        // max_reduce sizes its single block from the number of per-block maxima, not from the
        // configured block size, so it is checked at its own maximum rather than at block_size.
        {"MEM-004", "max_reduce final reduction", block_reduction_bytes(kMaxMovementBlocks)},
        {"MEM-004", "reduce_stage2 sums and counts", stage2_shared_bytes()},
        {"SMEM-003", "assign_smem centroid tile", assign_smem_bytes(tile_k, tile_d)},
    };
    for (const Request &r : requests) {
        if (r.bytes > limit) {
            throw CudaError(std::string("[") + r.rule + "] the " + r.what + " needs " +
                                std::to_string(r.bytes) +
                                " B of shared memory per block, but the device offers only " +
                                std::to_string(limit) + " B",
                            kExitInvalidInput);
        }
    }
}

// Bytes of MEM-001 other than the Approach B partial buffers P and Q. This is the `other_bytes`
// input of the MEM-003 reduction budget: the budget is a quarter of what is left after the
// buffers every run needs.
inline uint64_t non_reduction_bytes(int64_t N, int64_t D, int64_t K, int64_t movement_blocks)
{
    const MemoryModel m = device_memory_bytes(N, D, K, 0, movement_blocks);
    return m.overflow ? UINT64_MAX : m.total;
}

// RED-006 / RED-007 for a concrete device. Approach A passes `strategy == kAtomic` and gets G = 0,
// since it allocates no partial buffers.
inline ChunkSelection select_chunks(const DeviceInfo &info, const Config &config,
                                    uint64_t user_cap_bytes)
{
    ChunkSelection sel;
    if (config.strategy != Strategy::kReduction) {
        sel.G = 0;
        return sel;
    }
    const int movement_blocks = movement_grid_size(config.K, config.block_size, info.sm_count);
    const uint64_t other = non_reduction_bytes(config.N, config.D, config.K, movement_blocks);
    sel = choose_reduce_chunks(config.N, config.D, config.K, info.sm_count,
                               static_cast<uint64_t>(info.free_global_mem), other, user_cap_bytes,
                               config.reduce_chunks);
    if (!sel.ok) {
        // RED-007: fail before allocation with the required byte count, as exit code 3.
        throw CudaError("[RED-007] " + sel.message, kExitUnsupportedDevice);
    }
    return sel;
}

// VAL-009 against the free memory of this device, with the selected G included in the model.
inline void check_capacity(const DeviceInfo &info, const Config &config, int64_t reduce_chunks)
{
    const int movement_blocks = movement_grid_size(config.K, config.block_size, info.sm_count);
    const MemoryModel m =
        device_memory_bytes(config.N, config.D, config.K, reduce_chunks, movement_blocks);
    if (m.overflow) {
        throw CudaError("[VAL-009] required device memory overflows 64-bit arithmetic for N=" +
                            std::to_string(config.N) + " D=" + std::to_string(config.D) +
                            " K=" + std::to_string(config.K) +
                            " G=" + std::to_string(reduce_chunks),
                        kExitInvalidInput);
    }
    const uint64_t budget =
        static_cast<uint64_t>(static_cast<double>(info.free_global_mem) * 0.95);
    if (m.total > budget) {
        throw CudaError("[VAL-009] required " + std::to_string(m.total) +
                            " B exceeds the usable " + std::to_string(budget) + " B (95 % of " +
                            std::to_string(info.free_global_mem) + " B free on " + info.name + ")",
                        kExitUnsupportedDevice);
    }
}

}  // namespace kmeans

#endif  // KMEANS_DEVICE_CONTEXT_CUH


In [ ]:
%%writefile kernels.cuh
// kernels.cuh — host-side launch interface for the CUDA-002 kernel set (ARC-05, ARC-06, ARC-07,
// ARC-08).
//
// The kernels themselves stay inside kernels.cu. The driver only ever calls the launchers below,
// which keeps every launch configuration in one place (CUDA-007) and means no translation unit
// other than kernels.cu contains a kernel launch, so the build needs no relocatable device code.
//
// Every launcher is stream-ordered on the single stream of DEC-010 and checks cudaGetLastError
// after the launch (CUDA-008).

#ifndef KMEANS_KERNELS_CUH
#define KMEANS_KERNELS_CUH

#include <cuda_runtime.h>

#include <cstdint>

#include "common.h"
#include "launch_config.h"

namespace kmeans {
namespace gpu {

// The 16 bytes copied to pinned host memory once per iteration (CUDA-011, MEM-001). Keeping both
// scalars in one device object is what makes it a single 16-byte transfer, which is what the
// transfer-discipline check of TEST-P03 looks for in the Nsight Systems trace.
struct ConvergenceScalars {
    double delta;                // Delta of the update just performed (section 2.6)
    unsigned long long changed;  // number of points whose assignment differed (FR-007)
};

// FR-004..FR-007, DEC-008: one thread per point, grid-stride, ascending k and j.
// `count_changes` is false for the initial pass of section 2.7, which has no previous assignment
// to compare against and therefore never reads A.
void launch_assign(const float *x_col, int64_t N, int64_t D, const float *centroids, int64_t K,
                   int32_t *assignments, unsigned long long *changed, bool count_changes,
                   int block, int sm_count, cudaStream_t stream);

// Section 6, ATOM-002..ATOM-004: one thread per element of X, grid-stride over N*D.
void launch_accum_atomic(const float *x_col, int64_t N, int64_t D, const int32_t *assignments,
                         int64_t K, double *sums, unsigned int *counts, int block, int sm_count,
                         cudaStream_t stream);

// RED-002: one logical worker per (g, j), grid-stride over G*D.
void launch_reduce_stage1(const float *x_row, int64_t N, int64_t D, const int32_t *assignments,
                          int64_t K, int64_t G, double *partials, unsigned int *partial_counts,
                          int block, int sm_count, cudaStream_t stream);

// RED-003: one 32x8 block per (cluster, 32-dimension tile), grid-stride over tiles.
void launch_reduce_stage2(const double *partials, const unsigned int *partial_counts, int64_t D,
                          int64_t K, int64_t G, double *sums, unsigned int *counts, int sm_count,
                          cudaStream_t stream);

// Section 2.5 / FR-010 / FR-011: grid-stride over K*D into the next centroid buffer.
void launch_update(const double *sums, const unsigned int *counts, int64_t K, int64_t D,
                   const float *centroids_cur, float *centroids_next, int block, int sm_count,
                   cudaStream_t stream);

// Section 2.6 / FR-012: delta_k^2 per cluster, then one maximum per block.
void launch_movement(const float *centroids_cur, const float *centroids_next,
                     const unsigned int *counts, int64_t K, int64_t D, double *block_max,
                     int block, int sm_count, cudaStream_t stream);

// Final maximum over the per-block maxima in a single block, with the one square root.
void launch_max_reduce(const double *block_max, int block_count, double *delta,
                       cudaStream_t stream);

}  // namespace gpu
}  // namespace kmeans

#endif  // KMEANS_KERNELS_CUH


In [ ]:
%%writefile kernels.cu
// kernels.cu — the CUDA-002 kernel set.
//
// Thread mapping, data ownership and synchronisation points for each kernel are documented at the
// kernel and summarised in the notebook (tools/cells/05_kernel_contracts.md and
// 05_reduction_contracts.md). Three rules hold throughout:
//
//   * Every global index is guarded against its range (CUDA-005), and threads that fall outside
//     skip work but still reach every __syncthreads().
//   * Every element offset is formed in unsigned 64-bit arithmetic (CUDA-006).
//   * No kernel reads data written by another block of the same launch; phase boundaries are
//     kernel boundaries on one stream (CUDA-004, DEC-010).

#include "kernels.cuh"

#include "cuda_util.cuh"

namespace kmeans {
namespace gpu {
namespace {

// Grid-stride step, in 64-bit (CUDA-006).
__device__ inline uint64_t grid_stride()
{
    return static_cast<uint64_t>(gridDim.x) * static_cast<uint64_t>(blockDim.x);
}

__device__ inline uint64_t global_thread_index()
{
    return static_cast<uint64_t>(blockIdx.x) * static_cast<uint64_t>(blockDim.x) +
           static_cast<uint64_t>(threadIdx.x);
}

// CUDA-007 validates the block size as a multiple of 32, which is not necessarily a power of two
// (96, 160, 192 are all legal). The block reductions below therefore start from the next power of
// two at or above blockDim.x and guard the partner index, which is correct for any block size.
__device__ inline unsigned reduction_start_stride(unsigned block_dim)
{
    unsigned stride = 1;
    while (stride < block_dim) {
        stride <<= 1;
    }
    return stride >> 1;
}

// --- FR-004..FR-007: assignment ----------------------------------------------------------------
//
// Thread mapping   one thread per point i, 1-D grid-stride (DEC-008).
// Ownership        thread i is the only reader and writer of assignments[i], so the in-place
//                  update of FR-007 needs no synchronisation. x_col and centroids are read-only.
// Synchronisation  none for the assignment itself. The change count is reduced through shared
//                  memory, so every thread of the block reaches every barrier, including threads
//                  whose grid-stride loop ran zero times (CUDA-005).
// Numerics         NUM-002: terms in binary32, accumulated in binary32 in ascending j from 0, and
//                  never reordered. DEC-005: k ascends from 0 and the best candidate is replaced
//                  only on a strictly smaller distance, so exactly equal computed distances keep
//                  the lowest index.
// Coalescing       x_col is dimension-major, so for fixed j consecutive threads read consecutive
//                  addresses; the centroid read is uniform across the warp and broadcasts
//                  (CUDA-009).
__global__ void assign_kernel(const float *__restrict__ x_col, int64_t N, int64_t D,
                              const float *__restrict__ centroids, int64_t K,
                              int32_t *__restrict__ assignments,
                              unsigned long long *__restrict__ changed, int count_changes)
{
    extern __shared__ unsigned long long s_changed[];
    const unsigned tid = threadIdx.x;
    const uint64_t n = static_cast<uint64_t>(N);
    unsigned long long local_changed = 0;

    for (uint64_t i = global_thread_index(); i < n; i += grid_stride()) {
        // k = 0 initialises the best candidate (DEC-005).
        float best = 0.0f;
        for (int64_t j = 0; j < D; ++j) {
            const float diff = x_col[static_cast<uint64_t>(j) * n + i] - centroids[j];
            best += diff * diff;
        }
        int32_t best_k = 0;

        for (int64_t k = 1; k < K; ++k) {
            const float *centroid = centroids + static_cast<uint64_t>(k) * static_cast<uint64_t>(D);
            float distance = 0.0f;
            for (int64_t j = 0; j < D; ++j) {
                const float diff = x_col[static_cast<uint64_t>(j) * n + i] - centroid[j];
                distance += diff * diff;
            }
            if (distance < best) {  // strictly less: ties keep the lower k
                best = distance;
                best_k = static_cast<int32_t>(k);
            }
        }

        if (count_changes != 0 && assignments[i] != best_k) {
            ++local_changed;
        }
        assignments[i] = best_k;
    }

    // CUDA-002: one 64-bit atomicAdd per block, not per point. Integer atomics are exact, so the
    // counter is deterministic and does not compromise the atomic-free accumulation of Approach B.
    s_changed[tid] = local_changed;
    __syncthreads();
    for (unsigned stride = reduction_start_stride(blockDim.x); stride > 0; stride >>= 1) {
        if (tid < stride && tid + stride < blockDim.x) {
            s_changed[tid] += s_changed[tid + stride];
        }
        __syncthreads();
    }
    if (tid == 0 && count_changes != 0) {
        atomicAdd(changed, s_changed[0]);
    }
}

// --- Section 6: Approach A, atomic accumulation ------------------------------------------------
//
// Thread mapping   one thread per element e of X, 1-D grid-stride over N*D, with
//                  i = e mod N and j = e / N over the dimension-major copy (ATOM-002, CUDA-009).
//                  Since x_col holds element (i, j) at j*N + i, that index is exactly e, so the
//                  read is a straight sequential walk and coalesces.
// Ownership        sums and counts are shared by every thread; all writes are atomic
//                  read-modify-writes, so no update is lost (ATOM-007). The caller zeroes both
//                  buffers on the same stream before the launch (ATOM-001).
// Synchronisation  none inside the kernel. The ordering against `update` is the kernel boundary
//                  of ATOM-005 / DEC-010.
// Numerics         binary64 accumulation (DEC-007) in an order the hardware chooses, so S may
//                  differ in its last bits between runs; counts are integers and are exact.
__global__ void accum_atomic_kernel(const float *__restrict__ x_col, int64_t N, int64_t D,
                                    const int32_t *__restrict__ assignments, int64_t K,
                                    double *__restrict__ sums, unsigned int *__restrict__ counts)
{
    const uint64_t n = static_cast<uint64_t>(N);
    const uint64_t total = n * static_cast<uint64_t>(D);

    for (uint64_t e = global_thread_index(); e < total; e += grid_stride()) {
        const uint64_t i = e % n;
        const uint64_t j = e / n;
        const int32_t k = assignments[i];
        // CUDA-005: every global index is guarded. A is always in [0, K) because `assign`
        // produced it; the guard keeps a deliberately corrupted A (fault injection, poisoning)
        // from writing outside the buffers, so memcheck stays clean in those tests.
        if (k < 0 || static_cast<int64_t>(k) >= K) {
            continue;
        }
        atomicAdd(&sums[static_cast<uint64_t>(k) * static_cast<uint64_t>(D) + j],
                  static_cast<double>(x_col[e]));
        if (j == 0) {
            atomicAdd(&counts[k], 1u);  // ATOM-004
        }
    }
}

// --- Section 7 Stage 1: ownership-based privatised accumulation --------------------------------
//
// Thread mapping   one logical worker per (g, j) with flat index w = g*D + j, 1-D grid-stride
//                  over G*D; a thread may process several workers in sequence (RED-002).
// Ownership        worker (g, j) exclusively owns the column P[g][.][j] -- the K elements at
//                  stride D -- and, when j == 0, the count row Q[g][.]. Every element of P and Q
//                  has exactly one writer, so there are no write-write races, no atomics and no
//                  lost updates (RED-004). x_row and assignments are read-only here.
// Synchronisation  none at all inside the kernel. The only ordering needed is Stage 1 -> Stage 2,
//                  which is a kernel boundary (DEC-010).
// Initialisation   step 1 writes zero to every element the worker owns on every call, so no stale
//                  value from an earlier iteration can survive (RED-006, MEM-006, TEST-U06).
// Numerics         RED-009: ascending i within the chunk, so for fixed (N, D, K, G) the order is
//                  fixed and the result is expected to be bitwise reproducible.
// Coalescing       x_row is row-major, so for fixed i the D workers of one chunk read consecutive
//                  addresses (CUDA-009).
__global__ void reduce_stage1_kernel(const float *__restrict__ x_row, int64_t N, int64_t D,
                                     const int32_t *__restrict__ assignments, int64_t K, int64_t G,
                                     double *__restrict__ partials,
                                     unsigned int *__restrict__ partial_counts)
{
    const uint64_t d = static_cast<uint64_t>(D);
    const uint64_t workers = static_cast<uint64_t>(G) * d;

    for (uint64_t w = global_thread_index(); w < workers; w += grid_stride()) {
        const int64_t g = static_cast<int64_t>(w / d);
        const int64_t j = static_cast<int64_t>(w % d);

        // P[g][k][j] lives at (g*K + k)*D + j, so the owned column starts here and strides by D.
        double *owned_sums =
            partials + static_cast<uint64_t>(g) * static_cast<uint64_t>(K) * d + static_cast<uint64_t>(j);
        unsigned int *owned_counts =
            (j == 0) ? partial_counts + static_cast<uint64_t>(g) * static_cast<uint64_t>(K)
                     : nullptr;

        for (int64_t k = 0; k < K; ++k) {
            owned_sums[static_cast<uint64_t>(k) * d] = 0.0;
            if (owned_counts != nullptr) {
                owned_counts[k] = 0u;
            }
        }

        // DEC-001 chunk bounds in 64-bit: g <= G <= N <= 2^31 - 1, so g*N <= 2^62. The chunks
        // share endpoints and cover [0, N) exactly (RED-005).
        const int64_t begin = (g * N) / G;
        const int64_t end = ((g + 1) * N) / G;
        for (int64_t i = begin; i < end; ++i) {
            const int32_t k = assignments[i];
            if (k < 0 || static_cast<int64_t>(k) >= K) {
                continue;  // CUDA-005, as in accum_atomic_kernel
            }
            owned_sums[static_cast<uint64_t>(k) * d] +=
                static_cast<double>(x_row[static_cast<uint64_t>(i) * d + static_cast<uint64_t>(j)]);
            if (owned_counts != nullptr) {
                owned_counts[k] += 1u;
            }
        }
    }
}

// --- Section 7 Stage 2: fixed-order tree combine -----------------------------------------------
//
// Thread mapping   one 32x8 block per tile tau in [0, K*T), T = ceil(D/32), grid-stride over
//                  tiles. Lane x in [0,32) selects j = (tau mod T)*32 + x; lane y in [0,8)
//                  selects the strided subset g = y, y+8, y+16, ... (RED-003).
// Ownership        each element of S and n is written by exactly one thread -- (x, 0) of the tile
//                  that owns that j, and (0, 0) of the j-tile 0 for the count. P and Q are
//                  read-only. Each shared slot is written only by its own thread.
// Synchronisation  three barriers per tile: one before the shared slots are overwritten (so the
//                  previous tile's tree reads have finished), one after they are filled, and one
//                  after each of the three tree levels. tau is derived from blockIdx only, so the
//                  tile loop and the j >= D guards are block-uniform and no barrier is reached by
//                  only part of the block (CUDA-005, RED-003).
// Numerics         RED-009: ascending g within each y-lane, then the fixed pairwise tree with
//                  strides 4, 2, 1. The order depends only on (G, D, K), never on scheduling.
__global__ void reduce_stage2_kernel(const double *__restrict__ partials,
                                     const unsigned int *__restrict__ partial_counts, int64_t D,
                                     int64_t K, int64_t G, double *__restrict__ sums,
                                     unsigned int *__restrict__ counts)
{
    __shared__ double s_sums[kStage2ChunkLanes][kStage2DimTile];
    __shared__ unsigned int s_counts[kStage2ChunkLanes];

    const int x = static_cast<int>(threadIdx.x);
    const int y = static_cast<int>(threadIdx.y);
    const uint64_t d = static_cast<uint64_t>(D);
    const int64_t tiles_per_cluster = (D + kStage2DimTile - 1) / kStage2DimTile;
    const int64_t tile_count = K * tiles_per_cluster;

    for (int64_t tau = blockIdx.x; tau < tile_count; tau += gridDim.x) {
        const int64_t k = tau / tiles_per_cluster;
        const int64_t dim_tile = tau % tiles_per_cluster;
        const int64_t j = dim_tile * kStage2DimTile + x;

        double partial_sum = 0.0;
        if (j < D) {
            for (int64_t g = y; g < G; g += kStage2ChunkLanes) {
                partial_sum += partials[(static_cast<uint64_t>(g) * static_cast<uint64_t>(K) +
                                         static_cast<uint64_t>(k)) *
                                            d +
                                        static_cast<uint64_t>(j)];
            }
        }
        // Counts are reduced by the tiles with j-tile index 0 only, through lane x == 0 (RED-003).
        unsigned int partial_count = 0;
        if (dim_tile == 0 && x == 0) {
            for (int64_t g = y; g < G; g += kStage2ChunkLanes) {
                partial_count +=
                    partial_counts[static_cast<uint64_t>(g) * static_cast<uint64_t>(K) +
                                   static_cast<uint64_t>(k)];
            }
        }

        __syncthreads();  // the previous tile's tree reads are complete before these writes
        s_sums[y][x] = partial_sum;
        if (x == 0) {
            s_counts[y] = partial_count;
        }
        __syncthreads();

        for (int stride = kStage2ChunkLanes / 2; stride >= 1; stride >>= 1) {
            if (y < stride) {
                s_sums[y][x] += s_sums[y + stride][x];
                if (x == 0) {
                    s_counts[y] += s_counts[y + stride];
                }
            }
            __syncthreads();
        }

        if (y == 0 && j < D) {
            sums[static_cast<uint64_t>(k) * d + static_cast<uint64_t>(j)] = s_sums[0][x];
        }
        if (y == 0 && x == 0 && dim_tile == 0) {
            counts[k] = s_counts[0];
        }
    }
}

// --- Section 2.5: centroid update --------------------------------------------------------------
//
// Thread mapping   one thread per centroid element, 1-D grid-stride over K*D.
// Ownership        element (k, j) of centroids_next is written by exactly one thread; sums,
//                  counts and centroids_cur are read-only. Double buffering (FR-010) is what
//                  makes that true: writing into centroids_cur would race with the reads of the
//                  retention branch and with `movement`.
// Synchronisation  none inside the kernel; the boundary to `movement` is a kernel boundary.
// Numerics         section 2.5: the division is performed in binary64 and the quotient is rounded
//                  once to binary32. DEC-003: when the count is zero no division is performed at
//                  all and the previous centroid is copied bit-exactly. Every (k, j) is written
//                  on every call, so centroids_next needs no initialisation (MEM-005).
__global__ void update_kernel(const double *__restrict__ sums,
                              const unsigned int *__restrict__ counts, int64_t K, int64_t D,
                              const float *__restrict__ centroids_cur,
                              float *__restrict__ centroids_next)
{
    const uint64_t d = static_cast<uint64_t>(D);
    const uint64_t total = static_cast<uint64_t>(K) * d;

    for (uint64_t e = global_thread_index(); e < total; e += grid_stride()) {
        const uint64_t k = e / d;
        const unsigned int count = counts[k];
        if (count == 0u) {
            centroids_next[e] = centroids_cur[e];  // DEC-003
        } else {
            centroids_next[e] = static_cast<float>(sums[e] / static_cast<double>(count));
        }
    }
}

// --- Section 2.6: centroid movement ------------------------------------------------------------
//
// Thread mapping   one thread per cluster, 1-D grid-stride over K; each thread computes the whole
//                  squared displacement of its cluster in ascending j.
// Ownership        block_max[blockIdx.x] is written by thread 0 of that block and by no one else.
//                  The centroid buffers and counts are read-only.
// Synchronisation  the per-block maximum is reduced through shared memory, so all threads reach
//                  all barriers; clusters outside the range contribute 0 but still participate.
// Numerics         differences are formed in binary64 from the binary32 centroids and summed in
//                  ascending j (section 2.6). Empty clusters have delta_k = 0 by construction and
//                  are excluded from the maximum, so no 0/0 can appear. The kernel keeps
//                  delta_k^2; the single square root is taken once in max_reduce_kernel.
__global__ void movement_kernel(const float *__restrict__ centroids_cur,
                                const float *__restrict__ centroids_next,
                                const unsigned int *__restrict__ counts, int64_t K, int64_t D,
                                double *__restrict__ block_max)
{
    extern __shared__ double s_max[];
    const unsigned tid = threadIdx.x;
    const uint64_t d = static_cast<uint64_t>(D);
    double local_max = 0.0;

    for (uint64_t k = global_thread_index(); k < static_cast<uint64_t>(K); k += grid_stride()) {
        if (counts[k] == 0u) {
            continue;  // section 2.6: excluded from the maximum
        }
        const uint64_t base = k * d;
        double squared = 0.0;
        for (int64_t j = 0; j < D; ++j) {
            const double diff = static_cast<double>(centroids_next[base + static_cast<uint64_t>(j)]) -
                                static_cast<double>(centroids_cur[base + static_cast<uint64_t>(j)]);
            squared += diff * diff;
        }
        if (squared > local_max) {
            local_max = squared;
        }
    }

    s_max[tid] = local_max;
    __syncthreads();
    for (unsigned stride = reduction_start_stride(blockDim.x); stride > 0; stride >>= 1) {
        if (tid < stride && tid + stride < blockDim.x) {
            if (s_max[tid + stride] > s_max[tid]) {
                s_max[tid] = s_max[tid + stride];
            }
        }
        __syncthreads();
    }
    if (tid == 0) {
        block_max[blockIdx.x] = s_max[0];
    }
}

// --- Final maximum -----------------------------------------------------------------------------
//
// Thread mapping   a single block (CUDA-002), grid-stride over the per-block maxima.
// Ownership        *delta is written by thread 0 only; block_max is read-only.
// Synchronisation  shared-memory tree with a barrier after each level; one block, so no grid-wide
//                  synchronisation is involved (CUDA-004).
// Numerics         sqrt is applied once, to the maximum of the squared displacements. Because
//                  sqrt is monotone this selects the same cluster as section 2.6 and rounds the
//                  same binary64 sum once, so the value is identical to taking the square root
//                  per cluster and then the maximum.
__global__ void max_reduce_kernel(const double *__restrict__ block_max, int block_count,
                                  double *__restrict__ delta)
{
    extern __shared__ double s_max[];
    const unsigned tid = threadIdx.x;
    double local_max = 0.0;

    for (int b = static_cast<int>(tid); b < block_count; b += static_cast<int>(blockDim.x)) {
        if (block_max[b] > local_max) {
            local_max = block_max[b];
        }
    }

    s_max[tid] = local_max;
    __syncthreads();
    for (unsigned stride = reduction_start_stride(blockDim.x); stride > 0; stride >>= 1) {
        if (tid < stride && tid + stride < blockDim.x) {
            if (s_max[tid + stride] > s_max[tid]) {
                s_max[tid] = s_max[tid + stride];
            }
        }
        __syncthreads();
    }
    if (tid == 0) {
        *delta = sqrt(s_max[0]);
    }
}

}  // namespace

// --- launchers ---------------------------------------------------------------------------------

void launch_assign(const float *x_col, int64_t N, int64_t D, const float *centroids, int64_t K,
                   int32_t *assignments, unsigned long long *changed, bool count_changes,
                   int block, int sm_count, cudaStream_t stream)
{
    const int grid = grid_size(N, block, sm_count);
    const size_t shared = static_cast<size_t>(block_reduction_bytes(block));
    assign_kernel<<<grid, block, shared, stream>>>(x_col, N, D, centroids, K, assignments, changed,
                                                   count_changes ? 1 : 0);
    KMEANS_CHECK_LAUNCH("assign");
}

void launch_accum_atomic(const float *x_col, int64_t N, int64_t D, const int32_t *assignments,
                         int64_t K, double *sums, unsigned int *counts, int block, int sm_count,
                         cudaStream_t stream)
{
    uint64_t elements = 0;
    if (!checked_mul(static_cast<uint64_t>(N), static_cast<uint64_t>(D), &elements)) {
        throw CudaError("[CUDA-006] N*D overflows 64-bit arithmetic", kExitInvalidInput);
    }
    const int grid = grid_size(static_cast<int64_t>(elements), block, sm_count);
    accum_atomic_kernel<<<grid, block, 0, stream>>>(x_col, N, D, assignments, K, sums, counts);
    KMEANS_CHECK_LAUNCH("accum_atomic");
}

void launch_reduce_stage1(const float *x_row, int64_t N, int64_t D, const int32_t *assignments,
                          int64_t K, int64_t G, double *partials, unsigned int *partial_counts,
                          int block, int sm_count, cudaStream_t stream)
{
    uint64_t workers = 0;
    if (!checked_mul(static_cast<uint64_t>(G), static_cast<uint64_t>(D), &workers)) {
        throw CudaError("[CUDA-006] G*D overflows 64-bit arithmetic", kExitInvalidInput);
    }
    const int grid = grid_size(static_cast<int64_t>(workers), block, sm_count);
    reduce_stage1_kernel<<<grid, block, 0, stream>>>(x_row, N, D, assignments, K, G, partials,
                                                     partial_counts);
    KMEANS_CHECK_LAUNCH("reduce_stage1");
}

void launch_reduce_stage2(const double *partials, const unsigned int *partial_counts, int64_t D,
                          int64_t K, int64_t G, double *sums, unsigned int *counts, int sm_count,
                          cudaStream_t stream)
{
    // One block per tile per grid-stride step; the block shape is fixed by RED-003.
    const int grid = grid_size(stage2_tiles(K, D), 1, sm_count);
    const dim3 block(kStage2DimTile, kStage2ChunkLanes);
    reduce_stage2_kernel<<<grid, block, 0, stream>>>(partials, partial_counts, D, K, G, sums,
                                                     counts);
    KMEANS_CHECK_LAUNCH("reduce_stage2");
}

void launch_update(const double *sums, const unsigned int *counts, int64_t K, int64_t D,
                   const float *centroids_cur, float *centroids_next, int block, int sm_count,
                   cudaStream_t stream)
{
    uint64_t elements = 0;
    if (!checked_mul(static_cast<uint64_t>(K), static_cast<uint64_t>(D), &elements)) {
        throw CudaError("[CUDA-006] K*D overflows 64-bit arithmetic", kExitInvalidInput);
    }
    const int grid = grid_size(static_cast<int64_t>(elements), block, sm_count);
    update_kernel<<<grid, block, 0, stream>>>(sums, counts, K, D, centroids_cur, centroids_next);
    KMEANS_CHECK_LAUNCH("update");
}

void launch_movement(const float *centroids_cur, const float *centroids_next,
                     const unsigned int *counts, int64_t K, int64_t D, double *block_max,
                     int block, int sm_count, cudaStream_t stream)
{
    const int grid = movement_grid_size(K, block, sm_count);
    const size_t shared = static_cast<size_t>(block_reduction_bytes(block));
    movement_kernel<<<grid, block, shared, stream>>>(centroids_cur, centroids_next, counts, K, D,
                                                     block_max);
    KMEANS_CHECK_LAUNCH("movement");
}

void launch_max_reduce(const double *block_max, int block_count, double *delta,
                       cudaStream_t stream)
{
    // A single block (CUDA-002). The movement grid is capped at kMaxMovementBlocks = 1024, which
    // is also the maximum block size, so one thread per per-block maximum always fits; the block
    // is rounded up to a warp so the shared-memory tree has no partial warp.
    int threads = block_count < kMaxMovementBlocks ? block_count : kMaxMovementBlocks;
    threads = ((threads + 31) / 32) * 32;
    if (threads < 32) {
        threads = 32;
    }
    const size_t shared = static_cast<size_t>(block_reduction_bytes(threads));
    max_reduce_kernel<<<1, threads, shared, stream>>>(block_max, block_count, delta);
    KMEANS_CHECK_LAUNCH("max_reduce");
}

}  // namespace gpu
}  // namespace kmeans


In [ ]:
%%writefile kmeans_gpu.h
// kmeans_gpu.h — the device driver loop and finalization (ARC-09, section 2.7, section 2.8).
//
// The entry point mirrors cpu::run so the two can be compared field by field (section 10.3,
// ARC-03/ARC-09 interface contract): same inputs, same Result record. Everything that differs
// between Approach A and Approach B is inside the accumulation step, which is what FR-018
// requires -- assignment, update, convergence and finalization are one code path.
//
// Errors are raised as CudaError (cuda_util.cuh) carrying the BLD-005 exit code, so every early
// return unwinds through the RAII buffer owners and releases device memory (MEM-007).

#ifndef KMEANS_KMEANS_GPU_H
#define KMEANS_KMEANS_GPU_H

#include <cstdint>
#include <functional>
#include <string>
#include <vector>

#include "common.h"
#include "device_context.cuh"

namespace kmeans {
namespace gpu {

// FR-019 verification modes.
enum class Verify {
    kNone,
    kFinal,     // P1 and P2 checked after the run (verify_final below)
    kLockstep   // every iteration compared against the CPU reference (section 10.3)
};

struct Options {
    Config config;         // N, D, K, max_iter, tol, strategy, assign kernel, block size, G
    double m_range = 0.0;  // the M of VAL-007, which fixes the NUM-006 scale
    int device = 0;
    uint64_t reduction_cap_bytes = 0;  // MEM-003 user cap on B_red; 0 means no explicit cap
    Verify verify = Verify::kNone;

    // TEST-U06: fill P, Q, S and n with 0xFF bytes (NaN and 0xFFFFFFFF) after allocation. A
    // correct Approach B overwrites every element of P and Q in Stage 1 and every element of S
    // and n in Stage 2, so the poison must not survive into any result.
    bool poison_buffers = false;
};

// What the run reports about itself beyond the clustering Result.
struct RunInfo {
    DeviceInfo device;
    int64_t reduce_chunks = 0;   // the G of RED-006, 0 for Approach A; reported per RED-007
    int64_t g_memory = 0;        // G_mem, for the record
    int64_t g_target = 0;        // max(1, ceil(T_target / D))
    uint64_t reduction_bytes = 0;  // 8GKD + 4GK actually allocated
    int assign_grid = 0;
    int accum_grid = 0;
    int movement_grid = 0;

    // PERF-005 timings in milliseconds.
    double t_alg_ms = 0.0;
    double t_e2e_ms = 0.0;
    double t_alloc_ms = 0.0;
    double t_h2d_ms = 0.0;
    double t_d2h_ms = 0.0;
    // FR-019: lockstep downloads sit inside the measured region, so a lockstep run's timings are
    // not comparable and must never be used as a benchmark sample.
    bool timing_valid = true;

    double max_centroid_difference = 0.0;  // worst |C_final - M| from the P2 check (section 2.8)
};

// One iteration of device state, handed to the lockstep checker (section 10.3 item 1). The
// snapshot is taken while A(t) still exists, before the assignment pass overwrites it in place.
struct LockstepSnapshot {
    int64_t iteration = 0;              // t before this update, so the update produces C(t+1)
    std::vector<int32_t> a_in;          // A(t), the input of the accumulation
    std::vector<float> c_in;            // C(t)
    std::vector<uint32_t> counts;       // device n from the accumulation
    std::vector<double> sums;           // device S from the accumulation
    std::vector<float> c_out;           // C(t+1) from the device update
    std::vector<int32_t> a_out;         // A(t+1) from the device assignment pass
    double delta = 0.0;                 // device Delta
    int64_t changed = 0;                // device change count
};

using LockstepHook = std::function<void(const LockstepSnapshot &)>;

// Section 2.7 on the device. X and C0 are host arrays in row-major binary32 (section 2.1).
// `info` may be null. `hook` is called once per iteration when options.verify == kLockstep.
Result run(const float *X, const float *C0, const Options &options, RunInfo *info,
           const LockstepHook &hook);

inline Result run(const float *X, const float *C0, const Options &options, RunInfo *info)
{
    return run(X, C0, options, info, LockstepHook());
}

// --- FR-019 `final` verification ---------------------------------------------------------------

struct FinalVerification {
    int64_t p1_hard_mismatches = 0;       // non-near-tie disagreements: failures (NUM-005)
    int64_t p1_near_tie_mismatches = 0;   // near-tie disagreements: reported, not failures
    bool p2_flag_correct = false;         // the device's P2 flag equals the reference's
    bool reference_p2 = false;
    double max_centroid_difference = 0.0;
    std::vector<std::string> reports;     // one line per near-tie or hard mismatch
};

// P1: every reported assignment is the nearest reported centroid, judged with the CPU reference's
// binary32 distance path and the near-tie rule of NUM-005. P2: the reference's finalization pass
// agrees with the device's centroid_consistent flag (section 2.8 step 2).
FinalVerification verify_final(const float *X, const Result &result, const Config &config,
                               double m_range);

}  // namespace gpu
}  // namespace kmeans

#endif  // KMEANS_KMEANS_GPU_H


In [ ]:
%%writefile kmeans_gpu.cu
#include "kmeans_gpu.h"

#include <chrono>
#include <cstdio>

#include "cpu_reference.h"
#include "cuda_util.cuh"
#include "kernels.cuh"
#include "launch_config.h"

namespace kmeans {
namespace gpu {
namespace {

using Clock = std::chrono::steady_clock;

double ms_since(const Clock::time_point &start)
{
    return std::chrono::duration<double, std::milli>(Clock::now() - start).count();
}

// DEC-009: the dimension-major copy is produced once on the host from the row-major input, before
// the timed loop. Doing it here rather than in a kernel keeps the device kernel set exactly as
// CUDA-002 enumerates it; DEC-009 permits either. The transpose is blocked because the naive
// version writes with stride N and misses the cache on every store once N*D leaves L2.
void transpose_to_dimension_major(const float *x_row, int64_t N, int64_t D, float *x_col)
{
    constexpr int64_t kTile = 64;
    for (int64_t i0 = 0; i0 < N; i0 += kTile) {
        const int64_t i_end = (i0 + kTile < N) ? i0 + kTile : N;
        for (int64_t j0 = 0; j0 < D; j0 += kTile) {
            const int64_t j_end = (j0 + kTile < D) ? j0 + kTile : D;
            for (int64_t i = i0; i < i_end; ++i) {
                for (int64_t j = j0; j < j_end; ++j) {
                    x_col[static_cast<size_t>(j) * static_cast<size_t>(N) +
                          static_cast<size_t>(i)] =
                        x_row[static_cast<size_t>(i) * static_cast<size_t>(D) +
                              static_cast<size_t>(j)];
                }
            }
        }
    }
}

// Device-side state of one run. Declared as a struct so the RAII members are released in reverse
// declaration order on every exit path, including an exception out of a launcher (MEM-007).
struct DeviceState {
    CudaStream stream;
    DeviceBuffer<float> x_col;
    DeviceBuffer<float> x_row;
    DeviceBuffer<float> centroids_a;
    DeviceBuffer<float> centroids_b;
    DeviceBuffer<int32_t> assignments;
    DeviceBuffer<double> sums;
    DeviceBuffer<unsigned int> counts;
    DeviceBuffer<double> partials;              // Approach B only
    DeviceBuffer<unsigned int> partial_counts;  // Approach B only
    DeviceBuffer<double> block_max;
    DeviceBuffer<ConvergenceScalars> scalars;
    PinnedBuffer<ConvergenceScalars> host_scalars;
};

}  // namespace

Result run(const float *X, const float *C0, const Options &options, RunInfo *info,
           const LockstepHook &hook)
{
    const Config &cfg = options.config;
    const int64_t N = cfg.N;
    const int64_t D = cfg.D;
    const int64_t K = cfg.K;

    if (cfg.assign_kernel == AssignKernel::kSharedMemory) {
        // SMEM-004 requires the baseline to pass first; assign_smem arrives in M4.
        throw CudaError("[SMEM-001] the shared-memory assignment kernel is not implemented yet "
                        "(milestone M4); use --assign baseline",
                        kExitInvalidInput);
    }

    RunInfo local_info;
    RunInfo &out = (info != nullptr) ? *info : local_info;

    // --- ARC-04: device query, capability gate, G selection, capacity check -------------------
    out.device = query_device(options.device);
    check_shared_memory(out.device, cfg.block_size, kAssignTileK, kAssignTileD);
    const ChunkSelection chunks = select_chunks(out.device, cfg, options.reduction_cap_bytes);
    check_capacity(out.device, cfg, chunks.G);

    const bool reduction = cfg.strategy == Strategy::kReduction;
    const int64_t G = reduction ? chunks.G : 0;
    const int sm_count = out.device.sm_count;
    const int block = cfg.block_size;
    const int movement_blocks = movement_grid_size(K, block, sm_count);

    out.reduce_chunks = G;
    out.g_memory = chunks.g_memory;
    out.g_target = chunks.g_target;
    out.reduction_bytes = reduction ? chunks.required_bytes : 0;
    out.assign_grid = grid_size(N, block, sm_count);
    out.accum_grid = reduction ? grid_size(G * D, block, sm_count)
                               : grid_size(N * D, block, sm_count);
    out.movement_grid = movement_blocks;
    out.timing_valid = options.verify != Verify::kLockstep;

    const size_t nd = static_cast<size_t>(N) * static_cast<size_t>(D);
    const size_t kd = static_cast<size_t>(K) * static_cast<size_t>(D);

    // --- allocation (T_alloc, excluded from both T_alg and T_e2e per PERF-005) ----------------
    const Clock::time_point alloc_start = Clock::now();
    DeviceState dev;
    dev.x_col.allocate(nd);
    dev.x_row.allocate(nd);
    dev.centroids_a.allocate(kd);
    dev.centroids_b.allocate(kd);
    dev.assignments.allocate(static_cast<size_t>(N));
    dev.sums.allocate(kd);
    dev.counts.allocate(static_cast<size_t>(K));
    if (reduction) {
        dev.partials.allocate(static_cast<size_t>(G) * kd);
        dev.partial_counts.allocate(static_cast<size_t>(G) * static_cast<size_t>(K));
    }
    dev.block_max.allocate(static_cast<size_t>(movement_blocks));
    dev.scalars.allocate(1);
    dev.host_scalars.allocate(1);
    out.t_alloc_ms = ms_since(alloc_start);

    // MEM-006: nothing is read before it is written. The assignment array is zeroed once because
    // the initial pass writes it without reading it, and a partial final block would otherwise
    // leave untouched elements that the accumulation then reads; the scalars are zeroed so the
    // 16-byte copy never reads an uninitialised field (verified by initcheck, TEST-S03).
    dev.assignments.fill_bytes(0, dev.stream.get());
    dev.scalars.fill_bytes(0, dev.stream.get());

    if (options.poison_buffers) {
        // TEST-U06: 0xFF bytes are a quiet NaN as binary64 and 0xFFFFFFFF as uint32, so any
        // element that Stage 1 or Stage 2 fails to overwrite poisons the result visibly.
        dev.sums.fill_bytes(0xFF, dev.stream.get());
        dev.counts.fill_bytes(0xFF, dev.stream.get());
        if (reduction) {
            dev.partials.fill_bytes(0xFF, dev.stream.get());
            dev.partial_counts.fill_bytes(0xFF, dev.stream.get());
        }
    }

    // --- T_e2e starts at the layout preparation and the input copies (PERF-005, DEC-009) -----
    const Clock::time_point e2e_start = Clock::now();
    {
        std::vector<float> x_col_host(nd);
        transpose_to_dimension_major(X, N, D, x_col_host.data());
        const Clock::time_point h2d_start = Clock::now();
        dev.x_col.copy_from_host(x_col_host.data(), nd, dev.stream.get());
        dev.x_row.copy_from_host(X, nd, dev.stream.get());
        dev.centroids_a.copy_from_host(C0, kd, dev.stream.get());
        dev.stream.synchronize();  // the host transpose buffer dies at the end of this scope
        out.t_h2d_ms = ms_since(h2d_start);
    }

    float *c_cur = dev.centroids_a.get();
    float *c_next = dev.centroids_b.get();
    ConvergenceScalars *scalars = dev.scalars.get();
    ConvergenceScalars *host_scalars = dev.host_scalars.get();

    // One accumulation pass with the strategy selected by FR-018. Only this lambda differs
    // between Approach A and Approach B.
    auto accumulate = [&]() {
        if (reduction) {
            // RED-002/RED-003: Stage 1 overwrites every element of P and Q, Stage 2 every element
            // of S and n, so neither buffer needs zeroing here (MEM-005).
            launch_reduce_stage1(dev.x_row.get(), N, D, dev.assignments.get(), K, G,
                                 dev.partials.get(), dev.partial_counts.get(), block, sm_count,
                                 dev.stream.get());
            launch_reduce_stage2(dev.partials.get(), dev.partial_counts.get(), D, K, G,
                                 dev.sums.get(), dev.counts.get(), sm_count, dev.stream.get());
        } else {
            // ATOM-001: zero the shared sum and count buffers on the same stream first.
            dev.sums.fill_bytes(0, dev.stream.get());
            dev.counts.fill_bytes(0, dev.stream.get());
            launch_accum_atomic(dev.x_col.get(), N, D, dev.assignments.get(), K, dev.sums.get(),
                                dev.counts.get(), block, sm_count, dev.stream.get());
        }
    };

    CudaEvent alg_start;
    CudaEvent alg_end;

    // --- section 2.7 ---------------------------------------------------------------------------
    alg_start.record(dev.stream.get());

    // The initial assignment pass has no previous assignment, so it counts no changes.
    launch_assign(dev.x_col.get(), N, D, c_cur, K, dev.assignments.get(), &scalars->changed, false,
                  block, sm_count, dev.stream.get());

    int64_t t = 0;
    Status status = Status::ITERATION_LIMIT;
    double last_delta = 0.0;
    bool has_last_delta = false;

    while (t < cfg.max_iter) {
        // CUDA-003 launch sequence, in order, on one stream.
        accumulate();
        launch_update(dev.sums.get(), dev.counts.get(), K, D, c_cur, c_next, block, sm_count,
                      dev.stream.get());
        launch_movement(c_cur, c_next, dev.counts.get(), K, D, dev.block_max.get(), block,
                        sm_count, dev.stream.get());
        launch_max_reduce(dev.block_max.get(), movement_blocks, &scalars->delta, dev.stream.get());

        LockstepSnapshot snapshot;
        const bool lockstep = options.verify == Verify::kLockstep && static_cast<bool>(hook);
        if (lockstep) {
            // A(t) must be read before the assignment pass overwrites it in place (FR-007).
            snapshot.iteration = t;
            snapshot.a_in.resize(static_cast<size_t>(N));
            snapshot.c_in.resize(kd);
            snapshot.c_out.resize(kd);
            snapshot.counts.resize(static_cast<size_t>(K));
            snapshot.sums.resize(kd);
            dev.assignments.copy_to_host(snapshot.a_in.data(), static_cast<size_t>(N),
                                         dev.stream.get());
            KMEANS_CUDA_CHECK(cudaMemcpyAsync(snapshot.c_in.data(), c_cur, kd * sizeof(float),
                                              cudaMemcpyDeviceToHost, dev.stream.get()));
            KMEANS_CUDA_CHECK(cudaMemcpyAsync(snapshot.c_out.data(), c_next, kd * sizeof(float),
                                              cudaMemcpyDeviceToHost, dev.stream.get()));
            dev.counts.copy_to_host(snapshot.counts.data(), static_cast<size_t>(K),
                                    dev.stream.get());
            dev.sums.copy_to_host(snapshot.sums.data(), kd, dev.stream.get());
            dev.stream.synchronize();
        }

        // The change counter is zeroed immediately before the pass that fills it (FR-007). Only
        // the 8-byte counter is cleared; Delta was just written by max_reduce and must survive.
        KMEANS_CUDA_CHECK(cudaMemsetAsync(&scalars->changed, 0, sizeof(scalars->changed),
                                          dev.stream.get()));
        // c_next holds C(t+1): the pointer swap below is bookkeeping, so the assignment pass of
        // iteration t reads the buffer the update just wrote.
        launch_assign(dev.x_col.get(), N, D, c_next, K, dev.assignments.get(), &scalars->changed,
                      true, block, sm_count, dev.stream.get());

        // CUDA-011: the only in-loop transfer, 16 bytes into pinned host memory.
        KMEANS_CUDA_CHECK(cudaMemcpyAsync(host_scalars, scalars, sizeof(ConvergenceScalars),
                                          cudaMemcpyDeviceToHost, dev.stream.get()));
        dev.stream.synchronize();

        const double delta = host_scalars->delta;
        const int64_t changed = static_cast<int64_t>(host_scalars->changed);

        ++t;  // DEC-006: the count is the number of completed centroid updates
        last_delta = delta;
        has_last_delta = true;

        // DEC-002 on the host: assignment stability first, then the inclusive movement test.
        Status decided = Status::ERROR;
        if (!cfg.fixed_iters) {
            if (changed == 0) {
                decided = Status::CONVERGED_ASSIGNMENTS;
            } else if (delta <= cfg.tol) {
                decided = Status::CONVERGED_MOVEMENT;
            }
        }

        if (lockstep) {
            snapshot.a_out.resize(static_cast<size_t>(N));
            dev.assignments.copy_to_host(snapshot.a_out.data(), static_cast<size_t>(N),
                                         dev.stream.get());
            dev.stream.synchronize();
            snapshot.delta = delta;
            snapshot.changed = changed;
            hook(snapshot);
        }

        // CUDA-003: the pointer swap closes the iteration. It happens before any break so the
        // reported pair is (A(t), C(t)) with C(t) in c_cur (section 2.8 P1).
        float *swap = c_cur;
        c_cur = c_next;
        c_next = swap;

        if (decided != Status::ERROR) {
            status = decided;
            break;
        }
    }

    // --- section 2.8 finalization -------------------------------------------------------------
    // Step 1: one accumulation pass over A_final with the selected strategy, then the update
    // kernel turns (S, n) into M = fl32(S/n) for non-empty k. C_final itself is never touched:
    // the means land in the scratch centroid buffer.
    accumulate();
    launch_update(dev.sums.get(), dev.counts.get(), K, D, c_cur, c_next, block, sm_count,
                  dev.stream.get());
    alg_end.record(dev.stream.get());
    out.t_alg_ms = static_cast<double>(CudaEvent::elapsed_ms(alg_start, alg_end));

    Result result;
    result.assignments.resize(static_cast<size_t>(N));
    result.centroids.resize(kd);
    result.counts.resize(static_cast<size_t>(K));
    std::vector<float> means(kd);

    const Clock::time_point d2h_start = Clock::now();
    dev.assignments.copy_to_host(result.assignments.data(), static_cast<size_t>(N),
                                 dev.stream.get());
    KMEANS_CUDA_CHECK(cudaMemcpyAsync(result.centroids.data(), c_cur, kd * sizeof(float),
                                      cudaMemcpyDeviceToHost, dev.stream.get()));
    KMEANS_CUDA_CHECK(cudaMemcpyAsync(means.data(), c_next, kd * sizeof(float),
                                      cudaMemcpyDeviceToHost, dev.stream.get()));
    KMEANS_CUDA_CHECK(cudaMemcpyAsync(result.counts.data(), dev.counts.get(),
                                      static_cast<size_t>(K) * sizeof(uint32_t),
                                      cudaMemcpyDeviceToHost, dev.stream.get()));
    dev.stream.synchronize();
    out.t_d2h_ms = ms_since(d2h_start);
    out.t_e2e_ms = ms_since(e2e_start);

    // Step 2: P2 holds when every non-empty cluster's reported centroid agrees with the mean of
    // its assigned points within NUM-006. Empty clusters are not evaluated: DEC-003 retained
    // their centroid, which is not the mean of anything.
    const double s_max = s_max_from_range(options.m_range);
    bool consistent = true;
    double worst = 0.0;
    for (int64_t k = 0; k < K; ++k) {
        if (result.counts[static_cast<size_t>(k)] == 0u) {
            continue;
        }
        for (int64_t j = 0; j < D; ++j) {
            const size_t e = static_cast<size_t>(k * D + j);
            const double diff = std::fabs(static_cast<double>(result.centroids[e]) -
                                          static_cast<double>(means[e]));
            if (diff > worst) {
                worst = diff;
            }
            if (!centroid_agrees(result.centroids[e], means[e], s_max)) {
                consistent = false;
            }
        }
    }

    result.iterations = t;
    result.status = status;
    result.last_delta = last_delta;
    result.has_last_delta = has_last_delta;
    result.centroid_consistent = consistent;
    result.assignment_consistent = true;  // P1 by construction (section 2.8)
    // FR-016: J in binary64 on the host, after the download and outside every timed region.
    result.objective = cpu::objective(X, N, D, result.assignments.data(), result.centroids.data());
    out.max_centroid_difference = worst;
    return result;
}

FinalVerification verify_final(const float *X, const Result &result, const Config &config,
                               double m_range)
{
    FinalVerification v;
    const int64_t N = config.N;
    const int64_t D = config.D;
    const int64_t K = config.K;

    // P1: the reported assignment of every point must be the nearest reported centroid under the
    // reference's binary32 distance path. A disagreement is a failure unless the point is a
    // near-tie, where FMA contraction alone can reorder two candidates (NUM-005).
    for (int64_t i = 0; i < N; ++i) {
        const float *point = X + i * D;
        const int32_t expected = cpu::nearest_centroid(point, D, result.centroids.data(), K);
        const int32_t got = result.assignments[static_cast<size_t>(i)];
        if (expected == got) {
            continue;
        }
        const cpu::TieInfo tie = cpu::tie_info(point, D, result.centroids.data(), K);
        char line[256];
        std::snprintf(line, sizeof(line),
                      "point %lld: device k=%d, reference k=%d, margin %.3e, threshold %.3e%s",
                      static_cast<long long>(i), got, expected, tie.margin,
                      near_tie_threshold(D), tie.near_tie ? " (near-tie)" : " (HARD)");
        v.reports.push_back(line);
        if (tie.near_tie) {
            ++v.p1_near_tie_mismatches;
        } else {
            ++v.p1_hard_mismatches;
        }
    }

    // P2: the reference's own finalization pass must reach the same verdict as the device's flag.
    const cpu::Finalization fin =
        cpu::finalize(X, N, D, result.assignments.data(), result.centroids.data(), K, m_range);
    v.reference_p2 = fin.centroid_consistent;
    v.p2_flag_correct = (fin.centroid_consistent == result.centroid_consistent);
    v.max_centroid_difference = fin.max_centroid_difference;
    return v;
}

}  // namespace gpu
}  // namespace kmeans


In [ ]:
%%writefile test_gpu.cu
// test_gpu.cu — GPU test driver for milestones M2 (baseline assign, Approach A, update,
// convergence, finalization) and M3 (Approach B).
//
// Covers, for both accumulation strategies:
//   TEST-I01 / TEST-I02   H1-H6 and H2-a..e against the expected results of section 16.0
//   TEST-I03 / I04 / I05  iteration counting, the movement boundary and the iteration limit,
//                         which are exactly what the H2 variants encode
//   TEST-I06              the P2 flag of section 2.8 against the CPU reference
//   TEST-I07              P1 and the P2 flag on blob data (gpu::verify_final, FR-019 `final`)
//   TEST-I10 / section 10.3  lockstep: every iteration of a 100k x 16, K=8 blob run compared
//                         against the CPU reference, kernel by kernel
//   TEST-U03              the change count, exactly, from the lockstep snapshots
//   TEST-U04 / U05        accumulation (atomic; reduction over G in {1, 2, 7, N} and auto)
//   TEST-U06              poisoned P, Q, S, n must not survive into any result
//   TEST-U07              partial Stage 2 tiles: K = 1024, D in {1, 33, 65}, small chunks
//   TEST-U08 / U10        movement and update, from the lockstep snapshots
//   TEST-E16 / EDGE-16    N, D, K not multiples of the block, tile or warp size
//
// One executable, non-zero exit on any failure, so a single notebook cell can gate the run
// (NFR-015). `--quick` runs only the subset the Batch A notebook cells need; the full run adds
// the G sweep, the partial-tile cases and the non-aligned size sweep.
//
// No result here is evidence of anything until an executed notebook in results/ shows it
// (CLAUDE.md): this file only defines what must hold.

#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <string>
#include <vector>

#include "common.h"
#include "cpu_reference.h"
#include "cuda_util.cuh"
#include "gen.h"
#include "hand_data.h"
#include "kmeans_gpu.h"
#include "launch_config.h"
#include "validate.h"

using namespace kmeans;

namespace {

int g_checks = 0;
int g_failures = 0;
int g_case_failures = 0;

void check(bool condition, const std::string &what)
{
    ++g_checks;
    if (!condition) {
        ++g_failures;
        ++g_case_failures;
        std::printf("      FAIL  %s\n", what.c_str());
    }
}

void check_eq_i64(int64_t got, int64_t want, const std::string &what)
{
    check(got == want, what + ": got " + std::to_string(got) + ", want " + std::to_string(want));
}

std::string float_str(float v)
{
    char buf[48];
    std::snprintf(buf, sizeof(buf), "%.9g", static_cast<double>(v));
    return std::string(buf);
}

std::string double_str(double v)
{
    char buf[48];
    std::snprintf(buf, sizeof(buf), "%.17g", v);
    return std::string(buf);
}

// Scope guard for the per-test PASS/FAIL line, as in tests/test_cpu.cpp.
class TestScope {
public:
    TestScope(const char *id, const std::string &title)
    {
        g_case_failures = 0;
        std::printf("  %-9s %s\n", id, title.c_str());
    }
    ~TestScope() { std::printf("      %s\n", g_case_failures == 0 ? "PASS" : "-> FAILED"); }
};

const char *strategy_name(Strategy s)
{
    return s == Strategy::kAtomic ? "atomic" : "reduction";
}

// Every GPU call path can raise CudaError (cuda_util.cuh). A test must record that as a failure
// and carry on, not abort the executable, so the remaining tests still report.
bool run_device(const float *X, const float *C0, const gpu::Options &options, Result *result,
                gpu::RunInfo *info, const gpu::LockstepHook &hook, const std::string &what)
{
    try {
        *result = gpu::run(X, C0, options, info, hook);
        return true;
    } catch (const CudaError &err) {
        check(false, what + ": " + err.what() + " (exit code " +
                         std::to_string(err.exit_code()) + ")");
        return false;
    } catch (const std::exception &err) {
        check(false, what + ": " + err.what());
        return false;
    }
}

// --- TEST-I01 / TEST-I02: the hand-verifiable datasets on the device ---------------------------

void test_hand_cases(Strategy strategy)
{
    for (const tests::HandCase &c : tests::hand_cases()) {
        TestScope scope(strategy == Strategy::kAtomic ? "I01" : "I02",
                        std::string(strategy_name(strategy)) + "  " + c.id +
                            (c.note.empty() ? "" : "  (" + c.note + ")"));

        double m_range = 0.0;
        const Validation data = validate_data(c.X, c.N, c.D, c.C0, c.K, &m_range);
        check(data.ok, c.id + ": data accepted");

        gpu::Options options;
        options.config.N = c.N;
        options.config.D = c.D;
        options.config.K = c.K;
        options.config.max_iter = c.max_iter;
        options.config.tol = c.tol;
        options.config.strategy = strategy;
        options.m_range = m_range;
        options.verify = gpu::Verify::kFinal;

        Result r;
        gpu::RunInfo info;
        if (!run_device(c.X.data(), c.C0.data(), options, &r, &info, gpu::LockstepHook(),
                        c.id + ": device run")) {
            continue;
        }

        check_eq_i64(r.iterations, c.expected_iterations, c.id + ": iteration count (DEC-006)");
        check(r.status == c.expected_status,
              c.id + ": status: got " + status_name(r.status) + ", want " +
                  status_name(c.expected_status));
        check(r.centroid_consistent == c.expected_centroid_consistent,
              c.id + ": P2 flag: got " + (r.centroid_consistent ? "true" : "false") + ", want " +
                  (c.expected_centroid_consistent ? "true" : "false"));
        check(r.assignment_consistent, c.id + ": P1 flag true by construction");

        for (int64_t i = 0; i < c.N; ++i) {
            check_eq_i64(r.assignments[static_cast<size_t>(i)],
                         c.expected_A[static_cast<size_t>(i)],
                         c.id + ": A[" + std::to_string(i) + "]");
        }
        for (int64_t k = 0; k < c.K; ++k) {
            check_eq_i64(r.counts[static_cast<size_t>(k)], c.expected_n[static_cast<size_t>(k)],
                         c.id + ": n[" + std::to_string(k) + "]");
        }

        const double s_max = s_max_from_range(m_range);
        for (int64_t e = 0; e < c.K * c.D; ++e) {
            const float got = r.centroids[static_cast<size_t>(e)];
            const float want = c.expected_C[static_cast<size_t>(e)];
            const bool ok = c.centroids_exact ? (got == want) : centroid_agrees(got, want, s_max);
            check(ok, c.id + ": C[" + std::to_string(e) + "]: got " + float_str(got) + ", want " +
                          float_str(want) + (c.centroids_exact ? " (exact)" : " (NUM-006)"));
        }

        // DEC-003: an empty cluster keeps its initial centroid bit-exactly. True for every H case
        // because none of them ever re-populates an empty cluster.
        for (int64_t k = 0; k < c.K; ++k) {
            if (c.expected_n[static_cast<size_t>(k)] != 0) {
                continue;
            }
            for (int64_t j = 0; j < c.D; ++j) {
                const size_t e = static_cast<size_t>(k * c.D + j);
                check(r.centroids[e] == c.C0[e],
                      c.id + ": empty cluster " + std::to_string(k) + " retained C0[" +
                          std::to_string(j) + "] bit-exactly");
            }
        }

        // TEST-I06 and the CPU oracle side by side: the device must reach the same record the
        // reference does on these cases, which section 16.0 fixes by hand.
        const Result ref = cpu::run(c.X.data(), c.N, c.D, c.C0.data(), c.K, c.max_iter, c.tol,
                                    m_range, false);
        check(ref.status == r.status, c.id + ": status matches the CPU reference");
        check_eq_i64(r.iterations, ref.iterations, c.id + ": iterations match the CPU reference");
        check(r.centroid_consistent == ref.centroid_consistent,
              c.id + ": P2 flag matches the CPU reference");

        const gpu::FinalVerification v = gpu::verify_final(c.X.data(), r, options.config, m_range);
        check_eq_i64(v.p1_hard_mismatches, 0, c.id + ": P1 hard mismatches (NUM-005)");
        check(v.p2_flag_correct, c.id + ": P2 flag agrees with the reference finalization");
    }
}

// --- section 10.3 lockstep -----------------------------------------------------------------------

struct LockstepStats {
    int64_t iterations = 0;
    int64_t assign_hard_mismatches = 0;
    int64_t assign_near_tie_mismatches = 0;
    int64_t count_mismatches = 0;
    int64_t sum_violations = 0;
    int64_t centroid_violations = 0;
    int64_t change_count_mismatches = 0;
    int64_t delta_violations = 0;
    double max_sum_diff = 0.0;
    double max_centroid_diff = 0.0;
    double max_delta_diff = 0.0;
    std::vector<std::string> notes;
};

// One iteration of section 10.3 item 1. Every kernel of the iteration is judged separately, so a
// failure names the kernel rather than only the end result:
//   accumulation  device n exactly, device S within 1e-12 * n_k * S_max (TEST-U04/U05 criterion)
//   update        device C(t+1) within NUM-006 of the reference update of the same (S, n)
//   movement      device Delta against the reference movement of the device's own centroids
//   assign        device A(t+1) against the reference assignment of the device's own C(t+1),
//                 with disagreements allowed only on near-ties (NUM-005)
//   change count  exactly, against the reference count of differences between A(t) and A(t+1)
void check_lockstep_iteration(const gpu::LockstepSnapshot &snap, const float *X,
                              const Config &config, double m_range, LockstepStats *stats)
{
    const int64_t N = config.N;
    const int64_t D = config.D;
    const int64_t K = config.K;
    const double s_max = s_max_from_range(m_range);
    ++stats->iterations;

    // The very first snapshot also pins down the initial assignment pass of section 2.7, which no
    // later snapshot covers: for t > 0, A(t) was the previous iteration's A(t+1).
    if (snap.iteration == 0) {
        std::vector<int32_t> a_ref(static_cast<size_t>(N), 0);
        cpu::assign(X, N, D, snap.c_in.data(), K, a_ref.data(), nullptr);
        for (int64_t i = 0; i < N; ++i) {
            if (a_ref[static_cast<size_t>(i)] == snap.a_in[static_cast<size_t>(i)]) {
                continue;
            }
            const cpu::TieInfo tie = cpu::tie_info(X + i * D, D, snap.c_in.data(), K);
            if (tie.near_tie) {
                ++stats->assign_near_tie_mismatches;
            } else {
                ++stats->assign_hard_mismatches;
                if (stats->notes.size() < 10) {
                    stats->notes.push_back("initial assign: point " + std::to_string(i) +
                                           " device " +
                                           std::to_string(snap.a_in[static_cast<size_t>(i)]) +
                                           ", reference " +
                                           std::to_string(a_ref[static_cast<size_t>(i)]) +
                                           ", margin " + double_str(tie.margin));
                }
            }
        }
    }

    // Accumulation: the reference is given the device's own A(t).
    std::vector<double> sums_ref(static_cast<size_t>(K * D), 0.0);
    std::vector<uint32_t> counts_ref(static_cast<size_t>(K), 0);
    cpu::accumulate(X, N, D, snap.a_in.data(), K, sums_ref.data(), counts_ref.data());

    uint64_t total_counted = 0;
    for (int64_t k = 0; k < K; ++k) {
        const uint32_t got = snap.counts[static_cast<size_t>(k)];
        total_counted += got;
        if (got != counts_ref[static_cast<size_t>(k)]) {
            ++stats->count_mismatches;
            if (stats->notes.size() < 10) {
                stats->notes.push_back("n[" + std::to_string(k) + "] device " +
                                       std::to_string(got) + ", reference " +
                                       std::to_string(counts_ref[static_cast<size_t>(k)]));
            }
        }
    }
    // RED-005 / FR-009: every point contributes to exactly one cluster, so the counts sum to N.
    if (total_counted != static_cast<uint64_t>(N)) {
        ++stats->count_mismatches;
        if (stats->notes.size() < 10) {
            stats->notes.push_back("counts sum to " + std::to_string(total_counted) +
                                   ", want N = " + std::to_string(N));
        }
    }

    for (int64_t k = 0; k < K; ++k) {
        // |S_kj| <= n_k * M, and binary64 accumulation of n_k terms loses far less than 1e-12 of
        // that, so the bound is proportional to the largest value the sum could take.
        const double scale = static_cast<double>(counts_ref[static_cast<size_t>(k)]) * s_max;
        const double tol = 1e-12 * (scale > s_max ? scale : s_max);
        for (int64_t j = 0; j < D; ++j) {
            const size_t e = static_cast<size_t>(k * D + j);
            const double diff = std::fabs(snap.sums[e] - sums_ref[e]);
            if (diff > stats->max_sum_diff) {
                stats->max_sum_diff = diff;
            }
            if (!(diff <= tol)) {  // negated so a NaN counts as a violation
                ++stats->sum_violations;
                if (stats->notes.size() < 10) {
                    stats->notes.push_back("S[" + std::to_string(k) + "][" + std::to_string(j) +
                                           "] device " + double_str(snap.sums[e]) +
                                           ", reference " + double_str(sums_ref[e]) +
                                           ", tolerance " + double_str(tol));
                }
            }
        }
    }

    // Update: the reference update of the same (S, n) must reproduce the device's C(t+1). The
    // reference is fed its own sums, so this checks the division and the DEC-003 retention, and
    // tolerates only the one-ulp difference NUM-006 allows.
    std::vector<float> c_ref(static_cast<size_t>(K * D), 0.0f);
    cpu::update(sums_ref.data(), counts_ref.data(), K, D, snap.c_in.data(), c_ref.data());
    for (int64_t k = 0; k < K; ++k) {
        for (int64_t j = 0; j < D; ++j) {
            const size_t e = static_cast<size_t>(k * D + j);
            const double diff = std::fabs(static_cast<double>(snap.c_out[e]) -
                                          static_cast<double>(c_ref[e]));
            if (diff > stats->max_centroid_diff) {
                stats->max_centroid_diff = diff;
            }
            if (!centroid_agrees(snap.c_out[e], c_ref[e], s_max)) {
                ++stats->centroid_violations;
                if (stats->notes.size() < 10) {
                    stats->notes.push_back("C[" + std::to_string(k) + "][" + std::to_string(j) +
                                           "] device " + float_str(snap.c_out[e]) +
                                           ", reference " + float_str(c_ref[e]));
                }
            }
            if (counts_ref[static_cast<size_t>(k)] == 0u && snap.c_out[e] != snap.c_in[e]) {
                ++stats->centroid_violations;  // DEC-003: retained bit-exactly
                if (stats->notes.size() < 10) {
                    stats->notes.push_back("empty cluster " + std::to_string(k) +
                                           " was not retained bit-exactly");
                }
            }
        }
    }

    // Movement: judged against the device's own centroids and counts, so this isolates the
    // movement and max_reduce kernels from any difference in the centroids themselves.
    const double delta_ref = cpu::movement(snap.c_in.data(), snap.c_out.data(),
                                           snap.counts.data(), K, D);
    const double delta_diff = std::fabs(delta_ref - snap.delta);
    if (delta_diff > stats->max_delta_diff) {
        stats->max_delta_diff = delta_diff;
    }
    const double delta_tol = 1e-12 * (delta_ref > 1.0 ? delta_ref : 1.0);
    if (!(delta_diff <= delta_tol)) {
        ++stats->delta_violations;
        if (stats->notes.size() < 10) {
            stats->notes.push_back("Delta device " + double_str(snap.delta) + ", reference " +
                                   double_str(delta_ref));
        }
    }

    // Assignment: the reference is given the device's own C(t+1).
    const int64_t near_ties_before = stats->assign_near_tie_mismatches;
    std::vector<int32_t> a_ref(static_cast<size_t>(N), 0);
    const int64_t changed_ref =
        cpu::assign(X, N, D, snap.c_out.data(), K, a_ref.data(), snap.a_in.data());
    for (int64_t i = 0; i < N; ++i) {
        if (a_ref[static_cast<size_t>(i)] == snap.a_out[static_cast<size_t>(i)]) {
            continue;
        }
        const cpu::TieInfo tie = cpu::tie_info(X + i * D, D, snap.c_out.data(), K);
        if (tie.near_tie) {
            ++stats->assign_near_tie_mismatches;
        } else {
            ++stats->assign_hard_mismatches;
            if (stats->notes.size() < 10) {
                stats->notes.push_back("assign: point " + std::to_string(i) + " device " +
                                       std::to_string(snap.a_out[static_cast<size_t>(i)]) +
                                       ", reference " +
                                       std::to_string(a_ref[static_cast<size_t>(i)]) +
                                       ", margin " + double_str(tie.margin) + " > threshold " +
                                       double_str(near_tie_threshold(D)));
            }
        }
    }

    // TEST-U03: the change count is an integer reduction and must be exact. The reference counts
    // differences between the device's A(t) and its own A(t+1); where a near-tie made the two
    // assignments differ in this iteration the counts may legitimately differ, so that case is
    // reported, not failed. The comparison is per iteration: a near-tie seen earlier in the run
    // must not excuse a wrong count here.
    if (changed_ref != snap.changed) {
        if (stats->assign_near_tie_mismatches == near_ties_before) {
            ++stats->change_count_mismatches;
            if (stats->notes.size() < 10) {
                stats->notes.push_back("change count device " + std::to_string(snap.changed) +
                                       ", reference " + std::to_string(changed_ref));
            }
        } else if (stats->notes.size() < 10) {
            stats->notes.push_back("change count differs by " +
                                   std::to_string(changed_ref - snap.changed) +
                                   " with near-ties present (NUM-005), reported not failed");
        }
    }
}

void report_lockstep(const LockstepStats &stats, const std::string &prefix)
{
    check(stats.iterations > 0, prefix + ": at least one iteration was checked");
    check_eq_i64(stats.assign_hard_mismatches, 0, prefix + ": assign hard mismatches (NUM-005)");
    check_eq_i64(stats.count_mismatches, 0, prefix + ": count mismatches (exact)");
    check_eq_i64(stats.sum_violations, 0, prefix + ": sum violations");
    check_eq_i64(stats.centroid_violations, 0, prefix + ": centroid violations (NUM-006)");
    check_eq_i64(stats.delta_violations, 0, prefix + ": Delta violations");
    check_eq_i64(stats.change_count_mismatches, 0, prefix + ": change count mismatches");
    std::printf("      %lld iterations checked; near-tie assignment differences %lld; "
                "max |dS| %.3e, max |dC| %.3e, max |dDelta| %.3e\n",
                static_cast<long long>(stats.iterations),
                static_cast<long long>(stats.assign_near_tie_mismatches), stats.max_sum_diff,
                stats.max_centroid_diff, stats.max_delta_diff);
    for (const std::string &note : stats.notes) {
        std::printf("      note: %s\n", note.c_str());
    }
}

// The 100k x 16, K = 8 blob dataset the Batch A notebook cell runs. The parameters live here
// alone, so the dataset and the sizes the test reports can never drift apart.
GenParams lockstep_params()
{
    GenParams p;
    p.N = 100000;
    p.D = 16;
    p.K = 8;
    p.sep = 10.0;
    p.spread = 1.0;
    p.seed = 20261018;
    return p;
}

void test_lockstep(Strategy strategy, const Dataset &data, const GenParams &params)
{
    TestScope scope("I10", std::string(strategy_name(strategy)) +
                               "  lockstep vs. CPU reference on " +
                               std::to_string(params.N) + " x " + std::to_string(params.D) +
                               ", K=" + std::to_string(params.K) + " blobs (section 10.3)");

    double m_range = 0.0;
    const Validation v = validate_data(data.X, params.N, params.D, data.C0, params.K, &m_range);
    check(v.ok, "generated data is valid");

    gpu::Options options;
    options.config.N = params.N;
    options.config.D = params.D;
    options.config.K = params.K;
    options.config.max_iter = 20;
    // tol = 0 makes the movement criterion unreachable in practice, so the run converges by
    // assignment stability and every iteration up to that point is lockstep-checked.
    options.config.tol = 0.0;
    options.config.strategy = strategy;
    options.m_range = m_range;
    options.verify = gpu::Verify::kLockstep;

    LockstepStats stats;
    const Config config = options.config;
    const gpu::LockstepHook hook = [&](const gpu::LockstepSnapshot &snap) {
        check_lockstep_iteration(snap, data.X.data(), config, m_range, &stats);
    };

    Result r;
    gpu::RunInfo info;
    if (!run_device(data.X.data(), data.C0.data(), options, &r, &info, hook, "lockstep run")) {
        return;
    }

    report_lockstep(stats, std::string(strategy_name(strategy)));
    check(!info.timing_valid,
          "a lockstep run is marked as not timing-valid (FR-019: never a benchmark sample)");
    if (strategy == Strategy::kReduction) {
        check(info.reduce_chunks >= 1 && info.reduce_chunks <= params.N,
              "the selected G is in [1, N]: " + std::to_string(info.reduce_chunks));
        std::printf("      G = %lld (G_target %lld, G_mem %lld, %llu B of partials)\n",
                    static_cast<long long>(info.reduce_chunks),
                    static_cast<long long>(info.g_target),
                    static_cast<long long>(info.g_memory),
                    static_cast<unsigned long long>(info.reduction_bytes));
    }

    // TEST-I07: the same run judged by FR-019 `final`.
    const gpu::FinalVerification fv = gpu::verify_final(data.X.data(), r, config, m_range);
    check_eq_i64(fv.p1_hard_mismatches, 0, "P1 hard mismatches after the run");
    check(fv.p2_flag_correct, "the P2 flag agrees with the reference finalization");
    check(is_converged(r.status) || r.status == Status::ITERATION_LIMIT,
          std::string("status is a non-error status: ") + status_name(r.status));
    check(std::isfinite(r.objective) && r.objective > 0.0,
          "the objective J is finite and positive: " + double_str(r.objective));
    std::printf("      status %s, t = %lld, J = %.9e, T_alg %.3f ms (not a benchmark sample)\n",
                status_name(r.status), static_cast<long long>(r.iterations), r.objective,
                info.t_alg_ms);
}

// --- TEST-U06: poisoned buffers must not survive ------------------------------------------------

void test_poisoning(Strategy strategy)
{
    TestScope scope("U06", std::string(strategy_name(strategy)) +
                               "  poisoned P, Q, S and n do not reach any result (MEM-006)");

    GenParams p;
    p.N = 10000;
    p.D = 8;
    p.K = 5;
    p.sep = 10.0;
    p.spread = 1.0;
    p.seed = 606;
    const Dataset data = generate(p);

    double m_range = 0.0;
    const Validation v = validate_data(data.X, p.N, p.D, data.C0, p.K, &m_range);
    check(v.ok, "generated data is valid");

    gpu::Options clean;
    clean.config.N = p.N;
    clean.config.D = p.D;
    clean.config.K = p.K;
    // Three accumulations at least: two in the loop plus the finalization pass, which is what
    // "call twice" in TEST-U06 asks for.
    clean.config.max_iter = 2;
    clean.config.tol = 0.0;
    clean.config.fixed_iters = true;
    clean.config.strategy = strategy;
    clean.m_range = m_range;

    gpu::Options poisoned = clean;
    poisoned.poison_buffers = true;

    Result r_clean, r_poisoned;
    gpu::RunInfo info_clean, info_poisoned;
    if (!run_device(data.X.data(), data.C0.data(), clean, &r_clean, &info_clean,
                    gpu::LockstepHook(), "unpoisoned run")) {
        return;
    }
    if (!run_device(data.X.data(), data.C0.data(), poisoned, &r_poisoned, &info_poisoned,
                    gpu::LockstepHook(), "poisoned run")) {
        return;
    }

    check_eq_i64(r_poisoned.iterations, r_clean.iterations, "iteration count unchanged");
    check(r_poisoned.status == r_clean.status, "status unchanged");

    int64_t assignment_diffs = 0, count_diffs = 0, non_finite = 0, centroid_diffs = 0;
    double worst = 0.0;
    for (int64_t i = 0; i < p.N; ++i) {
        if (r_poisoned.assignments[static_cast<size_t>(i)] !=
            r_clean.assignments[static_cast<size_t>(i)]) {
            ++assignment_diffs;
        }
    }
    for (int64_t k = 0; k < p.K; ++k) {
        if (r_poisoned.counts[static_cast<size_t>(k)] != r_clean.counts[static_cast<size_t>(k)]) {
            ++count_diffs;
        }
    }
    const double s_max = s_max_from_range(m_range);
    for (int64_t e = 0; e < p.K * p.D; ++e) {
        const float got = r_poisoned.centroids[static_cast<size_t>(e)];
        const float want = r_clean.centroids[static_cast<size_t>(e)];
        if (!std::isfinite(got)) {
            ++non_finite;
            continue;
        }
        const double diff = std::fabs(static_cast<double>(got) - static_cast<double>(want));
        if (diff > worst) {
            worst = diff;
        }
        if (strategy == Strategy::kReduction) {
            // RED-009: a fixed summation order, so two runs of the same configuration are
            // expected to agree bit for bit; a surviving poison would show up here first.
            if (got != want) {
                ++centroid_diffs;
            }
        } else if (!centroid_agrees(got, want, s_max)) {
            // ATOM-007: Approach A fixes no summation order, so the comparison is NUM-006.
            ++centroid_diffs;
        }
    }

    check_eq_i64(non_finite, 0, "every centroid is finite (no NaN survived)");
    check_eq_i64(assignment_diffs, 0, "assignments identical to the unpoisoned run");
    check_eq_i64(count_diffs, 0, "counts identical to the unpoisoned run");
    check_eq_i64(centroid_diffs, 0,
                 std::string("centroids ") +
                     (strategy == Strategy::kReduction ? "bitwise identical (RED-009)"
                                                       : "within NUM-006 (ATOM-007)"));
    check(std::isfinite(r_poisoned.objective), "the objective J is finite");
    std::printf("      max |dC| between the poisoned and clean runs: %.3e\n", worst);
}

// --- TEST-U05: the G sweep ----------------------------------------------------------------------

// Accumulation is compared through the lockstep snapshot of a single iteration, which is the only
// place the device's S and n are visible to a test (ARC-07 outputs).
void test_chunk_sweep()
{
    GenParams p;
    p.N = 20000;
    p.D = 6;
    p.K = 7;
    p.sep = 10.0;
    p.spread = 1.0;
    p.seed = 505;
    const Dataset data = generate(p);

    double m_range = 0.0;
    const Validation v = validate_data(data.X, p.N, p.D, data.C0, p.K, &m_range);

    const int64_t chunk_counts[] = {1, 2, 7, p.N, 0};  // 0 selects automatically (RED-006)
    for (int64_t user_g : chunk_counts) {
        TestScope scope("U05", "reduction accumulation with G = " +
                                   (user_g == 0 ? std::string("auto") : std::to_string(user_g)));
        check(v.ok, "generated data is valid");

        gpu::Options options;
        options.config.N = p.N;
        options.config.D = p.D;
        options.config.K = p.K;
        options.config.max_iter = 1;
        options.config.tol = 0.0;
        options.config.fixed_iters = true;
        options.config.strategy = Strategy::kReduction;
        options.config.reduce_chunks = user_g;
        options.m_range = m_range;
        options.verify = gpu::Verify::kLockstep;

        LockstepStats stats;
        const Config config = options.config;
        const gpu::LockstepHook hook = [&](const gpu::LockstepSnapshot &snap) {
            check_lockstep_iteration(snap, data.X.data(), config, m_range, &stats);
        };

        Result r;
        gpu::RunInfo info;
        if (!run_device(data.X.data(), data.C0.data(), options, &r, &info, hook, "run")) {
            continue;
        }
        if (user_g != 0) {
            check_eq_i64(info.reduce_chunks, user_g, "the override was honoured");
        }
        report_lockstep(stats, "G=" + std::to_string(info.reduce_chunks));
    }
}

// --- TEST-U07: partial Stage 2 tiles -------------------------------------------------------------

void test_partial_tiles()
{
    // K = 1024 with only a few thousand points means most (chunk, cluster) pairs are empty, which
    // is the EDGE-35 case; D in {1, 33, 65} makes the last 32-dimension tile of Stage 2 partial.
    const int64_t dims[] = {1, 33, 65};
    for (int64_t D : dims) {
        TestScope scope("U07", "reduction with K = 1024, D = " + std::to_string(D) +
                                   ", G = 7: partial tiles and empty chunks (EDGE-35)");

        GenParams p;
        p.N = 5000;
        p.D = D;
        p.K = 1024;
        p.sep = 10.0;
        p.spread = 1.0;
        p.seed = 707;
        const Dataset data = generate(p);

        double m_range = 0.0;
        const Validation v = validate_data(data.X, p.N, p.D, data.C0, p.K, &m_range);
        check(v.ok, "generated data is valid");

        gpu::Options options;
        options.config.N = p.N;
        options.config.D = p.D;
        options.config.K = p.K;
        options.config.max_iter = 1;
        options.config.tol = 0.0;
        options.config.fixed_iters = true;
        options.config.strategy = Strategy::kReduction;
        options.config.reduce_chunks = 7;
        options.m_range = m_range;
        options.verify = gpu::Verify::kLockstep;

        LockstepStats stats;
        const Config config = options.config;
        std::vector<double> column_sums(static_cast<size_t>(D), 0.0);
        std::vector<double> device_column_sums(static_cast<size_t>(D), 0.0);
        const gpu::LockstepHook hook = [&](const gpu::LockstepSnapshot &snap) {
            check_lockstep_iteration(snap, data.X.data(), config, m_range, &stats);
            // RED-005: summing S over all clusters must reproduce the column sums of X, because
            // every (i, j) contributed exactly once.
            for (int64_t k = 0; k < config.K; ++k) {
                for (int64_t j = 0; j < config.D; ++j) {
                    device_column_sums[static_cast<size_t>(j)] +=
                        snap.sums[static_cast<size_t>(k * config.D + j)];
                }
            }
        };

        for (int64_t i = 0; i < p.N; ++i) {
            for (int64_t j = 0; j < D; ++j) {
                column_sums[static_cast<size_t>(j)] +=
                    static_cast<double>(data.X[static_cast<size_t>(i * D + j)]);
            }
        }

        Result r;
        gpu::RunInfo info;
        if (!run_device(data.X.data(), data.C0.data(), options, &r, &info, hook, "run")) {
            continue;
        }
        report_lockstep(stats, "K=1024, D=" + std::to_string(D));

        int64_t column_violations = 0;
        double worst = 0.0;
        const double s_max = s_max_from_range(m_range);
        for (int64_t j = 0; j < D; ++j) {
            const double want = column_sums[static_cast<size_t>(j)];
            const double got = device_column_sums[static_cast<size_t>(j)];
            const double diff = std::fabs(got - want);
            if (diff > worst) {
                worst = diff;
            }
            const double scale = static_cast<double>(p.N) * s_max;
            if (!(diff <= 1e-10 * (scale > s_max ? scale : s_max))) {
                ++column_violations;
            }
        }
        check_eq_i64(column_violations, 0, "column sums of X preserved by the reduction");
        std::printf("      max column-sum difference %.3e over %lld dimensions\n", worst,
                    static_cast<long long>(D));
    }
}

// --- TEST-E16 / EDGE-16: non-aligned sizes -------------------------------------------------------

void test_non_aligned(Strategy strategy)
{
    const int64_t n_values[] = {1, 31, 33, 255, 257, 1023};
    const int64_t d_values[] = {1, 3, 31, 33, 65};
    const int64_t k_values[] = {1, 3, 7, 9, 17};

    TestScope scope("E16", std::string(strategy_name(strategy)) +
                               "  N, D, K not multiples of the block, tile or warp size");

    int64_t configurations = 0, mismatches = 0;
    for (int64_t N : n_values) {
        for (int64_t D : d_values) {
            for (int64_t K : k_values) {
                GenParams p;
                p.N = N;
                p.D = D;
                p.K = K;
                // Section 10.3 method 2: separation far above 10 sigma, so no near-tie is
                // expected and device and reference must agree exactly.
                p.sep = 50.0;
                p.spread = 0.1;
                p.seed = 1600 + static_cast<uint64_t>(N * 1000 + D * 10 + K);
                const Dataset data = generate(p);

                double m_range = 0.0;
                const Validation v = validate_data(data.X, N, D, data.C0, K, &m_range);
                if (!v.ok) {
                    check(false, "N=" + std::to_string(N) + " D=" + std::to_string(D) +
                                     " K=" + std::to_string(K) + ": generated data rejected: " +
                                     v.message);
                    continue;
                }

                gpu::Options options;
                options.config.N = N;
                options.config.D = D;
                options.config.K = K;
                options.config.max_iter = 25;
                options.config.tol = 0.0;
                options.config.strategy = strategy;
                options.m_range = m_range;

                Result r;
                gpu::RunInfo info;
                if (!run_device(data.X.data(), data.C0.data(), options, &r, &info,
                                gpu::LockstepHook(),
                                "N=" + std::to_string(N) + " D=" + std::to_string(D) +
                                    " K=" + std::to_string(K))) {
                    continue;
                }
                ++configurations;

                const Result ref = cpu::run(data.X.data(), N, D, data.C0.data(), K, 25, 0.0,
                                            m_range, false);
                const std::string tag = "N=" + std::to_string(N) + " D=" + std::to_string(D) +
                                        " K=" + std::to_string(K);
                bool equal = r.status == ref.status && r.iterations == ref.iterations &&
                             r.centroid_consistent == ref.centroid_consistent;
                for (int64_t i = 0; equal && i < N; ++i) {
                    equal = r.assignments[static_cast<size_t>(i)] ==
                            ref.assignments[static_cast<size_t>(i)];
                }
                for (int64_t k = 0; equal && k < K; ++k) {
                    equal = r.counts[static_cast<size_t>(k)] == ref.counts[static_cast<size_t>(k)];
                }
                const double s_max = s_max_from_range(m_range);
                for (int64_t e = 0; equal && e < K * D; ++e) {
                    equal = centroid_agrees(r.centroids[static_cast<size_t>(e)],
                                            ref.centroids[static_cast<size_t>(e)], s_max);
                }
                if (!equal) {
                    ++mismatches;
                    if (mismatches <= 5) {
                        const gpu::FinalVerification fv =
                            gpu::verify_final(data.X.data(), r, options.config, m_range);
                        check(false, tag + ": device and reference disagree (status " +
                                         status_name(r.status) + " vs " + status_name(ref.status) +
                                         ", t " + std::to_string(r.iterations) + " vs " +
                                         std::to_string(ref.iterations) + ", P1 hard mismatches " +
                                         std::to_string(fv.p1_hard_mismatches) + ")");
                    }
                }
            }
        }
    }
    check_eq_i64(mismatches, 0, "every non-aligned configuration matched the CPU reference");
    std::printf("      %lld configurations run\n", static_cast<long long>(configurations));
}

}  // namespace

int main(int argc, char **argv)
{
    // The notebook runs the hand cases and the kernel checks as separate cells (section 15.3
    // parts 7 and 8), so the sections are selectable; --quick drops the sweeps that the Batch A
    // cells do not need.
    bool quick = false;
    bool run_hand = true;
    bool run_kernels = true;
    for (int i = 1; i < argc; ++i) {
        if (std::strcmp(argv[i], "--quick") == 0) {
            quick = true;
        } else if (std::strcmp(argv[i], "--only") == 0 && i + 1 < argc) {
            const char *what = argv[++i];
            if (std::strcmp(what, "hand") == 0) {
                run_kernels = false;
            } else if (std::strcmp(what, "kernels") == 0) {
                run_hand = false;
            } else if (std::strcmp(what, "all") != 0) {
                std::fprintf(stderr, "error[VAL-010]: --only expects hand, kernels or all\n");
                return kExitInvalidInput;
            }
        } else {
            std::fprintf(stderr, "usage: test_gpu [--only {all|hand|kernels}] [--quick]\n");
            return kExitInvalidInput;
        }
    }

    std::printf("M2-M3 GPU test suite (%s%s)\n", quick ? "quick" : "full",
                run_hand && run_kernels ? "" : (run_hand ? ", hand cases only"
                                                         : ", kernel checks only"));
    std::printf("requirements.md sections 6, 7, 16.0, 16.1, 16.2, 10.3\n\n");

    // The device is reported once, from the same query path the driver uses (CUDA-001).
    try {
        const DeviceInfo info = query_device(0);
        std::printf("device: %s\n", describe_device(info).c_str());
        std::printf("        CUDA runtime %d, driver %d, native atomicAdd(double): %s\n\n",
                    info.runtime_version, info.driver_version,
                    info.has_native_fp64_atomics() ? "yes" : "no");
    } catch (const CudaError &err) {
        std::fprintf(stderr, "error[CUDA-001]: %s\n", err.what());
        return err.exit_code();
    }

    const Strategy strategies[] = {Strategy::kAtomic, Strategy::kReduction};

    if (run_hand) {
        std::printf("TEST-I01 / TEST-I02: H1-H6 and H2-a..e (section 16.0)\n");
        for (Strategy strategy : strategies) {
            test_hand_cases(strategy);
        }
    }

    if (run_kernels) {
        std::printf("\nTEST-U06: buffer initialisation under poisoning\n");
        for (Strategy strategy : strategies) {
            test_poisoning(strategy);
        }

        std::printf("\nTEST-I10 / section 10.3: lockstep against the CPU reference\n");
        {
            const GenParams params = lockstep_params();
            const Dataset data = generate(params);
            std::printf("  dataset %s, minimum centre distance %.3f (%.1f sigma)\n",
                        data.dataset_id.c_str(), data.min_centre_distance,
                        data.spread > 0.0 ? data.min_centre_distance / data.spread : 0.0);
            for (Strategy strategy : strategies) {
                test_lockstep(strategy, data, params);
            }
        }

        if (!quick) {
            std::printf("\nTEST-U05: reduction accumulation over the G sweep (RED-006)\n");
            test_chunk_sweep();

            std::printf("\nTEST-U07: partial Stage 2 tiles and empty chunks (RED-003, EDGE-35)\n");
            test_partial_tiles();

            std::printf("\nTEST-E16: non-aligned N, D, K (CUDA-012, EDGE-16)\n");
            for (Strategy strategy : strategies) {
                test_non_aligned(strategy);
            }
        } else {
            std::printf("\nskipped in --quick: TEST-U05 (G sweep), TEST-U07 (partial tiles), "
                        "TEST-E16 (non-aligned sizes)\n");
        }
    }

    std::printf("\n%d checks, %d failures\n", g_checks, g_failures);
    if (g_failures != 0) {
        std::printf("RESULT: FAILED\n");
        return 1;
    }
    std::printf("RESULT: all GPU-level tests in this run passed\n");
    return 0;
}


## 6. Compile the host code

Builds the two host executables with the BLD-002 host flags, extended with `-Wall -Wextra`:

```
-std=c++17 -O2 -ffp-contract=off -Wall -Wextra
```

`-ffp-contract=off` (NUM-003) is the one flag that changes results rather than diagnostics. The
CPU reference is the oracle the device is judged against, and §2.2 fixes the distance as a
binary32 multiply followed by a binary32 add. If the compiler were allowed to contract that pair
into an FMA, the oracle's rounding would shift, every near-tie margin computed from it would be
measured against the wrong baseline, and NUM-005 mismatch diagnosis would become meaningless. The
device is permitted to contract (`-fmad=true`), which is precisely why the reference must not.

The cell treats **any** compiler output as a failure, not only a non-zero exit status: NFR-014
requires the compile cells to produce no host-code warnings, and a warning that merely scrolls past
in a saved notebook is indistinguishable from one nobody read.

In [ ]:
# Host compile cell — BLD-002 host flags plus -Wall -Wextra; no warnings tolerated (NFR-014).
import subprocess

if "GXX_REFERENCE_FLAGS" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first.")

# One source of truth: section 3 owns -O2 -std=c++17 -ffp-contract=off (NUM-003, BLD-002).
GXX_FLAGS = GXX_REFERENCE_FLAGS + ["-Wall", "-Wextra"]
CPU_LIB_SOURCES = ["validate.cpp", "io_kmd.cpp", "cpu_reference.cpp", "gen.cpp"]


def compile_host(output, sources):
    """Compile one host executable; raise on a non-zero exit or on any compiler output."""
    cmd = ["g++"] + GXX_FLAGS + ["-I", "."] + sources + ["-o", output]
    print("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    diagnostics = (proc.stdout + proc.stderr).strip()
    if diagnostics:
        print(diagnostics)
    if proc.returncode != 0:
        raise RuntimeError(f"g++ failed with exit code {proc.returncode} building {output}")
    if diagnostics:
        raise RuntimeError(
            f"g++ emitted diagnostics while building {output}; NFR-014 requires a warning-free "
            "host build. Fix the source in src/ or tests/, regenerate the notebook, and re-run."
        )
    print(f"  -> {output}")


compile_host("kmeans-gen", ["gen_main.cpp", "gen.cpp", "io_kmd.cpp", "validate.cpp"])
compile_host("test_cpu", ["test_cpu.cpp"] + CPU_LIB_SOURCES)
print("\nhost build clean: kmeans-gen and test_cpu, no warnings")

### 6b. CUDA compile cell (BLD-002, CUDA-010)

Builds the M2–M3 GPU test executable with the release flags of §15.3 part 3:
`-O3 -lineinfo -arch=$SM_ARCH -std=c++17`, **no `--use_fast_math`** (it would change the distance
semantics of §2.2, NUM-002), and `-Xcompiler -ffp-contract=off` so the CPU reference compiled into
the same binary stays the oracle it is meant to be (NUM-003).

`-Xptxas -v` prints per-kernel register and shared-memory usage, which CUDA-010 requires in the
build log and NFR-010 reports alongside benchmark results. Watch for the line for `assign` —
spills to local memory there would matter for M4's `assign_smem` comparison.

The cell raises on a non-zero `nvcc` exit code, so "restart kernel and run all" stops here rather
than at a confusing runtime failure (NFR-014).

In [ ]:
# CUDA compile cell — BLD-002 release flags, CUDA-010 register/shared usage in the build log.
import re
import subprocess

if "NVCC_RELEASE_FLAGS" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first.")

# Section 3 owns the flags. The CPU reference is linked into the GPU test binary, so the host FP
# flags of NUM-003 apply to the whole compile: a contracted multiply-add in distance2_fp32 would
# change the oracle the device is judged against.
CUDA_SOURCES = ["kernels.cu", "kmeans_gpu.cu"]
HOST_SOURCES = ["cpu_reference.cpp", "validate.cpp", "io_kmd.cpp", "gen.cpp"]


def compile_cuda(output, sources, extra_flags=()):
    """Compile one nvcc executable; raise on a non-zero exit. Returns the combined output."""
    cmd = (["nvcc"] + NVCC_RELEASE_FLAGS + NVCC_HOST_FP_FLAGS + list(extra_flags) +
           ["-I", "."] + sources + ["-o", output])
    print("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    diagnostics = (proc.stdout + proc.stderr).strip()
    if diagnostics:
        print(diagnostics)
    if proc.returncode != 0:
        raise RuntimeError(
            f"nvcc failed with exit code {proc.returncode} building {output}. Fix the source in "
            "src/ or tests/, regenerate the notebook with tools/build_notebook.py, and re-run."
        )
    print(f"  -> {output}")
    return diagnostics


build_log = compile_cuda("test_gpu", ["test_gpu.cu"] + CUDA_SOURCES + HOST_SOURCES)

# CUDA-010 / NFR-010: surface the per-kernel resource usage rather than leaving it buried in the
# -Xptxas -v noise, and flag any spill to local memory, which SMEM/PERF conclusions depend on.
print("\nper-kernel resource usage (-Xptxas -v, CUDA-010):")
usage = re.findall(
    r"Function properties for (\S+).*?(\d+) bytes stack frame, (\d+) bytes spill stores, "
    r"(\d+) bytes spill loads\s*ptxas info\s*:\s*Used (\d+) registers",
    build_log,
    re.DOTALL,
)
if usage:
    for name, stack, spill_store, spill_load, registers in usage:
        spilled = int(spill_store) + int(spill_load)
        flag = "   <-- SPILLS TO LOCAL MEMORY" if spilled else ""
        print(f"  {name:<48} {registers:>4} registers, {stack:>4} B stack, "
              f"{spilled:>4} B spill{flag}")
else:
    # Not a failure: ptxas wording varies between toolkit versions, and the raw log is above.
    print("  (could not parse the ptxas lines; read the raw -Xptxas -v output printed above)")

print("\nCUDA build clean: test_gpu")

## 7. Hand datasets H1–H6 and the CPU-level tests

Runs the test executable, which checks the CPU reference against every hand-verifiable case of
§16.0 — H1–H6 and the five H2 variants — and then TEST-U08, U09, U11, U12 and U13 at host level.
The cell raises if any check fails, so a broken oracle stops "restart kernel and run all" here,
before any GPU result could be compared against it (NFR-015).

What the hand cases pin down, case by case:

| Case | What it fixes |
|---|---|
| H1 | the ordinary path: one update, then assignment stability, `t = 1` |
| H2 | a three-update trajectory with the deltas 2, 1.5, 3.5, ending in `CONVERGED_ASSIGNMENTS` |
| H2-a, H2-b | $\Delta \le \varepsilon$ **inclusive** (DEC-002, ENV-10): the tolerance equals the delta exactly, and the run must stop |
| H2-c | one notch below that delta, so the run continues — the other side of the same boundary |
| H2-d, H2-e | the iteration limit, including $I_{\max} = 0$: `ITERATION_LIMIT`, never reported as convergence (§2.7) |
| H3 | an exact tie resolved to the lowest index (DEC-005) and an empty cluster retained (DEC-003) |
| H4 | duplicate initial centroids (`W-DUPC`): every point ties, so index 0 takes them all |
| H5 | $K > N$ (`W-KGTN`): at least $K - N$ clusters stay empty for the whole run |
| H6 | the only 2-D case, which catches a transposed or stride-confused layout that every 1-D case above would pass |

The P2 flag is checked in every case, including the four where it is correctly **false**: after
`CONVERGED_MOVEMENT` or `ITERATION_LIMIT` the reported centroids are generally not the means of
the reported assignments, and §2.8 requires that to be reported honestly rather than repaired by
extra hidden iterations.

The cell finishes by generating one small Gaussian-blob dataset through the `kmeans-gen` CLI
(FR-003, BLD-003a), which confirms the binary format round-trips in this environment and prints
the measured minimum centre separation in units of the spread — the quantity §10.3 uses to call a
dataset well-separated.

In [ ]:
# CPU-level test suite: §16.0 hand datasets plus TEST-U08/U09/U11/U12/U13. Raises on any failure.
import os
import subprocess

if "DATA_DIR" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first.")

tests = subprocess.run(["./test_cpu"], capture_output=True, text=True)
print(tests.stdout + tests.stderr, end="")
if tests.returncode != 0:
    raise RuntimeError(
        f"./test_cpu exited with code {tests.returncode}: the CPU reference does not reproduce the "
        "hand-verified results of requirements.md §16.0. Every GPU comparison depends on this "
        "oracle, so fix it in src/ before continuing."
    )

# FR-003 / BLD-003a: one small dataset, to confirm the generator and the BLD-004 format work here.
prefix = os.path.join(DATA_DIR, "demo_n2000_d4_k3")
gen = subprocess.run(
    ["./kmeans-gen", "--n", "2000", "--d", "4", "--k", "3", "--sep", "10", "--spread", "1",
     "--seed", "20261017", "--out-prefix", prefix],
    capture_output=True, text=True,
)
print("\n$ kmeans-gen --n 2000 --d 4 --k 3 --sep 10 --spread 1 --seed 20261017")
print(gen.stdout + gen.stderr, end="")
if gen.returncode != 0:
    raise RuntimeError(f"kmeans-gen exited with code {gen.returncode} (see BLD-005 for the codes)")

for suffix, expected_rows in (("_X.kmd", 2000), ("_C0.kmd", 3)):
    path = prefix + suffix
    # BLD-004: 24-byte header (magic, version, rows, cols) then rows*cols binary32.
    expected_bytes = 24 + expected_rows * 4 * 4
    actual = os.path.getsize(path)
    if actual != expected_bytes:
        raise RuntimeError(f"{path} is {actual} B, expected {expected_bytes} B for "
                           f"{expected_rows}x4 binary32 values plus the BLD-004 header")
    print(f"  {os.path.basename(path)}: {actual} B as the format requires")

### 7b. Hand datasets on the GPU, both strategies (TEST-I01, TEST-I02)

The same §16.0 cases the CPU reference reproduced above, now run on the device with **Approach A
(atomic)** and **Approach B (reduction)**, which is TEST-I01 and TEST-I02. Because the H2 variants
encode the convergence semantics, this one cell also carries TEST-I03 (iteration counting: H1 at
`t=1`, H2 at `t=3`), TEST-I04 (the movement boundary: H2-a/b/c), TEST-I05 (the limit, including
`I_max = 0`: H2-d/e) and TEST-I06 (the P2 flag).

Expected output: every case `PASS`, for both strategies, with `0 failures`. The values checked are
exactly the ones derived by hand in §16.0 — assignments, centroids, counts, iteration count,
status and the P2 flag — compared exactly, except `5/3` in H2 and H2-c which goes through NUM-006.
Empty clusters are additionally checked to have retained their initial centroid **bit-exactly**
(DEC-003), and each case is re-judged by `gpu::verify_final`, so P1 is verified rather than
assumed (FR-019, §2.8).

A failure here means a kernel disagrees with arithmetic done by hand on four points, so fix it in
`src/` before reading anything into the lockstep or benchmark cells.

In [ ]:
# TEST-I01 / TEST-I02: the §16.0 hand datasets on the device, both strategies.
import subprocess

if "WORK_DIR" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first.")

hand = subprocess.run(["./test_gpu", "--only", "hand"], capture_output=True, text=True)
print(hand.stdout + hand.stderr, end="")
if hand.returncode != 0:
    raise RuntimeError(
        f"./test_gpu --only hand exited with code {hand.returncode}: a kernel disagrees with the "
        "hand-derived results of requirements.md §16.0. Fix the source in src/, regenerate the "
        "notebook, and re-run; do not continue to the kernel checks or the benchmarks."
    )

### 8. Kernel checks: lockstep against the CPU reference, and buffer initialization

This section grows into the full TEST-B02 suite in M5. At M2–M3 it carries the two checks that
judge the kernels on realistic data rather than on four hand-computed points:

**TEST-U06 — buffer initialization (MEM-006).** `P`, `Q`, `S` and `n` are filled with `0xFF` bytes
before the run, which is a quiet NaN as binary64 and `0xFFFFFFFF` as uint32. A correct Approach B
overwrites every element of `P` and `Q` in Stage 1 and every element of `S` and `n` in Stage 2, so
nothing of the poison may reach a result. Expected: identical assignments and counts, centroids
**bitwise** identical for Approach B (RED-009) and within NUM-006 for Approach A (ATOM-007, which
fixes no summation order), and every centroid finite.

**TEST-I10 / §10.3 — lockstep.** A 100 000 × 16, K = 8 Gaussian-blob dataset, both strategies, with
every iteration compared against the CPU reference **kernel by kernel**, so a failure names the
kernel instead of only the end result:

| Checked | Against | Criterion |
|---|---|---|
| `n` | `cpu::accumulate` on the device's own `A(t)` | **exact**, and `Σ_k n_k = N` (RED-005, FR-009) |
| `S` | the same | `1e-12 · n_k · S_max` — proportional to the largest value the sum could take |
| `C(t+1)` | `cpu::update` on the same `(S, n)` | NUM-006; empty clusters retained **bit-exactly** (DEC-003) |
| `Δ` | `cpu::movement` on the device's own centroids and counts | isolates `movement`/`max_reduce` from any centroid difference |
| `A(t+1)` | `cpu::assign` on the device's own `C(t+1)` | disagreements allowed **only** on near-ties (NUM-005) |
| `changed` | the reference count of differences `A(t)` vs `A(t+1)` | **exact** (TEST-U03) |

The first snapshot additionally pins down the initial assignment pass of §2.7, which no later
snapshot covers. This validates every kernel on realistic data without requiring device and
reference to follow identical trajectories, which is exactly what §10.3 item 1 is for.

Expected output: `0 failures`, and for each strategy a line reporting the iterations checked, the
near-tie assignment differences (a count, not a failure) and the worst `|ΔS|`, `|ΔC|` and `|ΔΔ|`.
The reduction run also prints its selected `G` with `G_target`, `G_mem` and the bytes of partials
(RED-006, RED-007). Lockstep runs are marked **not timing-valid** — the per-iteration downloads sit
inside the measured region, so FR-019 bars them from being benchmark samples.

With `QUICK = False` the cell also runs the sweeps that take longer: TEST-U05 (`G ∈ {1, 2, 7, N}`
and auto), TEST-U07 (`K = 1024`, `D ∈ {1, 33, 65}`, `G = 7` — partial Stage 2 tiles and chunks with
no points of most clusters, EDGE-35) and TEST-E16 (every combination of `N ∈ {1, 31, 33, 255, 257,
1023}`, `D ∈ {1, 3, 31, 33, 65}`, `K ∈ {1, 3, 7, 9, 17}` against the CPU reference on
well-separated data, §10.3 method 2).

In [ ]:
# TEST-U06 and TEST-I10 / §10.3: buffer initialisation and lockstep verification.
# QUICK (section 3) drops the TEST-U05 / U07 / E16 sweeps; the lockstep run itself always runs.
import subprocess

if "QUICK" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first.")

cmd = ["./test_gpu", "--only", "kernels"] + (["--quick"] if QUICK else [])
print("$ " + " ".join(cmd))
checks = subprocess.run(cmd, capture_output=True, text=True)
print(checks.stdout + checks.stderr, end="")
if checks.returncode != 0:
    raise RuntimeError(
        f"{' '.join(cmd)} exited with code {checks.returncode}: the device disagrees with the CPU "
        "reference beyond what NUM-005/NUM-006 allow, or a poisoned buffer reached a result. The "
        "failure lines above name the kernel and the quantity; fix it in src/, regenerate the "
        "notebook, and re-run."
    )

if QUICK:
    print("\nQUICK = True: TEST-U05 (G sweep), TEST-U07 (partial tiles) and TEST-E16 "
          "(non-aligned sizes) were skipped. Set QUICK = False in section 3 to run them.")